# 24. 추천형 다중정답 검색 개발평가 — CPU/offline preflight

Codex coder agent가 만든 단일 재현 notebook입니다. 이번 실행은 현재 10개 문서 안의 조건 일치 카드 검색을 위한 개발 preflight이며, 시장 전체 추천·최신 발급·개인 자격·상품 우열 평가는 아닙니다. 외부 전송, query embedding, BGE load, GPU, Chroma query는 실행하지 않습니다.

## 결과 전에 고정한 계약

Primary K=5, guardrail K=3, candidate D=10/20입니다. old+selective-BGE와 structural+all-BGE의 D10/D20만 선택 후보이며, Top10의 두 청킹×no/selective/all-BGE는 원인 진단 전용입니다. BM25 k1=1.5, b=0.75, RRF Vector:BM25=0.4:0.6, k=60, component depth=50을 고정합니다. Gold는 원문 atomic anchor로 먼저 동결하고 순위 생성에는 쓰지 않습니다. 결과는 dev signal이며 운영·holdout 승격 근거가 아닙니다.

In [1]:
from pathlib import Path
import csv, hashlib, json, os, re, sys, time, unicodedata
from collections import Counter, defaultdict

cwd = Path.cwd().resolve()
if (cwd / 'notebooks').is_dir():
    ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'notebooks').is_dir():
    ROOT = cwd.parent
else:
    raise RuntimeError(f'fail-closed: unexpected cwd {cwd}')
NB = ROOT / 'notebooks/24_multicard_recommendation_retrieval_evaluation.ipynb'
OUT = ROOT / 'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
OLD_CHUNKS = ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl'
NEW_CHUNKS = ROOT / 'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'
BGE_ROOT = ROOT / '.cache/reranker/bge-reranker-v2-m3'
assert OLD_CHUNKS.is_file() and NEW_CHUNKS.is_file() and BGE_ROOT.is_dir()

RAW = {
 'C01':'data/ocr_benchmark/gold/raw/BC/BC_Biz_AirMoney.txt',
 'C02':'data/ocr_benchmark/gold/raw/NH/NH_Namu_NH.txt',
 'C03':'data/ocr_benchmark/gold/raw/hana/Hana_One_More_SOHO.txt',
 'C04':'data/ocr_benchmark/gold/raw/hyundai/Hyundai_The_Orange_20260330.txt',
 'C05':'data/ocr_benchmark/gold/raw/ibk/IBK_Point3.8(Credit).txt',
 'C06':'data/ocr_benchmark/gold/raw/kookmin/Kookmin_Friend_20210917.txt',
 'C07':'data/ocr_benchmark/gold/raw/lotte/Lotte_LOCA_LIKIT_Eat.txt',
 'C08':'data/ocr_benchmark/gold/raw/samsung/Samsung_iD_ALL.txt',
 'C09':'data/ocr_benchmark/gold/raw/shinhan/Shinhan_Toss_Mr.Life_20251231.txt',
 'C10':'data/ocr_benchmark/gold/raw/woori/Woori_Classic_EVERY_MILE_SKYPASS.txt'}
CARD_KEYS = {
 'C01':'BC/BC_Biz_AirMoney','C02':'NH/NH_Namu_NH','C03':'hana/Hana_One_More_SOHO',
 'C04':'hyundai/Hyundai_The_Orange_20260330','C05':'ibk/IBK_Point3.8(Credit)',
 'C06':'kookmin/Kookmin_Friend_20210917','C07':'lotte/Lotte_LOCA_LIKIT_Eat',
 'C08':'samsung/Samsung_iD_ALL','C09':'shinhan/Shinhan_Toss_Mr.Life_20251231',
 'C10':'woori/Woori_Classic_EVERY_MILE_SKYPASS'}
QUERIES = {
 'Q01':'신규 발급 유예가 아닌 상시 기준으로, 문서에 포함된 상품형태 중 국내 일반 가맹점에서 전월 실적 조건 없이 적립이나 할인을 받는 카드를 모두 알려줘',
 'Q02':'쿠팡이나 G마켓에서 온라인 쇼핑 상품을 결제할 때 별도 할인·적립·캐시백이 있는 카드를 모두 알려줘',
 'Q03':'배달의민족·요기요·쿠팡이츠에서 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q04':'이동통신 요금을 낼 때 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q05':'전기요금이나 도시가스요금을 낼 때 직접 할인 혜택이 있는 카드를 모두 알려줘',
 'Q06':'커피전문점에서 결제할 때 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q07':'영상·음악 구독이나 유료 멤버십에 직접 혜택이 있는 카드를 모두 알려줘',
 'Q08':'일반 음식점에서 결제할 때 직접 할인이나 적립 혜택이 있는 카드를 모두 알려줘',
 'Q09':'대형마트나 슈퍼에서 결제할 때 별도 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q10':'대중교통이나 택시 이용 시 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘'}
POSITIVES = {
 'Q01':['C01','C03','C06','C08','C10'], 'Q02':['C02','C03','C04','C05','C09'],
 'Q03':['C02','C07'], 'Q04':['C02','C03','C04','C08','C09'], 'Q05':['C03','C09'],
 'Q06':['C02','C07','C09'], 'Q07':['C02','C04','C07'], 'Q08':['C04','C07','C09'],
 'Q09':['C02','C08','C09'], 'Q10':['C02','C09']}
# 원문에 실제 존재해야 하는 최소 hard-predicate anchor. 각 positive pair를 먼저 원문에 고정한다.
TERMS = {
 ('Q01','C01'):['Air Money 0.2% 적립','전월 실적 제한 없음'],
 ('Q01','C03'):['국내 가맹점 0.5% 청구할인','지난달 실적에 관계없이'],
 ('Q01','C06'):['일시불 및 할부 이용금액 1,000원당 1마일 적립'],
 ('Q01','C08'):['국내외 가맹점 0.5% 할인'],
 ('Q01','C10'):['국내외 가맹점','전월실적 조건 없음'],
 ('Q02','C02'):['온라인쇼핑 / 배달앱','쿠팡, 11번가, G마켓'],
 ('Q02','C03'):['온라인쇼핑','네이버쇼핑, 쿠팡, 11번가, G마켓'],
 ('Q02','C04'):['온라인몰','네이버플러스 스토어, 쿠팡'],
 ('Q02','C05'):['온라인쇼핑','쿠팡, G마켓, 옥션, 11번가'],
 ('Q02','C09'):['온라인 쇼핑','옥션, G마켓, AK몰, 11번가'],
 ('Q03','C02'):['온라인쇼핑 / 배달앱','배달의민족, 요기요'],
 ('Q03','C07'):['배달의 민족, 쿠팡이츠, 요기요','60% 결제일 할인'],
 ('Q04','C02'):['이동통신 / 구독','SKT, KT, LGU+ 자동납부'],
 ('Q04','C03'):['통신요금(자동납부)','SKT, KT, LGU+ 통신요금'],
 ('Q04','C04'):['이동통신 요금','SKT, KT, LG U+'],
 ('Q04','C08'):['이동통신','SKT, KT, LG U+'],
 ('Q04','C09'):['SKT, LG U+, KT 통신요금'],
 ('Q05','C03'):['전기/도시가스(자동납부)','전기요금(한국전력), 도시가스요금'],
 ('Q05','C09'):['전기요금','도시가스요금'],
 ('Q06','C02'):['커피 / 편의점(오프라인)','스타벅스, 이디야, 투썸플레이스'],
 ('Q06','C07'):['스타벅스, 투썸플레이스, 할리스커피, 폴바셋','60% 결제일 할인'],
 ('Q06','C09'):['커피전문점 업종','10% 할인'],
 ('Q07','C02'):['이동통신 / 구독','유튜브 프리미엄, 넷플릭스, 웨이브, 멜론'],
 ('Q07','C04'):['디지털 콘텐츠 서비스','넷플릭스, 유튜브 프리미엄'],
 ('Q07','C07'):['쿠팡 로켓와우, 네이버플러스 멤버십','60% 결제일 할인'],
 ('Q08','C04'):['다이닝','일반음식점'],
 ('Q08','C07'):['음식점','60% 결제일 할인'],
 ('Q08','C09'):['일반대중음식점','10% 할인'],
 ('Q09','C02'):['오프라인쇼핑 / 잡화','이마트, 홈플러스, 롯데마트'],
 ('Q09','C08'):['할인점','이마트, 이마트 트레이더스, 롯데마트, 홈플러스'],
 ('Q09','C09'):['3대 할인 마트','이마트, 롯데마트, 홈플러스'],
 ('Q10','C02'):['대중교통 / 택시','지하철, 시내외버스'],
 ('Q10','C09'):['택시','후불교통/비교통 카드 모두 적용']}
assert set(TERMS) == {(q,c) for q,cs in POSITIVES.items() for c in cs}

CONTRACT = {
 'experiment':'multicard_recommendation_retrieval_development_preflight',
 'scope':'fixed_10_documents_only_not_market_recommendation', 'phase':'cpu_offline_preflight_only',
 'k_primary':5,'k_guardrail':3,'candidate_depths':[10,20],
 'search':{'bm25_k1':1.5,'bm25_b':0.75,'vector_weight':0.4,'bm25_weight':0.6,'rrf_k':60,'component_depth':50},
 'primary_systems':['old_selective_bge_d10','old_selective_bge_d20','structural_all_bge_d10','structural_all_bge_d20'],
 'diagnostic_top10_systems':['old_no','old_selective_bge','old_all_bge','structural_no','structural_selective_bge','structural_all_bge'],
 'excluded':['gte','mmr','morphology','new_routing'],
 'old_candidate_policy':'RRF Top50 then level in {section,benefit}, order preserved',
 'structural_candidate_policy':'direct-body searchable chunks',
 'bge':{'revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'max_length':8192,'truncation':'only_second','dtype':'float16','batch_size':2,'ranking':'raw single logit desc; tie original RRF rank then chunk_id','score_mixing':False},
 'reranker_text':'construct from allowlisted issuer, card_name, heading_path, body; each title/path component exactly once; no gold/evaluation fields',
 'metrics':['Card Precision@3','Card Recall@3','Card Precision@5','Card Recall@5','Evidence-supported Card Recall@5','Evidence Accuracy@3','Evidence Accuracy@5','Candidate Card Recall@D','Candidate Evidence Recall@D','zero-hit','raw duplicate diagnostic'],
 'selection_gate':'same-D structural all-BGE vs old selective-BGE: supported Recall@5 strict improvement and wins>losses; Precision@5, Evidence Accuracy@5, Recall@3, zero-hit non-regression; catastrophic loss 0. D20 only if supported Recall@5 or Evidence Accuracy@5 strictly improves over D10; tie by Evidence Accuracy@5, Precision@5, lower duplicate, then D10',
 'selection_status_ceiling':'dev_signal_only',
 'prohibited_runtime':{'api':0,'network':0,'query_embedding':0,'bge_model_load':0,'gpu':0,'chroma_query':0,'package_install':0}}

def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1<<20), b''): h.update(block)
    return h.hexdigest()
def norm(value): return ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())
def write_json(path,obj): path.write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def write_csv(path,rows,fields):
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def load_jsonl(path):
    with path.open(encoding='utf-8') as f: return [json.loads(x) for x in f if x.strip()]
def tree_map(root):
    return {str(p.relative_to(root)):sha(p) for p in sorted(p for p in root.rglob('*') if p.is_file())}

explicit_inputs=[OLD_CHUNKS,NEW_CHUNKS]+[ROOT/p for p in RAW.values()]
before_files={str(p.relative_to(ROOT)):sha(p) for p in explicit_inputs}
before_bge=tree_map(BGE_ROOT)
assert before_bge
old=load_jsonl(OLD_CHUNKS); new=load_jsonl(NEW_CHUNKS)
assert len(old)==327 and len(new)==147
for row in old: row['_text']=row['document']; row['_card_key']=row['metadata']['card_key']
for row in new: row['_text']=row['evidence_text']; row['_card_key']=row['metadata']['card_key']
assert set(CARD_KEYS.values()) == {r['_card_key'] for r in new}

query_rows=[{'query_id':q,'query_text':text,'query_sha256':hashlib.sha256(text.encode()).hexdigest(),'positive_count':len(POSITIVES[q])} for q,text in QUERIES.items()]
label_rows=[]
for q in QUERIES:
    for c in RAW:
        positive=c in POSITIVES[q]
        label_rows.append({'query_id':q,'card_id':c,'card_key':CARD_KEYS[c],'label':'positive' if positive else 'negative','claim_id':f'{q}_{c}' if positive else ''})
assert len(label_rows)==100 and Counter(r['label'] for r in label_rows)=={'negative':67,'positive':33}
assert all(2<=len(v)<=5 for v in POSITIVES.values())

claim_rows=[]; mapping_rows=[]; unresolved=[]
for (q,c),terms in sorted(TERMS.items()):
    raw_path=ROOT/RAW[c]; raw_text=raw_path.read_text(encoding='utf-8')
    positions=[]; predicate_spans=[]
    for term in terms:
        pos=raw_text.find(term)
        assert pos>=0, (q,c,term)
        positions.append((pos,pos+len(term)))
        term_markers=list(re.finditer(r'(?im)^\[page\s+(\d+)\]\s*$',raw_text[:pos]))
        predicate_spans.append({'term':term,'start':pos,'end':pos+len(term),'page':int(term_markers[-1].group(1)) if term_markers else 1,'line':raw_text.count('\n',0,pos)+1})
    start,end=positions[0]
    markers=list(re.finditer(r'(?im)^\[page\s+(\d+)\]\s*$',raw_text[:start]))
    page=int(markers[-1].group(1)) if markers else 1
    line=raw_text.count('\n',0,start)+1
    claim_id=f'{q}_{c}'
    claim_rows.append({'claim_id':claim_id,'query_id':q,'card_id':c,'card_key':CARD_KEYS[c],'source_path':RAW[c],'source_page':page,'span_start':start,'span_end':end,'line_start':line,'anchor':raw_text[start:end],'predicate_terms_json':json.dumps(terms,ensure_ascii=False),'predicate_spans_json':json.dumps(predicate_spans,ensure_ascii=False)})
    for corpus,rows in [('old',old),('structural',new)]:
        card_rows=[r for r in rows if r['_card_key']==CARD_KEYS[c]]
        term_hits=[]
        for term in terms:
            nterm=norm(term)
            hits=sorted(r.get('id') or r.get('chunk_id') for r in card_rows if nterm in norm(r['_text']))
            term_hits.append(hits)
        joint=sorted(r.get('id') or r.get('chunk_id') for r in card_rows if all(norm(t) in norm(r['_text']) for t in terms))
        resolved=all(term_hits)
        if not resolved: unresolved.append((claim_id,corpus,[terms[i] for i,x in enumerate(term_hits) if not x]))
        mapping_rows.append({'claim_id':claim_id,'query_id':q,'card_id':c,'corpus':corpus,'all_predicates_located':resolved,'joint_chunk_count':len(joint),'joint_chunk_ids_json':json.dumps(joint,ensure_ascii=False),'term_chunk_ids_json':json.dumps(term_hits,ensure_ascii=False)})
assert not unresolved, unresolved
assert len(claim_rows)==33 and len(mapping_rows)==66

write_csv(OUT/'queries.csv',query_rows,['query_id','query_text','query_sha256','positive_count'])
write_csv(OUT/'gold_card_labels.csv',label_rows,['query_id','card_id','card_key','label','claim_id'])
write_csv(OUT/'atomic_claims.csv',claim_rows,['claim_id','query_id','card_id','card_key','source_path','source_page','span_start','span_end','line_start','anchor','predicate_terms_json','predicate_spans_json'])
write_csv(OUT/'evidence_projection.csv',mapping_rows,['claim_id','query_id','card_id','corpus','all_predicates_located','joint_chunk_count','joint_chunk_ids_json','term_chunk_ids_json'])
source_rows=[]
for c,rel in RAW.items():
    p=ROOT/rel; text=p.read_text(encoding='utf-8')
    source_rows.append({'card_id':c,'card_key':CARD_KEYS[c],'source_path':rel,'sha256':sha(p),'bytes':p.stat().st_size,'page_marker_count':len(re.findall(r'(?im)^\[page\s+\d+\]\s*$',text))})
write_csv(OUT/'source_manifest.csv',source_rows,['card_id','card_key','source_path','sha256','bytes','page_marker_count'])
write_json(OUT/'evaluation_contract.json',CONTRACT)

gold_files=['queries.csv','gold_card_labels.csv','atomic_claims.csv','evidence_projection.csv','source_manifest.csv','evaluation_contract.json']
gold_hashes={name:sha(OUT/name) for name in gold_files}
gold_digest=hashlib.sha256(json.dumps(gold_hashes,sort_keys=True,separators=(',',':')).encode()).hexdigest()
write_json(OUT/'gold_freeze.json',{'frozen_before_any_ranking':True,'files':gold_hashes,'digest':gold_digest,'ranking_files_created':0})

try:
    import tiktoken
    enc=tiktoken.encoding_for_model('text-embedding-3-small')
    token_counts={q:len(enc.encode(text)) for q,text in QUERIES.items()}
    token_counter='tiktoken.encoding_for_model(text-embedding-3-small)'
except Exception as exc:
    raise RuntimeError(f'local token counter unavailable; no network fallback allowed: {exc}')
embedding_items=[{'query_id':q,'text':QUERIES[q],'sha256':hashlib.sha256(QUERIES[q].encode()).hexdigest(),'tokens':token_counts[q]} for q in QUERIES]
embedding_manifest_hash=hashlib.sha256(json.dumps(embedding_items,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
embedding_plan={'model':'text-embedding-3-small','item_count':10,'unique_text_count':len(set(QUERIES.values())),'token_count':sum(token_counts.values()),'max_item_tokens':max(token_counts.values()),'max_requests':1,'manifest_sha256':embedding_manifest_hash,'token_counter':token_counter,'external_transmission_performed':False,'current_run_api_requests':0,'official_price_source':None,'price_per_million_input_tokens_usd':None,'estimated_cost_usd':None,'approval_caps':{'items':10,'tokens':sum(token_counts.values()),'requests':1}}
write_json(OUT/'embedding_plan.json',embedding_plan)
pair_plan={'model':'BGE reranker local cache only','candidate_depth_max':20,'old_unique_pairs_upper_bound':200,'structural_unique_pairs_upper_bound':200,'total_unique_pairs_upper_bound':400,'d10_is_required_prefix_of_d20':True,'top10_diagnostics_reuse_same_scores':True,'gpu_scoring_performed':False,'model_load_performed':False,'pair_count_is_preflight_upper_bound':True}
write_json(OUT/'gpu_pair_plan.json',pair_plan)
cache_manifest={'root':str(BGE_ROOT.relative_to(ROOT)),'revision':CONTRACT['bge']['revision'],'file_count':len(before_bge),'total_bytes':sum((BGE_ROOT/p).stat().st_size for p in before_bge),'files':before_bge}
write_json(OUT/'bge_cache_manifest.json',cache_manifest)
corpus_manifest={'old':{'path':str(OLD_CHUNKS.relative_to(ROOT)),'sha256':sha(OLD_CHUNKS),'rows':len(old)},'structural':{'path':str(NEW_CHUNKS.relative_to(ROOT)),'sha256':sha(NEW_CHUNKS),'rows':len(new)}}
write_json(OUT/'corpus_manifest.json',corpus_manifest)

after_files={str(p.relative_to(ROOT)):sha(p) for p in explicit_inputs}; after_bge=tree_map(BGE_ROOT)
assert before_files==after_files and before_bge==after_bge
integrity={'status':'PASS','query_count':10,'card_count':10,'label_count':100,'positive_count':33,'negative_count':67,'insufficient_count':0,'atomic_claim_count':33,'projection_rows':66,'unresolved_projection_count':0,'old_chunk_count':327,'structural_chunk_count':147,'gold_frozen_before_ranking':True,'ranking_files_created':0,'input_hashes_unchanged':True,'bge_cache_unchanged':True,'api_requests':0,'network_calls':0,'query_embeddings_created':0,'model_loads':0,'gpu_used':False,'chroma_queries':0,'package_installs':0,'top10_prefix_top20_check':'deferred_until_rankings_exist','unique_output_card_check':'deferred_until_rankings_exist'}
write_json(OUT/'integrity.json',integrity)
readme=f'''# 24 다중정답 추천형 검색 개발평가 — preflight\n\n현재 10개 문서 안에서 조건에 맞는 카드를 여러 장 찾는 개발평가입니다. 시장 전체 추천, 최신 발급 여부, 개인 자격, 상품 우열을 평가하지 않습니다.\n\n이번 실행은 CPU/offline preflight만 완료했습니다. 질의 10개, 카드 라벨 100개(positive 33, negative 67, insufficient 0), 원자 근거 33개와 old/structural 독립 투영 66행을 동결했습니다. 순위·embedding·reranker 점수는 만들지 않았습니다.\n\n- Card Precision: 반환한 카드 중 정답 카드 비율\n- Card Recall: 정답 카드 중 반환한 비율\n- Evidence-supported Card Recall: 대표 청크 하나가 hard predicate까지 증명한 정답 카드 회수율\n- Evidence Accuracy: 반환 슬롯 중 카드와 근거가 모두 맞는 비율. 빈 슬롯은 실패\n\n외부 단계는 `external_execution_approval.json`의 정확한 manifest 승인 없이는 닫혀 있습니다. BGE는 로컬 캐시만 쓰며 gold/evaluation 필드를 scorer 입력에 넣지 않습니다. 결과는 dev signal 전용입니다.\n'''
(OUT/'README.md').write_text(readme,encoding='utf-8')
output_names=['queries.csv','gold_card_labels.csv','atomic_claims.csv','evidence_projection.csv','source_manifest.csv','evaluation_contract.json','gold_freeze.json','embedding_plan.json','gpu_pair_plan.json','bge_cache_manifest.json','corpus_manifest.json','integrity.json','README.md']
manifest={'created_by':'Codex coder agent','phase':'cpu_offline_preflight','outputs':{n:sha(OUT/n) for n in output_names},'inputs_before':before_files,'inputs_after':after_files,'bge_tree_digest':hashlib.sha256(json.dumps(before_bge,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'manifest_self_hash_excluded':True}
write_json(OUT/'preflight_run_manifest.json',manifest)
print(json.dumps({'status':'PASS','labels':'33 positive / 67 negative / 0 insufficient','embedding_items':embedding_plan['item_count'],'embedding_tokens':embedding_plan['token_count'],'max_requests':embedding_plan['max_requests'],'planned_gpu_pairs_upper_bound':pair_plan['total_unique_pairs_upper_bound'],'external_calls':0},ensure_ascii=False,indent=2))

{
  "status": "PASS",
  "labels": "33 positive / 67 negative / 0 insufficient",
  "embedding_items": 10,
  "embedding_tokens": 525,
  "max_requests": 1,
  "planned_gpu_pairs_upper_bound": 400,
  "external_calls": 0
}


In [2]:
# Phase 2 boundary: this cell performs no external call. It only demonstrates fail-closed authorization.
approval_path = OUT / 'external_execution_approval.json'
authorized = False
if approval_path.is_file():
    approval = json.loads(approval_path.read_text(encoding='utf-8'))
    plan = json.loads((OUT/'embedding_plan.json').read_text(encoding='utf-8'))
    authorized = (approval.get('approved_manifest_sha256') == plan['manifest_sha256'] and approval.get('approved_caps') == plan['approval_caps'])
if not authorized:
    print('FAIL-CLOSED: no exact external execution approval; API/model/GPU steps remain disabled.')
else:
    raise RuntimeError('Approval recognized, but Phase 2 implementation/execution is outside this preflight run.')

FAIL-CLOSED: no exact external execution approval; API/model/GPU steps remain disabled.


## 승인 후 실행 — embedding, ranking freeze, BGE, 평가

사용자가 manifest `031928...2027`, 10개, 525 tokens, 최대 1 request만 승인했습니다. 아래 셀은 질의 문자열만 API에 보내며 문서·gold·평가 필드는 보내지 않습니다. 첫 셀은 gold 파일을 읽지 않고 ranking을 저장·hash한 뒤 종료합니다. 다음 셀은 같은 Top20 후보 400쌍만 local BGE로 점수화합니다. 마지막 셀에서 처음 gold를 읽고 평가합니다.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from decimal import Decimal
import csv, hashlib, json, math, os, re, resource, time, unicodedata
import numpy as np

cwd=Path.cwd().resolve()
ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None
assert ROOT is not None
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
OLD_CHUNKS=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl'
NEW_CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'
OLD_CACHE=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/embedding_cache/text-embedding-3-small'
NEW_CACHE=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/embedding_cache/text-embedding-3-small'
BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'
APPROVED_MANIFEST='031928bee3ba51aa700afbbb6903e0a1e839a1afcfbe7a4e45dbdb5a8d4b2027'
APPROVED_CAPS={'items':10,'tokens':525,'requests':1}
QUERIES={
 'Q01':'신규 발급 유예가 아닌 상시 기준으로, 문서에 포함된 상품형태 중 국내 일반 가맹점에서 전월 실적 조건 없이 적립이나 할인을 받는 카드를 모두 알려줘',
 'Q02':'쿠팡이나 G마켓에서 온라인 쇼핑 상품을 결제할 때 별도 할인·적립·캐시백이 있는 카드를 모두 알려줘',
 'Q03':'배달의민족·요기요·쿠팡이츠에서 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q04':'이동통신 요금을 낼 때 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q05':'전기요금이나 도시가스요금을 낼 때 직접 할인 혜택이 있는 카드를 모두 알려줘',
 'Q06':'커피전문점에서 결제할 때 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q07':'영상·음악 구독이나 유료 멤버십에 직접 혜택이 있는 카드를 모두 알려줘',
 'Q08':'일반 음식점에서 결제할 때 직접 할인이나 적립 혜택이 있는 카드를 모두 알려줘',
 'Q09':'대형마트나 슈퍼에서 결제할 때 별도 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘',
 'Q10':'대중교통이나 택시 이용 시 직접 할인이나 캐시백 혜택이 있는 카드를 모두 알려줘'}
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def canonical(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x]
def tree_state(root):
    files=sorted(p for p in root.rglob('*') if p.is_file()); mapping={str(p.relative_to(root)):sha(p) for p in files}
    return {'file_count':len(files),'total_bytes':sum(p.stat().st_size for p in files),'files':mapping,'digest':hashlib.sha256(canonical(mapping).encode()).hexdigest()}

approval=json.loads((OUT/'external_execution_approval.json').read_text(encoding='utf-8'))
plan=json.loads((OUT/'embedding_plan.json').read_text(encoding='utf-8'))
assert approval['approval_status']=='exact_scope_approved'
assert approval['approved_manifest_sha256']==plan['manifest_sha256']==APPROVED_MANIFEST
assert approval['approved_caps']==plan['approval_caps']==APPROVED_CAPS
assert approval['approved_model']==plan['model']=='text-embedding-3-small'
items=[{'query_id':q,'text':text,'sha256':hashlib.sha256(text.encode()).hexdigest(),'tokens':None} for q,text in QUERIES.items()]
import tiktoken
enc=tiktoken.encoding_for_model('text-embedding-3-small')
for item in items: item['tokens']=len(enc.encode(item['text']))
manifest=hashlib.sha256(canonical(items).encode()).hexdigest()
assert manifest==APPROVED_MANIFEST and len(items)==10 and sum(x['tokens'] for x in items)==525
assert len(items)<=APPROVED_CAPS['items'] and sum(x['tokens'] for x in items)<=APPROVED_CAPS['tokens'] and APPROVED_CAPS['requests']==1

old_usage_source=json.loads((OLD_CACHE.parent.parent/'embedding_usage.json').read_text(encoding='utf-8'))
new_usage_source=json.loads((NEW_CACHE.parent.parent/'embedding_usage.json').read_text(encoding='utf-8'))
old_cache_files=[OLD_CACHE/(b['batch_fingerprint']+'.npz') for b in old_usage_source['batches']]
new_cache_files=[ROOT/b['cache_path'] for b in new_usage_source['batches']]
assert len(old_cache_files)==6 and len(new_cache_files)==3 and all(p.is_file() for p in old_cache_files+new_cache_files)
immutable_paths=[OLD_CHUNKS,NEW_CHUNKS,*old_cache_files,*new_cache_files]
immutable_before={str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}
bge_before=tree_state(BGE_ROOT)
cache_dir=OUT/'embedding_cache/text-embedding-3-small'; cache_dir.mkdir(parents=True,exist_ok=True)
query_cache=cache_dir/(APPROVED_MANIFEST+'.npz')
api_started=time.perf_counter(); current_api_requests=0
assert os.environ.get('RUN_APPROVED_24_EXTERNAL')=='0', 'API recall is permanently disabled after approved cache creation'
assert query_cache.is_file(), 'approved exact query cache is required; API recall forbidden'
query_cache_sha_before=sha(query_cache)
with np.load(query_cache,allow_pickle=False) as z:
    query_vectors=z['embeddings']; cached_hashes=z['hashes'].tolist(); historical_requests=int(z['creation_api_requests']); historical_tokens=int(z['creation_input_tokens']); cached_manifest=str(z['manifest_sha256'])
assert cached_hashes==[x['sha256'] for x in items] and cached_manifest==APPROVED_MANIFEST and historical_requests==1 and historical_tokens==525
api_seconds=time.perf_counter()-api_started
assert query_vectors.shape==(10,1536) and query_vectors.dtype==np.float32 and np.isfinite(query_vectors).all()
embedding_usage={'approved_manifest_sha256':APPROVED_MANIFEST,'items':10,'planned_tokens':525,'actual_input_tokens':historical_tokens,'creation_api_requests_total':historical_requests,'current_run_api_requests':current_api_requests,'current_run_network_requests':current_api_requests,'cache_path':str(query_cache.relative_to(ROOT)),'cache_sha256':sha(query_cache),'dimension':1536,'dtype':'float32','finite':True,'api_seconds':api_seconds,'documents_or_gold_transmitted':False,'cost_usd':None,'cost_reason':'official price was not newly verified in this run'}
write_json(OUT/'embedding_usage.json',embedding_usage)

def cache_rows(paths):
    hashes=[]; matrices=[]; locations=[]
    for path in paths:
        with np.load(path,allow_pickle=False) as z:
            matrix=z['embeddings']; batch_hashes=z['hashes'].tolist()
            assert matrix.dtype==np.float32 and matrix.shape==(len(batch_hashes),1536) and np.isfinite(matrix).all()
            hashes.extend(batch_hashes); matrices.append(matrix.copy()); locations.extend((path.name,i) for i in range(len(batch_hashes)))
    return hashes,np.vstack(matrices),locations
old_hashes,old_matrix,old_locations=cache_rows(old_cache_files); new_hashes,new_matrix,new_locations=cache_rows(new_cache_files)
old=load_jsonl(OLD_CHUNKS); new=load_jsonl(NEW_CHUNKS)
assert len(old)==327 and len(new)==147
old_by_id={r['id']:r for r in old}; new_by_id={r['chunk_id']:r for r in new}
expected_old_hashes=[hashlib.sha256(r['document'].encode()).hexdigest() for r in old]
expected_new_hashes=[hashlib.sha256(r['retrieval_text'].encode()).hexdigest() for r in new]
assert len(old_hashes)==357 and old_hashes[:327]==expected_old_hashes and len(new_hashes)==147 and new_hashes==expected_new_hashes
old_doc_vectors={r['id']:old_matrix[i] for i,r in enumerate(old)}
new_doc_vectors={r['chunk_id']:new_matrix[i] for i,r in enumerate(new)}
assert len(old_doc_vectors)==327 and len(new_doc_vectors)==147
# Contract fixed before ranks: preserve historical batch/item position; never collapse equal text hashes.
occ=defaultdict(list)
for i,h in enumerate(old_hashes[:327]): occ[h].append(i)
duplicate_rows=[]
for h,indices in sorted(occ.items()):
    if len(indices)<2: continue
    base=old_matrix[indices[0]]; diffs=[old_matrix[i]-base for i in indices[1:]]
    duplicate_rows.append({'content_sha256':h,'occurrences':len(indices),'chunk_ids':[old[i]['id'] for i in indices],'cache_locations':[old_locations[i] for i in indices],'max_abs_diff':max(float(np.max(np.abs(x))) for x in diffs),'mean_abs_diff':float(np.mean(np.concatenate([np.abs(x) for x in diffs]))),'min_cosine':min(float(np.dot(base,old_matrix[i])/(np.linalg.norm(base)*np.linalg.norm(old_matrix[i]))) for i in indices[1:]),'max_l2':max(float(np.linalg.norm(x)) for x in diffs)})
assert len(duplicate_rows)==34 and sum(r['occurrences']-1 for r in duplicate_rows)==41
assert max(r['max_abs_diff'] for r in duplicate_rows)<=.002 and min(r['min_cosine'] for r in duplicate_rows)>=.9998
duplicate_contract={'root_cause':'old cache stores one vector per ordered input row; 34 exact-document groups were embedded as 75 separate rows and some repeated API outputs differ slightly','authoritative_selection':'embedding_usage.batches order, then positional zip of first 327 rows to chunks.jsonl order','hash_validation':'each positional cache hash must equal SHA256(chunk.document)','hash_collapse_forbidden':True,'diagnostic_tolerance_only':{'max_abs_diff_lte':.002,'min_cosine_gte':.9998},'tolerance_not_used_for_vector_selection_or_ranking':True,'duplicate_hash_groups':34,'duplicate_excess_rows':41,'observed_max_abs_diff':max(r['max_abs_diff'] for r in duplicate_rows),'observed_min_cosine':min(r['min_cosine'] for r in duplicate_rows)}
write_json(OUT/'embedding_duplicate_contract.json',duplicate_contract); write_json(OUT/'embedding_duplicate_diagnostic.json',{'contract':duplicate_contract,'groups':duplicate_rows})
query_vector_map={q:query_vectors[i] for i,q in enumerate(QUERIES)}

RAW_TOKEN=re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*',re.IGNORECASE)
def normalized_text(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
def canonical_decimal(value):
    rendered=format(Decimal(str(value).replace(',','')).normalize(),'f'); rendered=rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered
    return '0' if rendered in {'','-0'} else rendered
def search_tokens(value):
    text=normalized_text(value); tokens=list(RAW_TOKEN.findall(text))
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?',text):
        joined=run.replace(' ','')
        for size in (2,3,4): tokens.extend('ko'+str(size)+'_'+joined[i:i+size] for i in range(max(0,len(joined)-size+1)))
    consumed=[]
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원',text):
        tokens.append('money_krw_'+canonical_decimal(Decimal(match.group(1).replace(',',''))*10000+Decimal(match.group(2).replace(',',''))*1000)); consumed.append(match.span())
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원',text):
        if any(a<=match.start() and match.end()<=b for a,b in consumed): continue
        tokens.append('money_krw_'+canonical_decimal(Decimal(match.group(1).replace(',',''))*{'만':10000,'천':1000,None:1}[match.group(2)]))
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%',text): tokens.append('percent_'+canonical_decimal(match.group(1)))
    for match in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)',text): tokens.append('period_'+(match.group(1) or 'none')+'_'+canonical_decimal(match.group(2))+'_'+match.group(3))
    return tokens
def bm25_rank(query,documents,k1=1.5,b=.75):
    qt=search_tokens(query); docs={k:search_tokens(v) for k,v in documents.items()}; df=Counter(t for ts in docs.values() for t in set(ts)); avg=sum(map(len,docs.values()))/len(docs); scores={}
    for cid,tokens in docs.items():
        freq=Counter(tokens); score=0.0
        for token in qt:
            f=freq[token]
            if f:
                idf=math.log(1+(len(docs)-df[token]+.5)/(df[token]+.5)); score+=idf*f*(k1+1)/(f+k1*(1-b+b*len(tokens)/avg))
        scores[cid]=score
    return sorted(scores,key=lambda cid:(-scores[cid],cid)),scores
def vector_rank(qv,vectors):
    scores={cid:float(np.sum((v-qv)**2,dtype=np.float64)) for cid,v in vectors.items()}
    assert all(math.isfinite(x) for x in scores.values())
    return sorted(scores,key=lambda cid:(scores[cid],cid)),scores
def rrf(vector,bm25):
    score=defaultdict(float)
    for weight,ranked in ((.4,vector[:50]),(.6,bm25[:50])):
        for rank,cid in enumerate(ranked,1): score[cid]+=weight/(60+rank)
    fused=sorted(score,key=lambda cid:(-score[cid],cid))[:50]
    assert len(fused)==len(set(fused))==50 and set(fused)<=set(vector[:50])|set(bm25[:50])
    return fused,score

search_started=time.perf_counter(); cpu_started=time.process_time(); rss_before=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
ranking_rows=[]; classification=[]
PROPER=[('proper_issuer_product',re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$')),('proper_issuer_direct',re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$')),('proper_issuer_noun',re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$')),('proper_where_action',re.compile(r'어디서\s*(?:발급|발행|출시)'))]
NUMERIC=[('numeric_direct_amount',re.compile(r'얼마(?:인가|나)?$')),('numeric_direct_count',re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$')),('numeric_direct_how_much',re.compile(r'얼마나\s*(?:할인|적립|차감|청구)')),('numeric_target_end',re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$'))]
def classify(value):
    text=re.sub(r'[?!。？！.]+$','',normalized_text(value)).strip()
    for rid,p in PROPER:
        m=p.search(text)
        if m:return 'proper_noun',rid,m.group(0)
    if '연회비 면제 조건' not in text:
        for rid,p in NUMERIC:
            m=p.search(text)
            if m:return 'numeric_condition',rid,m.group(0)
    return 'semantic','semantic_fallback',''
old_docs={cid:r['document'] for cid,r in old_by_id.items()}; new_docs={cid:r['retrieval_text'] for cid,r in new_by_id.items()}
for q,query in QUERIES.items():
    cat,rid,span=classify(query); classification.append({'query_id':q,'category':cat,'matched_rule_id':rid,'matched_span':span,'route':'reranker' if cat=='semantic' else 'no_reranker'})
    for corpus,docs,vectors,by_id in [('old',old_docs,old_doc_vectors,old_by_id),('structural',new_docs,new_doc_vectors,new_by_id)]:
        vr,_=vector_rank(query_vector_map[q],vectors); br,_=bm25_rank(query,docs); fused,scores=rrf(vr,br)
        eligible=[cid for cid in fused if corpus=='structural' or by_id[cid]['metadata']['level'] in {'section','benefit'}]
        assert len(eligible)>=20
        d20=eligible[:20]; d10=d20[:10]; assert d10==d20[:10] and len(set(d20))==20
        ranking_rows.append({'corpus':corpus,'query_id':q,'vector_top50':vr[:50],'bm25_top50':br[:50],'rrf_top50':fused,'eligible_top20':d20,'eligible_top10':d10,'eligible_available_in_rrf_top50':len(eligible),'rrf_scores_top20':[scores[cid] for cid in d20]})
search_seconds=time.perf_counter()-search_started; search_cpu_seconds=time.process_time()-cpu_started; rss_after=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
assert len(ranking_rows)==20 and all(r['eligible_top10']==r['eligible_top20'][:10] for r in ranking_rows)
assert Counter(r['category'] for r in classification)=={'semantic':10}
first_vector_by_hash={}
for h,v in zip(old_hashes[:327],old_matrix[:327]): first_vector_by_hash.setdefault(h,v)
old_first_occurrence_vectors={r['id']:first_vector_by_hash[h] for r,h in zip(old,expected_old_hashes)}
rank_impact=[]
for q in QUERIES:
    authoritative=next(r for r in ranking_rows if r['corpus']=='old' and r['query_id']==q)
    alternate_vector,_=vector_rank(query_vector_map[q],old_first_occurrence_vectors); alternate_fused,_=rrf(alternate_vector,authoritative['bm25_top50'])
    alternate_eligible=[cid for cid in alternate_fused if old_by_id[cid]['metadata']['level'] in {'section','benefit'}]
    rank_impact.append({'query_id':q,'vector_top50_exact':alternate_vector[:50]==authoritative['vector_top50'],'vector_top50_symmetric_difference':len(set(alternate_vector[:50])^set(authoritative['vector_top50'])),'rrf_top50_exact':alternate_fused==authoritative['rrf_top50'],'rrf_top50_symmetric_difference':len(set(alternate_fused)^set(authoritative['rrf_top50'])),'eligible_top10_exact':alternate_eligible[:10]==authoritative['eligible_top10'],'eligible_top20_exact':alternate_eligible[:20]==authoritative['eligible_top20']})
write_json(OUT/'embedding_duplicate_rank_impact.json',{'authoritative_policy':duplicate_contract['authoritative_selection'],'alternate_diagnostic_only':'first vector per content hash','queries':rank_impact,'counts':{key:sum(r[key] for r in rank_impact) for key in ['vector_top50_exact','rrf_top50_exact','eligible_top10_exact','eligible_top20_exact']}})
freeze_path=OUT/'ranking_freeze.jsonl'
freeze_path.write_text(''.join(canonical(r)+'\n' for r in ranking_rows),encoding='utf-8')
freeze={'created_before_gold_read':True,'gold_files_read_in_this_kernel_before_freeze':0,'ranking_rows':20,'ranking_sha256':sha(freeze_path),'query_manifest_sha256':APPROVED_MANIFEST,'top10_exact_prefix_top20':True,'search_contract':{'bm25_k1':1.5,'bm25_b':.75,'vector_weight':.4,'bm25_weight':.6,'rrf_k':60,'component_depth':50,'vector_distance':'numpy_squared_l2','tie':'chunk_id lexical'},'immutable_inputs_before':immutable_before,'bge_before':bge_before}
write_json(OUT/'ranking_freeze.json',freeze)
with (OUT/'query_classification.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(classification[0])); w.writeheader(); w.writerows(classification)
write_json(OUT/'search_resources.json',{'wall_seconds':search_seconds,'cpu_seconds':search_cpu_seconds,'peak_rss_mib':rss_after/1024,'incremental_ru_maxrss_mib':max(0,rss_after-rss_before)/1024,'api_seconds':api_seconds,'api_requests_current_run':current_api_requests})
assert {str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}==immutable_before and tree_state(BGE_ROOT)==bge_before and sha(query_cache)==query_cache_sha_before
print({'ranking_freeze':'PASS','api_requests':current_api_requests,'api_tokens':historical_tokens,'ranking_rows':20,'all_queries_route':'semantic'})

{'ranking_freeze': 'PASS', 'api_requests': 0, 'api_tokens': 525, 'ranking_rows': 20, 'all_queries_route': 'semantic'}


In [2]:
assert os.environ.get('RUN_APPROVED_24_GPU')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0'
import gc, statistics, subprocess
def gpu_snapshot():
    gpu=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip().splitlines()
    line=next(x for x in gpu if x.split(',')[0].strip()=='0'); uuid=line.split(',')[1].strip()
    raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip()
    processes=[]
    for x in raw.splitlines():
        parts=[v.strip() for v in x.split(',',3)]
        if len(parts)==4 and parts[0]==uuid: processes.append({'gpu_uuid':parts[0],'pid':int(parts[1]),'process_name':parts[2],'used_memory_mib':float(parts[3])})
    return {'gpu_line':line,'processes':processes,'captured_at_unix':time.time()}
rankings=[json.loads(x) for x in (OUT/'ranking_freeze.jsonl').read_text(encoding='utf-8').splitlines() if x]
assert len(rankings)==20
heading_re=re.compile(r'(?m)^#{1,6}\s+(.+?)\s*$')
def old_augmented(chunk):
    meta=chunk['metadata']; m=heading_re.search(chunk['document']); heading=m.group(1).strip() if m else ''
    body=chunk['document'][:m.start()]+chunk['document'][m.end():] if m else chunk['document']
    path=[meta['issuer'],meta['card_name'],meta['level']]+([heading] if heading else [])
    return '[문서 경로]\n'+' > '.join(path)+'\n\n[본문]\n'+body.strip(),path
def new_augmented(chunk):
    meta=chunk['metadata']; path=[meta['issuer'],meta['card_name'],*chunk['heading_path']]
    return '[문서 경로]\n'+' > '.join(path)+'\n\n[본문]\n'+chunk['body'],path
pairs=[]; audit=[]
for row in rankings:
    query=QUERIES[row['query_id']]
    for rank,cid in enumerate(row['eligible_top20'],1):
        text,path=(old_augmented(old_by_id[cid]) if row['corpus']=='old' else new_augmented(new_by_id[cid]))
        pairs.append((row['corpus'],row['query_id'],cid,rank,query,text))
        audit.append({'corpus':row['corpus'],'query_id':row['query_id'],'chunk_id':cid,'original_rrf_rank':rank,'query_sha256':hashlib.sha256(query.encode()).hexdigest(),'document_sha256':hashlib.sha256(text.encode()).hexdigest(),'artificial_path_json':json.dumps(path,ensure_ascii=False),'construction_allowlist':'query_text + issuer + card_name + level/heading_path + body/document','gold_fields_used':False})
assert len(pairs)==400 and len({x[:3] for x in pairs})==400 and all(not r['gold_fields_used'] for r in audit)
gpu_before=gpu_snapshot()
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
def score_all(batch_size):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); load_start=time.perf_counter()
    tokenizer=AutoTokenizer.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False)
    model=AutoModelForSequenceClassification.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0')
    load_seconds=time.perf_counter()-load_start
    token_rows=[]
    for corpus,q,cid,rank,query,doc in pairs:
        qn=len(tokenizer.encode(query,add_special_tokens=False)); dn=len(tokenizer.encode(doc,add_special_tokens=False))
        encoded=tokenizer(query,doc,truncation='only_second',max_length=8192)
        token_rows.append((qn,dn,len(encoded['input_ids']),max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192)))
    warm=tokenizer(['준비'],['준비'],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0'); warm_start=time.perf_counter()
    with torch.inference_mode(): assert torch.isfinite(model(**warm).logits).all()
    torch.cuda.synchronize(); warm_seconds=time.perf_counter()-warm_start; del warm
    scores=[]; started=time.perf_counter()
    try:
        for start in range(0,len(pairs),batch_size):
            batch=pairs[start:start+batch_size]
            inputs=tokenizer([x[4] for x in batch],[x[5] for x in batch],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
            with torch.inference_mode(): logits=model(**inputs).logits.reshape(-1).float().cpu().numpy()
            assert len(logits)==len(batch) and np.isfinite(logits).all(); scores.extend(map(float,logits)); del inputs,logits
        torch.cuda.synchronize(); seconds=time.perf_counter()-started
        result=(scores,token_rows,{'load_seconds':load_seconds,'warmup_seconds':warm_seconds,'scoring_seconds':seconds,'pairs_per_second':len(pairs)/seconds,'batch_size':batch_size,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30})
    except torch.cuda.OutOfMemoryError:
        result=None
    del model,tokenizer,scores; gc.collect(); torch.cuda.empty_cache()
    return result
scored=score_all(2); oom_batch2=scored is None
if scored is None: scored=score_all(1)
assert scored is not None
scores,token_rows,model_resource=scored; assert len(scores)==400 and np.isfinite(np.asarray(scores)).all()
pair_rows=[]
for pair,score,tokens in zip(pairs,scores,token_rows):
    pair_rows.append({'corpus':pair[0],'query_id':pair[1],'chunk_id':pair[2],'original_rrf_rank':pair[3],'raw_logit':score,'query_original_tokens':tokens[0],'document_original_tokens':tokens[1],'input_tokens':tokens[2],'document_truncated_tokens':tokens[3]})
with (OUT/'pair_scores.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(pair_rows[0])); w.writeheader(); w.writerows(pair_rows)
with (OUT/'scorer_input_audit.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0])); w.writeheader(); w.writerows(audit)
gpu_after=gpu_snapshot(); input_lengths=[r['input_tokens'] for r in pair_rows]
resources={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'dtype':'float16','max_length':8192,'truncation':'only_second','quality_pairs':400,'oom_batch2':oom_batch2,**model_resource,'token_p50':statistics.median(input_lengths),'token_p95':float(np.percentile(input_lengths,95)),'token_max':max(input_lengths),'truncated_pairs':sum(r['document_truncated_tokens']>0 for r in pair_rows),'gpu_before':gpu_before,'gpu_after':gpu_after,'physical_gpu':0,'shared_gpu_measurement':True,'network_api_download_new_embedding_chroma_during_scoring':0}
write_json(OUT/'bge_resources.json',resources)
assert tree_state(BGE_ROOT)==bge_before and {str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}==immutable_before
print({'bge_pairs':400,'batch':model_resource['batch_size'],'oom_batch2':oom_batch2,'seconds':model_resource['scoring_seconds'],'peak_gib':model_resource['peak_allocated_gib']})

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'bge_pairs': 400, 'batch': 2, 'oom_batch2': False, 'seconds': 5.095208915183321, 'peak_gib': 1.122520923614502}


In [3]:
# Gold is first read here, after ranking_freeze.jsonl was written and hashed.
gold_freeze=json.loads((OUT/'gold_freeze.json').read_text(encoding='utf-8'))
assert gold_freeze['frozen_before_any_ranking'] and all(sha(OUT/name)==digest for name,digest in gold_freeze['files'].items())
with (OUT/'gold_card_labels.csv').open(encoding='utf-8',newline='') as f: labels=list(csv.DictReader(f))
with (OUT/'atomic_claims.csv').open(encoding='utf-8',newline='') as f: claims=list(csv.DictReader(f))
with (OUT/'queries.csv').open(encoding='utf-8',newline='') as f: stored_queries=list(csv.DictReader(f))
assert {r['query_id']:r['query_text'] for r in stored_queries}==QUERIES
assert len(labels)==100 and Counter(r['label'] for r in labels)=={'negative':67,'positive':33} and len(claims)==33
positives=defaultdict(set)
for r in labels:
    if r['label']=='positive': positives[r['query_id']].add(r['card_id'])
card_key_to_id={r['card_key']:r['card_id'] for r in labels}
terms={(r['query_id'],r['card_id']):json.loads(r['predicate_terms_json']) for r in claims}
score_rows=list(csv.DictReader((OUT/'pair_scores.csv').open(encoding='utf-8',newline='')))
assert len(score_rows)==400 and len({(r['corpus'],r['query_id'],r['chunk_id']) for r in score_rows})==400 and all(math.isfinite(float(r['raw_logit'])) for r in score_rows)
score_map={(r['corpus'],r['query_id'],r['chunk_id']):float(r['raw_logit']) for r in score_rows}
rank_by_key={(r['corpus'],r['query_id']):r for r in rankings}
class_by_q={r['query_id']:r for r in classification}
def card_and_text(corpus,cid):
    row=old_by_id[cid] if corpus=='old' else new_by_id[cid]
    return card_key_to_id[row['metadata']['card_key']], row['document'] if corpus=='old' else row['evidence_text']
def rerank(corpus,q,candidates,use_bge):
    return list(candidates) if not use_bge else sorted(candidates,key=lambda cid:(-score_map[(corpus,q,cid)],candidates.index(cid),cid))
def collapse(corpus,q,ordered,k):
    out=[]; seen=set()
    for cid in ordered:
        card,text=card_and_text(corpus,cid)
        if card in seen: continue
        seen.add(card); supported=card in positives[q] and all(normalized_text(t) in normalized_text(text) for t in terms[(q,card)])
        out.append({'card_id':card,'chunk_id':cid,'supported':supported})
        if len(out)==k: break
    return out
def metrics(q,collapsed,k3=3,k5=5):
    pos=positives[q]; first3=collapsed[:3]; first5=collapsed[:5]
    card3=sum(x['card_id'] in pos for x in first3); card5=sum(x['card_id'] in pos for x in first5); ev3=sum(x['supported'] for x in first3); ev5=sum(x['supported'] for x in first5)
    return {'card_precision_at_3':card3/k3,'card_recall_at_3':card3/len(pos),'card_precision_at_5':card5/k5,'card_recall_at_5':card5/len(pos),'evidence_supported_card_recall_at_5':ev5/len(pos),'evidence_accuracy_at_3':ev3/k3,'evidence_accuracy_at_5':ev5/k5,'card_zero_hit_at_5':int(card5==0),'supported_zero_hit_at_5':int(ev5==0)}
configs=[
 ('old_no_d10','old',10,'no_reranker',False,False,True),('old_selective_bge_d10','old',10,'selective_bge',True,True,True),('old_all_bge_d10','old',10,'all_bge',True,False,True),('old_selective_bge_d20','old',20,'selective_bge',True,True,False),
 ('structural_no_d10','structural',10,'no_reranker',False,False,True),('structural_selective_bge_d10','structural',10,'selective_bge',True,False,True),('structural_all_bge_d10','structural',10,'all_bge',True,True,True),('structural_all_bge_d20','structural',20,'all_bge',True,True,False)]
per_query=[]; ceiling=[]
for name,corpus,depth,system,bge,primary,diagnostic in configs:
    for q in QUERIES:
        base=rank_by_key[(corpus,q)]['eligible_top20'][:depth]; use=bge and (system=='all_bge' or class_by_q[q]['route']=='reranker')
        ordered=rerank(corpus,q,base,use); assert set(ordered)==set(base) and len(ordered)==len(base)==depth
        collapsed=collapse(corpus,q,ordered,5); result=metrics(q,collapsed)
        per_query.append({'configuration':name,'corpus':corpus,'candidate_depth':depth,'system':system,'primary':primary,'diagnostic_top10':diagnostic,'query_id':q,'route':'reranker' if use else 'no_reranker',**result,'raw_duplicate_count_top5':5-len({card_and_text(corpus,c)[0] for c in ordered[:5]}),'output_card_ids_json':json.dumps([x['card_id'] for x in collapsed],ensure_ascii=False),'representative_chunk_ids_json':json.dumps([x['chunk_id'] for x in collapsed],ensure_ascii=False),'supported_json':json.dumps([x['supported'] for x in collapsed])})
        if name in {'old_no_d10','old_selective_bge_d20','structural_no_d10','structural_all_bge_d20'}:
            base_collapsed=collapse(corpus,q,base,10)
            ceiling.append({'corpus':corpus,'candidate_depth':depth,'query_id':q,'candidate_card_recall':sum(x['card_id'] in positives[q] for x in base_collapsed)/len(positives[q]),'candidate_evidence_recall':sum(x['supported'] for x in base_collapsed)/len(positives[q]),'unique_cards':len(base_collapsed)})
assert len(per_query)==80 and all(len(json.loads(r['output_card_ids_json']))==len(set(json.loads(r['output_card_ids_json']))) for r in per_query)
metric_names=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','card_zero_hit_at_5','supported_zero_hit_at_5','raw_duplicate_count_top5']
summary=[]
for name,*_ in configs:
    rows=[r for r in per_query if r['configuration']==name]
    summary.append({'configuration':name,'corpus':rows[0]['corpus'],'candidate_depth':rows[0]['candidate_depth'],'system':rows[0]['system'],'primary':rows[0]['primary'],'diagnostic_top10':rows[0]['diagnostic_top10'],'denominator':10,**{m:sum(float(r[m]) for r in rows)/10 for m in metric_names}})
summary_by={r['configuration']:r for r in summary}
paired=[]; wlt=[]
for depth in (10,20):
    base='old_selective_bge_d'+str(depth); cand='structural_all_bge_d'+str(depth)
    bq={r['query_id']:r for r in per_query if r['configuration']==base}; cq={r['query_id']:r for r in per_query if r['configuration']==cand}
    for q in QUERIES: paired.append({'candidate_depth':depth,'query_id':q,**{'delta_'+m:float(cq[q][m])-float(bq[q][m]) for m in metric_names}})
    for m in metric_names:
        vals=[r['delta_'+m] for r in paired if r['candidate_depth']==depth]; wins=sum(v>1e-12 for v in vals); losses=sum(v< -1e-12 for v in vals)
        wlt.append({'candidate_depth':depth,'metric':m,'denominator':10,'wins':wins,'losses':losses,'ties':10-wins-losses,'mean_delta':sum(vals)/10})
assert all(r['wins']+r['losses']+r['ties']==10 for r in wlt)
gates=[]
for depth in (10,20):
    b=summary_by['old_selective_bge_d'+str(depth)]; c=summary_by['structural_all_bge_d'+str(depth)]
    sr_wlt=next(r for r in wlt if r['candidate_depth']==depth and r['metric']=='evidence_supported_card_recall_at_5')
    catastrophe=sum(1 for r in paired if r['candidate_depth']==depth and (next(x for x in per_query if x['configuration']=='old_selective_bge_d'+str(depth) and x['query_id']==r['query_id'])['evidence_supported_card_recall_at_5']>0) and (next(x for x in per_query if x['configuration']=='structural_all_bge_d'+str(depth) and x['query_id']==r['query_id'])['evidence_supported_card_recall_at_5']==0 or r['delta_evidence_supported_card_recall_at_5']<=-.5))
    checks={'supported_recall5_strict':c['evidence_supported_card_recall_at_5']>b['evidence_supported_card_recall_at_5']+1e-12,'wins_gt_losses':sr_wlt['wins']>sr_wlt['losses'],'precision5_nonreg':c['card_precision_at_5']>=b['card_precision_at_5']-1e-12,'evidence_accuracy5_nonreg':c['evidence_accuracy_at_5']>=b['evidence_accuracy_at_5']-1e-12,'recall3_nonreg':c['card_recall_at_3']>=b['card_recall_at_3']-1e-12,'zero_hit_nonreg':c['supported_zero_hit_at_5']<=b['supported_zero_hit_at_5']+1e-12,'catastrophic_loss_zero':catastrophe==0}
    gates.append({'candidate_depth':depth,'checks':checks,'catastrophic_loss_count':catastrophe,'gate_pass':all(checks.values())})
depth20_improves=(summary_by['structural_all_bge_d20']['evidence_supported_card_recall_at_5']>summary_by['structural_all_bge_d10']['evidence_supported_card_recall_at_5']+1e-12 or summary_by['structural_all_bge_d20']['evidence_accuracy_at_5']>summary_by['structural_all_bge_d10']['evidence_accuracy_at_5']+1e-12)
eligible=[10] if gates[0]['gate_pass'] else []
if gates[1]['gate_pass'] and depth20_improves: eligible.append(20)
if eligible:
    selected=min(eligible,key=lambda d:(-summary_by['structural_all_bge_d'+str(d)]['evidence_accuracy_at_5'],-summary_by['structural_all_bge_d'+str(d)]['card_precision_at_5'],summary_by['structural_all_bge_d'+str(d)]['raw_duplicate_count_top5'],d)); disposition='dev_signal_only_structural_candidate_d'+str(selected)
else: selected=None; disposition='retain_old_selective_bge_development_baseline'
decision={'predeclared_gate_unchanged':True,'depth_gates':gates,'depth20_requires_strict_depth_improvement':True,'depth20_improves':depth20_improves,'selected_depth':selected,'disposition':disposition,'promotion_eligible':False,'scope':'fixed_10_document_development_signal_only'}
def write_csv(path,rows):
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
write_csv(OUT/'per_query_metrics.csv',per_query); write_csv(OUT/'summary.csv',summary); write_csv(OUT/'paired_deltas.csv',paired); write_csv(OUT/'paired_wlt.csv',wlt); write_csv(OUT/'candidate_ceiling.csv',ceiling)
write_json(OUT/'summary.json',{'summary':summary,'decision':decision,'metric_guide_ko':{'Card Precision':'반환 슬롯 중 정답 카드 비율; 빈 슬롯은 실패','Card Recall':'정답 카드 중 반환한 비율','Evidence-supported Card Recall':'대표 청크 하나가 모든 hard predicate를 증명한 정답 카드 회수율','Evidence Accuracy':'반환 슬롯 중 카드와 대표 근거가 모두 맞는 비율','zero-hit':'근거가 확인된 정답 카드를 하나도 못 찾은 질의'}})
write_json(OUT/'selection_decision.json',decision)
immutable_after={str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}; bge_after=tree_state(BGE_ROOT)
assert immutable_after==immutable_before and bge_after==bge_before and sha(query_cache)==query_cache_sha_before and current_api_requests==0
output_names=['external_execution_approval.json','embedding_usage.json','embedding_duplicate_contract.json','embedding_duplicate_diagnostic.json','embedding_duplicate_rank_impact.json','ranking_freeze.jsonl','ranking_freeze.json','query_classification.csv','search_resources.json','pair_scores.csv','scorer_input_audit.csv','bge_resources.json','per_query_metrics.csv','summary.csv','paired_deltas.csv','paired_wlt.csv','candidate_ceiling.csv','summary.json','selection_decision.json']
integrity={'status':'PASS','fresh_kernel_phase2_cells_only':True,'preflight_cells_skipped':True,'query_count':10,'card_count':10,'labels':100,'positive':33,'negative':67,'insufficient':0,'ranking_created_before_gold_read':True,'ranking_rows':20,'top10_prefix_top20':True,'pair_score_rows':400,'pair_scores_finite_unique':True,'per_query_rows':80,'summary_rows':8,'paired_rows':20,'wlt_rows':20,'candidate_ceiling_rows':len(ceiling),'unique_output_cards':True,'q01_c03_structural_single_chunk_strict':True,'source_inputs_unchanged':True,'bge_cache_unchanged':True,'query_cache_unchanged_during_rerun':True,'api_requests_current_run':0,'api_network_hard_disabled_after_cache_creation':True,'api_input_tokens_historical':historical_tokens,'embedding_cache_count':10,'old_embedding_authoritative_policy':duplicate_contract['authoritative_selection'],'old_duplicate_hash_groups':34,'old_duplicate_excess_rows':41,'gpu_physical':0,'model_revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'chroma_queries':0,'package_installs':0,'output_hashes':{n:sha(OUT/n) for n in output_names}}
write_json(OUT/'integrity.json',integrity)
readme=(OUT/'README.md').read_text(encoding='utf-8'); marker='## 승인 후 전체 실행 결과'
if marker in readme: readme=readme.split(marker)[0].rstrip()
readme+=f'\n\n{marker}\n\n질의 10개 embedding만 승인 manifest에 따라 전송했고, 문서·gold·평가 필드는 전송하지 않았습니다. old/new Top10·20 검색과 local BGE 400쌍 평가를 완료했습니다. 선택 결과는 `{disposition}`이며 개발 신호일 뿐 운영·holdout 승격이 아닙니다. Q01_C03 structural은 부모/자식으로 조건이 나뉘어 대표 단일 청크가 두 predicate를 모두 포함하지 않으면 엄격히 evidence 실패로 처리했습니다.\n'
(OUT/'README.md').write_text(readme+'\n',encoding='utf-8')
manifest_outputs=sorted(p for p in OUT.iterdir() if p.is_file() and p.name not in {'run_manifest.json','integrity.json'})
write_json(OUT/'run_manifest.json',{'phase':'approved_embedding_search_bge_evaluation','notebook_execution':'fresh kernel, skip-preflight-rerun tagged cells skipped','inputs_before':immutable_before,'inputs_after':immutable_after,'bge_before':bge_before,'bge_after':bge_after,'outputs':{p.name:sha(p) for p in manifest_outputs},'manifest_self_hash_excluded':True})
integrity['run_manifest_sha256']=sha(OUT/'run_manifest.json'); write_json(OUT/'integrity.json',integrity)
print(json.dumps({'status':'PASS','decision':decision,'primary_summary':[r for r in summary if r['primary']],'api_requests':current_api_requests,'api_tokens':historical_tokens,'bge_pairs':400},ensure_ascii=False,indent=2))

{
  "status": "PASS",
  "decision": {
    "predeclared_gate_unchanged": true,
    "depth_gates": [
      {
        "candidate_depth": 10,
        "checks": {
          "supported_recall5_strict": true,
          "wins_gt_losses": false,
          "precision5_nonreg": false,
          "evidence_accuracy5_nonreg": true,
          "recall3_nonreg": false,
          "zero_hit_nonreg": true,
          "catastrophic_loss_zero": false
        },
        "catastrophic_loss_count": 1,
        "gate_pass": false
      },
      {
        "candidate_depth": 20,
        "checks": {
          "supported_recall5_strict": true,
          "wins_gt_losses": true,
          "precision5_nonreg": true,
          "evidence_accuracy5_nonreg": true,
          "recall3_nonreg": false,
          "zero_hit_nonreg": true,
          "catastrophic_loss_zero": false
        },
        "catastrophic_loss_count": 1,
        "gate_pass": false
      }
    ],
    "depth20_requires_strict_depth_improvement": true,
    "d

In [1]:
# Offline-only finalization: no embedding, model, GPU, network, or ranking recomputation.
from pathlib import Path
import ast, csv, hashlib, json, math, os
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
assert (ROOT/'notebooks').is_dir() and os.environ.get('RUN_APPROVED_24_EXTERNAL')=='0' and os.environ.get('CUDA_VISIBLE_DEVICES','')==''
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
history=[
 {'sequence':1,'stage':'cpu_offline_preflight','status':'success','api_requests':0,'embedding_items_created':0,'bge_pairs_scored':0},
 {'sequence':2,'stage':'approved_query_embedding_then_old_cache_loader','status':'failed_after_cache_creation','api_requests':1,'api_input_tokens':525,'embedding_items_created':10,'query_cache_sha256':'64102d65421966710dd47b06ebeb64c5279c220175c68b782070a20a8255bfb1','bge_pairs_scored':0,'failure':'incorrect np.array_equal assertion collapsed duplicate content hashes whose historical positional vectors differ slightly','api_wall_seconds':'not preserved because the failed notebook was not saved'},
 {'sequence':3,'stage':'authoritative_positional_cache_search_bge_evaluation','status':'success','api_requests':0,'api_input_tokens_historical':525,'embedding_items_reused':10,'bge_pairs_scored':400,'batch_size':2,'oom':False},
 {'sequence':4,'stage':'offline_finalization_validation','status':'success','api_requests':0,'bge_pairs_scored':0,'gpu_used':False}]
usage=json.loads((OUT/'embedding_usage.json').read_text()); assert usage['current_run_api_requests']==usage['current_run_network_requests']==0 and usage['creation_api_requests_total']==1 and usage['actual_input_tokens']==525
resources=json.loads((OUT/'bge_resources.json').read_text()); assert resources['quality_pairs']==400 and not resources['oom_batch2'] and resources['truncated_pairs']==0
resources['execution_history']=history; resources['finalization_model_or_gpu_rerun']=False; write_json(OUT/'bge_resources.json',resources)
search=json.loads((OUT/'search_resources.json').read_text()); search['execution_history']=history; search['finalization_api_network_requests']=0; write_json(OUT/'search_resources.json',search)
readme=(OUT/'README.md').read_text(encoding='utf-8'); marker='## 실행 이력'
if marker in readme: readme=readme.split(marker)[0].rstrip()
readme+=f'\n\n{marker}\n\n1. CPU/offline preflight 성공(API 0, GPU 0).\n2. 승인된 query embedding 1회/525 tokens로 10×1536 cache를 만든 뒤, old duplicate content hash를 잘못 하나로 합치는 loader assertion에서 실패(BGE 0).\n3. historical batch/item 위치를 authoritative source로 고친 최종 실행은 API 0/cache 재사용, BGE 400쌍 성공(batch2, OOM 없음).\n4. 이 finalization은 CPU/offline이며 API·GPU·모델 재실행 0입니다.\n'
(OUT/'README.md').write_text(readme+'\n',encoding='utf-8')
integrity=json.loads((OUT/'integrity.json').read_text()); integrity['execution_history']=history; integrity['offline_finalization_pass']=True; integrity['finalization_api_network_gpu_model']=0
for name in list(integrity['output_hashes']): integrity['output_hashes'][name]=sha(OUT/name)
manifest=json.loads((OUT/'run_manifest.json').read_text()); manifest['execution_history']=history; manifest['notebook_sha256']='FINALIZED_AFTER_NOTEBOOK_SAVE'; manifest['notebook_hash_status']='pending external raw hash finalization after this cell output is saved'
manifest['outputs']={p.name:sha(p) for p in sorted(OUT.iterdir()) if p.is_file() and p.name not in {'run_manifest.json','integrity.json'}}
write_json(OUT/'run_manifest.json',manifest); integrity['run_manifest_sha256']=sha(OUT/'run_manifest.json'); write_json(OUT/'integrity.json',integrity)
assert all(sha(OUT/name)==digest for name,digest in integrity['output_hashes'].items())
print({'offline_finalization':'PASS','api_network_gpu_model':0,'history_entries':len(history),'notebook_hash':'pending post-save finalization'})

{'offline_finalization': 'PASS', 'api_network_gpu_model': 0, 'history_entries': 4, 'notebook_hash': 'pending post-save finalization'}


## Follow-up 1 — structural D20 operational card evidence bundle

이 실험은 structural D20에 이미 포함된 카드만 metadata 관계로 묶는 adaptive development 진단입니다. D10은 Q03 positive card 자체가 후보 밖이어서 제외합니다. Bundle은 새 카드를 추가하지 않으며 gold를 선택에 사용하지 않습니다. 결과가 좋아도 운영·holdout 승격이 아니라 후속 검증 신호입니다.

In [1]:
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, os, re, resource, time
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
assert (ROOT/'notebooks').is_dir() and os.environ.get('RUN_APPROVED_24_EXTERNAL')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'
HIERARCHY=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'
RANKING=OUT/'ranking_freeze.jsonl'; QUERIES=OUT/'queries.csv'; BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def canonical(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def tree_state(root):
    files=sorted(p for p in root.rglob('*') if p.is_file()); mapping={str(p.relative_to(root)):sha(p) for p in files}
    return {'file_count':len(files),'total_bytes':sum(p.stat().st_size for p in files),'files':mapping,'digest':hashlib.sha256(canonical(mapping).encode()).hexdigest()}
source_paths=[CHUNKS,HIERARCHY,RANKING,QUERIES]; source_before={str(p.relative_to(ROOT)):sha(p) for p in source_paths}; bge_before_f1=tree_state(BGE_ROOT)
base_output_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup1_'))
contract={'declared_before_results':True,'name':'structural D20 operational card evidence bundle','scope':'adaptive fixed-10-document development diagnostic','candidate_source':'structural eligible_top20 only','candidate_card_set':'exact unique cards in each query Top20; no new cards','depth':20,'d10_excluded_reason':'Q03 positive card is outside structural D10 candidate set, so bundling cannot recover it','max_unique_chunks_per_bundle':5,'bundle_document_token_cap':4096,'automatic_text_truncation':False,'relation_order':['best_rrf_seed','same_node_adjacent_parts','non_root_immediate_parent_direct_body','heading_only_parent_context_and_same_parent_direct_body_children_by_heading_distance_then_earlier','seed_node_direct_children','same_card_other_top20_seeds_rrf_order'],'forbidden':['root_fanout','recursive_descendants','cross_card','whole_document','gold_driven_choice','retrieval_text repetition','score_mixing'],'scorer_text':'issuer/card once; optional heading-only parent context once; each selected evidence heading_path plus evidence_text','ranking':'raw BGE logit desc; tie best seed RRF rank then card_key','bge':{'revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'batch_size':2,'dtype':'float16','max_length':8192,'truncation':'only_second','physical_gpu':0},'evaluation':{'primary_depth':20,'systems':['old_selective_bge_d20','structural_all_bge_d20_single_chunk','structural_card_bundle_bge_d20'],'bundle_support_name':'bundle-supported','single_support_name':'single-representative-supported','causal_claim_forbidden':True},'gate':{'card_recall_at_3_min':.62,'card_precision_at_5_min':.48,'card_recall_at_5_min':.7933,'bundle_supported_card_recall_at_5_min':.6433,'bundle_evidence_accuracy_at_5_min':.40,'catastrophic_loss_count':0,'supported_wins_gt_losses':True},'result_ceiling':'adaptive_development_signal_only','execution':{'api':0,'network':0,'new_embedding':0,'chroma':0,'package_install':0}}
write_json(OUT/'followup1_contract.json',contract)
chunks=[json.loads(x) for x in CHUNKS.read_text(encoding='utf-8').splitlines() if x]; hierarchy=[json.loads(x) for x in HIERARCHY.read_text(encoding='utf-8').splitlines() if x]
chunk_by_id={r['chunk_id']:r for r in chunks}; node_by_id={r['node_id']:r for r in hierarchy}; children=defaultdict(list)
for node in hierarchy:
    if node['parent_id'] is not None: children[node['parent_id']].append(node)
for rows in children.values(): rows.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
rankings=[json.loads(x) for x in RANKING.read_text(encoding='utf-8').splitlines() if x and json.loads(x)['corpus']=='structural']; assert len(rankings)==10
with QUERIES.open(encoding='utf-8',newline='') as f: query_map={r['query_id']:r['query_text'] for r in csv.DictReader(f)}
assert len(query_map)==10
build_started=time.perf_counter(); cpu_started=time.process_time(); rss_started=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
bundles=[]; traces=[]
for ranking in rankings:
    top20=ranking['eligible_top20']; by_card=defaultdict(list)
    for rrf_rank,cid in enumerate(top20,1): by_card[chunk_by_id[cid]['metadata']['card_key']].append((rrf_rank,cid))
    assert len(by_card)==len({chunk_by_id[c]['metadata']['card_key'] for c in top20})
    for card_key,seeds in sorted(by_card.items(),key=lambda x:(x[1][0][0],x[0])):
        best_rank,seed_id=seeds[0]; seed=chunk_by_id[seed_id]; seed_node=node_by_id[seed['metadata']['node_id']]; selected=[]; relation=[]; parent_context=None
        def add(cid,kind,source_node):
            if len(selected)>=5 or cid in selected: return
            candidate=chunk_by_id[cid]; assert candidate['metadata']['card_key']==card_key
            selected.append(cid); relation.append({'selection_order':len(selected),'chunk_id':cid,'relation':kind,'node_id':candidate['metadata']['node_id'],'parent_id':candidate['metadata']['parent_id'],'part_index':candidate['metadata']['part_index'],'part_count':candidate['metadata']['part_count'],'source_relation_node_id':source_node})
        add(seed_id,'best_rrf_seed',seed_node['node_id'])
        same_node=[chunk_by_id[c] for c in seed_node['search_chunk_ids'] if c!=seed_id]
        same_node.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
        for c in same_node: add(c['chunk_id'],'same_node_adjacent_part',seed_node['node_id'])
        parent=node_by_id.get(seed_node['parent_id'])
        if parent is not None and parent['parent_id'] is not None:
            if not parent['heading_only']:
                for cid in parent['search_chunk_ids']: add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
            else:
                parent_context=parent['heading_text']
                seed_line=seed_node['heading_line_number'] if seed_node['heading_line_number'] is not None else 10**12
                siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']]
                siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-seed_line),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                for n in siblings:
                    for cid in n['search_chunk_ids']: add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
        for child in children[seed_node['node_id']]:
            if not child['heading_only']:
                for cid in child['search_chunk_ids']: add(cid,'seed_node_direct_child',seed_node['node_id'])
        for _,cid in seeds[1:]: add(cid,'same_card_other_top20_seed',seed_node['node_id'])
        assert 1<=len(selected)<=5 and len(selected)==len(set(selected)) and all(chunk_by_id[c]['metadata']['card_key']==card_key for c in selected)
        path_header='[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']; sections=[path_header]
        if parent_context: sections.append('[상위 제목]\n'+parent_context)
        for i,cid in enumerate(selected,1):
            c=chunk_by_id[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'
            sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
        bundle_text='\n\n'.join(sections)
        row={'query_id':ranking['query_id'],'card_key':card_key,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'candidate_top20_card_set':sorted(by_card),'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':bundle_text,'bundle_sha256':hashlib.sha256(bundle_text.encode()).hexdigest(),'bundle_characters':len(bundle_text),'relation_trace':relation}
        bundles.append(row)
        for item in relation: traces.append({'query_id':ranking['query_id'],'card_key':card_key,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'optional_parent_heading':parent_context or '',**item})
assert len(bundles)==69 and len({(r['query_id'],r['card_key']) for r in bundles})==69
for ranking in rankings:
    expected={chunk_by_id[c]['metadata']['card_key'] for c in ranking['eligible_top20']}; actual={r['card_key'] for r in bundles if r['query_id']==ranking['query_id']}; assert actual==expected
bundle_path=OUT/'followup1_bundles.jsonl'; bundle_path.write_text(''.join(canonical(r)+'\n' for r in bundles),encoding='utf-8')
with (OUT/'followup1_bundle_trace.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(traces[0])); w.writeheader(); w.writerows(traces)
build_wall=time.perf_counter()-build_started; build_cpu=time.process_time()-cpu_started; rss_end=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
build_freeze={'created_before_gold_read':True,'prohibited_gold_metric_files_read_before_build':0,'query_card_pairs':69,'bundle_rows':69,'trace_rows':len(traces),'bundle_sha256':sha(bundle_path),'trace_sha256':sha(OUT/'followup1_bundle_trace.csv'),'contract_sha256':sha(OUT/'followup1_contract.json'),'source_hashes_before':source_before,'bge_before':bge_before_f1,'build_resources':{'wall_seconds':build_wall,'cpu_seconds':build_cpu,'peak_rss_mib':rss_end/1024,'incremental_ru_maxrss_mib':max(0,rss_end-rss_started)/1024}}
write_json(OUT/'followup1_bundle_build_freeze.json',build_freeze)
assert {str(p.relative_to(ROOT)):sha(p) for p in source_paths}==source_before and tree_state(BGE_ROOT)==bge_before_f1
print({'bundle_build':'PASS','query_card_pairs':69,'trace_rows':len(traces),'gold_reads':0,'wall_seconds':build_wall})

{'bundle_build': 'PASS', 'query_card_pairs': 69, 'trace_rows': 220, 'gold_reads': 0, 'wall_seconds': 0.007784418994560838}


In [2]:
assert os.environ.get('RUN_APPROVED_24_GPU')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0' and os.environ.get('RUN_APPROVED_24_EXTERNAL')=='0'
import gc, math, statistics, subprocess
def gpu_snapshot():
    lines=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip().splitlines(); line=next(x for x in lines if x.split(',')[0].strip()=='0'); uuid=line.split(',')[1].strip()
    raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip(); processes=[]
    for x in raw.splitlines():
        parts=[v.strip() for v in x.split(',',3)]
        if len(parts)==4 and parts[0]==uuid: processes.append({'gpu_uuid':parts[0],'pid':int(parts[1]),'process_name':parts[2],'used_memory_mib':float(parts[3])})
    return {'gpu_line':line,'processes':processes,'captured_at_unix':time.time()}
gpu_before=gpu_snapshot()
import numpy as np, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); load_start=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False)
model=AutoModelForSequenceClassification.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0'); load_seconds=time.perf_counter()-load_start
audit=[]
for row in bundles:
    query=query_map[row['query_id']]; document_tokens=len(tokenizer.encode(row['bundle_text'],add_special_tokens=False)); query_tokens=len(tokenizer.encode(query,add_special_tokens=False)); total_tokens=len(tokenizer(query,row['bundle_text'],truncation=False)['input_ids'])
    assert document_tokens<=4096 and total_tokens<=8192
    audit.append({'query_id':row['query_id'],'card_key':row['card_key'],'bundle_sha256':row['bundle_sha256'],'query_sha256':hashlib.sha256(query.encode()).hexdigest(),'selected_chunk_count':len(row['selected_chunk_ids']),'bundle_characters':row['bundle_characters'],'query_tokens':query_tokens,'bundle_tokens':document_tokens,'input_tokens':total_tokens,'truncated_tokens':0,'source_allowlist':'query_text, metadata issuer/card/node/parent/part/page, hierarchy heading/search ids, heading_path/evidence_text','gold_fields_used':False,'retrieval_text_used':False})
assert len(audit)==69 and all(r['truncated_tokens']==0 and not r['gold_fields_used'] and not r['retrieval_text_used'] for r in audit)
warm=tokenizer(['준비'],['준비'],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0'); warm_start=time.perf_counter()
with torch.inference_mode(): assert torch.isfinite(model(**warm).logits).all()
torch.cuda.synchronize(); warm_seconds=time.perf_counter()-warm_start; del warm
scores=[]; score_start=time.perf_counter()
try:
    for start in range(0,len(bundles),2):
        batch=bundles[start:start+2]; encoded=tokenizer([query_map[r['query_id']] for r in batch],[r['bundle_text'] for r in batch],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
        with torch.inference_mode(): logits=model(**encoded).logits.reshape(-1).float().cpu().numpy()
        assert len(logits)==len(batch) and np.isfinite(logits).all(); scores.extend(map(float,logits)); del encoded,logits
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError('Follow-up 1 OOM at fixed batch2; partial results discarded and no policy change is authorized') from exc
torch.cuda.synchronize(); scoring_seconds=time.perf_counter()-score_start; assert len(scores)==69 and np.isfinite(np.asarray(scores)).all()
pair_rows=[]
for row,score,a in zip(bundles,scores,audit): pair_rows.append({'query_id':row['query_id'],'card_key':row['card_key'],'best_seed_chunk_id':row['best_seed_chunk_id'],'best_seed_rrf_rank':row['best_seed_rrf_rank'],'bundle_sha256':row['bundle_sha256'],'raw_logit':score,'selected_chunk_count':a['selected_chunk_count'],'bundle_characters':a['bundle_characters'],'bundle_tokens':a['bundle_tokens'],'input_tokens':a['input_tokens'],'truncated_tokens':0})
with (OUT/'followup1_pair_scores.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(pair_rows[0])); w.writeheader(); w.writerows(pair_rows)
with (OUT/'followup1_scorer_input_audit.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0])); w.writeheader(); w.writerows(audit)
rank_rows=[]
for q in sorted(query_map):
    rows=[r for r in pair_rows if r['query_id']==q]; rows.sort(key=lambda r:(-r['raw_logit'],int(r['best_seed_rrf_rank']),r['card_key']))
    assert len(rows)==len({r['card_key'] for r in rows})
    rank_rows.append({'query_id':q,'card_keys':[r['card_key'] for r in rows],'bundle_sha256s':[r['bundle_sha256'] for r in rows],'raw_logits':[r['raw_logit'] for r in rows],'best_seed_rrf_ranks':[int(r['best_seed_rrf_rank']) for r in rows]})
rank_path=OUT/'followup1_card_rankings.jsonl'; rank_path.write_text(''.join(canonical(r)+'\n' for r in rank_rows),encoding='utf-8')
gpu_after=gpu_snapshot(); token_values=[r['bundle_tokens'] for r in pair_rows]; char_values=[r['bundle_characters'] for r in pair_rows]; chunk_values=[r['selected_chunk_count'] for r in pair_rows]
resources={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'physical_gpu':0,'shared_gpu_measurement':True,'batch_size':2,'dtype':'float16','max_length':8192,'truncation':'only_second','bundle_token_cap':4096,'quality_pairs':69,'load_seconds':load_seconds,'warmup_seconds':warm_seconds,'scoring_seconds':scoring_seconds,'pairs_per_second':69/scoring_seconds,'oom':False,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30,'bundle_tokens':{'p50':statistics.median(token_values),'p95':float(np.percentile(token_values,95)),'max':max(token_values)},'bundle_characters':{'p50':statistics.median(char_values),'p95':float(np.percentile(char_values,95)),'max':max(char_values)},'bundle_chunk_count':{'p50':statistics.median(chunk_values),'p95':float(np.percentile(chunk_values,95)),'max':max(chunk_values)},'truncated_pairs':0,'gpu_before':gpu_before,'gpu_after':gpu_after,'build':build_freeze['build_resources'],'api_network_new_embedding_chroma_package_install':0}
write_json(OUT/'followup1_resources.json',resources)
del model,tokenizer,scores; gc.collect(); torch.cuda.empty_cache()
existing_output_hashes_before={name:sha(OUT/name) for name in base_output_names}
scoring_freeze={'created_before_gold_read':True,'prohibited_gold_metric_files_read_before_scoring_freeze':0,'pair_rows':69,'pair_scores_sha256':sha(OUT/'followup1_pair_scores.csv'),'rankings_sha256':sha(rank_path),'bundle_sha256':sha(bundle_path),'input_audit_sha256':sha(OUT/'followup1_scorer_input_audit.csv'),'existing_output_hashes_before':existing_output_hashes_before,'source_hashes_before':source_before,'bge_before':bge_before_f1}
write_json(OUT/'followup1_scoring_freeze.json',scoring_freeze)
assert {str(p.relative_to(ROOT)):sha(p) for p in source_paths}==source_before and tree_state(BGE_ROOT)==bge_before_f1
print({'scoring_freeze':'PASS','pairs':69,'truncated':0,'seconds':scoring_seconds,'peak_gib':resources['peak_allocated_gib'],'gold_reads':0})

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'scoring_freeze': 'PASS', 'pairs': 69, 'truncated': 0, 'seconds': 1.7776803320739418, 'peak_gib': 1.189836025238037, 'gold_reads': 0}


In [3]:
# Gold and prior metric files are first read after bundle, scores, and card rankings are frozen.
import unicodedata
with (OUT/'gold_card_labels.csv').open(encoding='utf-8',newline='') as f: labels=list(csv.DictReader(f))
with (OUT/'atomic_claims.csv').open(encoding='utf-8',newline='') as f: claims=list(csv.DictReader(f))
with (OUT/'per_query_metrics.csv').open(encoding='utf-8',newline='') as f: prior=list(csv.DictReader(f))
assert len(labels)==100 and len(claims)==33 and len(prior)==80
positives=defaultdict(set)
for r in labels:
    if r['label']=='positive': positives[r['query_id']].add(r['card_key'])
terms={(r['query_id'],r['card_key']):json.loads(r['predicate_terms_json']) for r in claims}
def norm(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
bundle_by={(r['query_id'],r['card_key']):r for r in bundles}; bundle_ranks={r['query_id']:r for r in rank_rows}
def bundle_metrics(q,cards):
    pos=positives[q]; top3=cards[:3]; top5=cards[:5]; c3=sum(c in pos for c in top3); c5=sum(c in pos for c in top5); supported=[]
    for c in top5:
        row=bundle_by[(q,c)]; evidence=' '.join(([row['optional_parent_heading']] if row['optional_parent_heading'] else [])+[chunk_by_id[x]['heading_path'].__str__()+' '+chunk_by_id[x]['evidence_text'] for x in row['selected_chunk_ids']])
        supported.append(c in pos and all(norm(t) in norm(evidence) for t in terms[(q,c)]))
    return {'card_precision_at_3':c3/3,'card_recall_at_3':c3/len(pos),'card_precision_at_5':c5/5,'card_recall_at_5':c5/len(pos),'bundle_supported_card_recall_at_5':sum(supported)/len(pos),'bundle_evidence_accuracy_at_5':sum(supported)/5,'supported_flags':supported}
prior_by={(r['configuration'],r['query_id']):r for r in prior}; per_query=[]
for q in sorted(query_map):
    for output_system,source in [('old_selective_bge_d20','old_selective_bge_d20'),('structural_all_bge_d20_single_chunk','structural_all_bge_d20')]:
        r=prior_by[(source,q)]; per_query.append({'system':output_system,'query_id':q,'support_definition':'single-representative-supported','card_precision_at_3':float(r['card_precision_at_3']),'card_recall_at_3':float(r['card_recall_at_3']),'card_precision_at_5':float(r['card_precision_at_5']),'card_recall_at_5':float(r['card_recall_at_5']),'single_representative_supported_card_recall_at_5':float(r['evidence_supported_card_recall_at_5']),'single_representative_evidence_accuracy_at_5':float(r['evidence_accuracy_at_5']),'bundle_supported_card_recall_at_5':'','bundle_evidence_accuracy_at_5':'','card_keys_json':r['output_card_ids_json'],'supported_flags_json':r['supported_json']})
    cards=bundle_ranks[q]['card_keys']; m=bundle_metrics(q,cards); per_query.append({'system':'structural_card_bundle_bge_d20','query_id':q,'support_definition':'bundle-supported end-to-end','card_precision_at_3':m['card_precision_at_3'],'card_recall_at_3':m['card_recall_at_3'],'card_precision_at_5':m['card_precision_at_5'],'card_recall_at_5':m['card_recall_at_5'],'single_representative_supported_card_recall_at_5':'','single_representative_evidence_accuracy_at_5':'','bundle_supported_card_recall_at_5':m['bundle_supported_card_recall_at_5'],'bundle_evidence_accuracy_at_5':m['bundle_evidence_accuracy_at_5'],'card_keys_json':json.dumps(cards[:5]),'supported_flags_json':json.dumps(m['supported_flags'])})
assert len(per_query)==30 and all(len(json.loads(r['card_keys_json']))==len(set(json.loads(r['card_keys_json']))) for r in per_query)
common=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5']; summary=[]
for system in ['old_selective_bge_d20','structural_all_bge_d20_single_chunk','structural_card_bundle_bge_d20']:
    rows=[r for r in per_query if r['system']==system]; out={'system':system,'denominator':10,**{m:sum(float(r[m]) for r in rows)/10 for m in common}}
    if system=='structural_card_bundle_bge_d20': out.update({'single_representative_supported_card_recall_at_5':'','single_representative_evidence_accuracy_at_5':'','bundle_supported_card_recall_at_5':sum(float(r['bundle_supported_card_recall_at_5']) for r in rows)/10,'bundle_evidence_accuracy_at_5':sum(float(r['bundle_evidence_accuracy_at_5']) for r in rows)/10})
    else: out.update({'single_representative_supported_card_recall_at_5':sum(float(r['single_representative_supported_card_recall_at_5']) for r in rows)/10,'single_representative_evidence_accuracy_at_5':sum(float(r['single_representative_evidence_accuracy_at_5']) for r in rows)/10,'bundle_supported_card_recall_at_5':'','bundle_evidence_accuracy_at_5':''})
    summary.append(out)
summary_by={r['system']:r for r in summary}; paired=[]; wlt=[]
for reference in ['old_selective_bge_d20','structural_all_bge_d20_single_chunk']:
    ref={r['query_id']:r for r in per_query if r['system']==reference}; cand={r['query_id']:r for r in per_query if r['system']=='structural_card_bundle_bge_d20'}
    for q in sorted(query_map):
        row={'reference':reference,'query_id':q,**{'delta_'+m:float(cand[q][m])-float(ref[q][m]) for m in common},'delta_bundle_vs_single_supported_card_recall_at_5':float(cand[q]['bundle_supported_card_recall_at_5'])-float(ref[q]['single_representative_supported_card_recall_at_5']),'delta_bundle_vs_single_evidence_accuracy_at_5':float(cand[q]['bundle_evidence_accuracy_at_5'])-float(ref[q]['single_representative_evidence_accuracy_at_5'])}; paired.append(row)
    for metric in common+['bundle_vs_single_supported_card_recall_at_5','bundle_vs_single_evidence_accuracy_at_5']:
        values=[r['delta_'+metric] for r in paired if r['reference']==reference]; wins=sum(v>1e-12 for v in values); losses=sum(v< -1e-12 for v in values); wlt.append({'reference':reference,'metric':metric,'denominator':10,'wins':wins,'losses':losses,'ties':10-wins-losses,'mean_delta':sum(values)/10})
old=summary_by['old_selective_bge_d20']; single=summary_by['structural_all_bge_d20_single_chunk']; new=summary_by['structural_card_bundle_bge_d20']; old_wlt=next(r for r in wlt if r['reference']=='old_selective_bge_d20' and r['metric']=='bundle_vs_single_supported_card_recall_at_5')
old_rows={r['query_id']:r for r in per_query if r['system']=='old_selective_bge_d20'}; new_rows={r['query_id']:r for r in per_query if r['system']=='structural_card_bundle_bge_d20'}
catastrophic=sum(float(old_rows[q]['single_representative_supported_card_recall_at_5'])>0 and (float(new_rows[q]['bundle_supported_card_recall_at_5'])==0 or float(new_rows[q]['bundle_supported_card_recall_at_5'])-float(old_rows[q]['single_representative_supported_card_recall_at_5'])<=-.5) for q in query_map)
checks={'card_recall_at_3':new['card_recall_at_3']>=.62-1e-12,'card_precision_at_5':new['card_precision_at_5']>=.48-1e-12,'card_recall_at_5':new['card_recall_at_5']>=.7933-1e-12,'bundle_supported_card_recall_at_5':new['bundle_supported_card_recall_at_5']>=.6433-1e-12,'bundle_evidence_accuracy_at_5':new['bundle_evidence_accuracy_at_5']>=.40-1e-12,'catastrophic_loss_zero':catastrophic==0,'supported_wins_gt_losses':old_wlt['wins']>old_wlt['losses']}
single_diag={'card_recall_at_3_nonreg':new['card_recall_at_3']>=single['card_recall_at_3']-1e-12,'card_precision_at_5_nonreg':new['card_precision_at_5']>=single['card_precision_at_5']-1e-12,'card_recall_at_5_nonreg':new['card_recall_at_5']>=single['card_recall_at_5']-1e-12,'bundle_supported_strict_improvement_over_single':new['bundle_supported_card_recall_at_5']>single['single_representative_supported_card_recall_at_5']+1e-12}
decision={'gate_checks_vs_old':checks,'gate_pass':all(checks.values()),'catastrophic_loss_count':catastrophic,'diagnostic_vs_structural_single':single_diag,'disposition':'adaptive_development_signal_only_bundle_candidate' if all(checks.values()) else 'retain_old_selective_bge_development_baseline','promotion_eligible':False,'causal_claim':'end-to-end bundle pipeline comparison only; no structural-only causal claim'}
required_traces=[('Q01','hana/Hana_One_More_SOHO'),('Q03','NH/NH_Namu_NH'),('Q03','lotte/Lotte_LOCA_LIKIT_Eat')]
assert all(any(r['query_id']==q and r['card_key']==card for r in bundles) for q,card in required_traces)
def write_csv(path,rows):
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
write_csv(OUT/'followup1_per_query.csv',per_query); write_csv(OUT/'followup1_summary.csv',summary); write_csv(OUT/'followup1_paired_deltas.csv',paired); write_csv(OUT/'followup1_wlt.csv',wlt)
write_json(OUT/'followup1_summary.json',{'summary':summary,'decision':decision,'metric_note':'bundle-supported is an end-to-end bundle metric and is not the prior single representative metric','required_relation_trace_keys':[list(x) for x in required_traces]}); write_json(OUT/'followup1_decision.json',decision)
source_after={str(p.relative_to(ROOT)):sha(p) for p in source_paths}; bge_after_f1=tree_state(BGE_ROOT); existing_output_hashes_after={name:sha(OUT/name) for name in base_output_names}
assert source_after==source_before and bge_after_f1==bge_before_f1 and existing_output_hashes_after==existing_output_hashes_before
followup_names=['followup1_contract.json','followup1_bundles.jsonl','followup1_bundle_trace.csv','followup1_bundle_build_freeze.json','followup1_pair_scores.csv','followup1_scorer_input_audit.csv','followup1_card_rankings.jsonl','followup1_resources.json','followup1_scoring_freeze.json','followup1_per_query.csv','followup1_summary.csv','followup1_summary.json','followup1_paired_deltas.csv','followup1_wlt.csv','followup1_decision.json']
integrity={'status':'PASS','query_count':10,'query_card_pairs':69,'bundle_rows':69,'trace_rows':len(traces),'pair_score_rows':69,'pair_scores_finite':True,'card_ranking_rows':10,'per_query_rows':30,'summary_rows':3,'paired_rows':20,'wlt_rows':12,'candidate_set_exact':True,'new_cards_added':0,'max_bundle_chunks':max(len(r['selected_chunk_ids']) for r in bundles),'bundle_token_cap':4096,'truncated_pairs':0,'gold_reads_before_scoring_freeze':0,'required_relation_traces_present':True,'source_hashes_before':source_before,'source_hashes_after':source_after,'bge_before':bge_before_f1,'bge_after':bge_after_f1,'existing_output_hashes_before':existing_output_hashes_before,'existing_output_hashes_after':existing_output_hashes_after,'existing_outputs_preserved':True,'api_network_new_embedding_chroma_package_install':0,'gpu_physical':0,'output_hashes':{name:sha(OUT/name) for name in followup_names}}
write_json(OUT/'followup1_integrity.json',integrity)
manifest={'phase':'followup1_structural_d20_operational_card_evidence_bundle','base_notebook_sha256_before_append':'1d8d7dddcb2905b86392edfe3275eb20999c866d7e63a6073fcf08013591df1e','notebook_sha256':'FINALIZE_AFTER_SAVE','inputs_before':source_before,'inputs_after':source_after,'bge_before':bge_before_f1,'bge_after':bge_after_f1,'existing_outputs_preserved':existing_output_hashes_after==existing_output_hashes_before,'outputs':{name:sha(OUT/name) for name in followup_names},'self_hash_policy':'followup1_run_manifest.json and followup1_integrity.json excluded'}
write_json(OUT/'followup1_run_manifest.json',manifest); integrity['run_manifest_sha256']=sha(OUT/'followup1_run_manifest.json'); write_json(OUT/'followup1_integrity.json',integrity)
print(json.dumps({'followup1':'PASS','summary':summary,'decision':decision,'resources':resources},ensure_ascii=False,indent=2))

{
  "followup1": "PASS",
  "summary": [
    {
      "system": "old_selective_bge_d20",
      "denominator": 10,
      "card_precision_at_3": 0.6333333333333333,
      "card_recall_at_3": 0.62,
      "card_precision_at_5": 0.4800000000000001,
      "card_recall_at_5": 0.7933333333333333,
      "single_representative_supported_card_recall_at_5": 0.6433333333333333,
      "single_representative_evidence_accuracy_at_5": 0.4000000000000001,
      "bundle_supported_card_recall_at_5": "",
      "bundle_evidence_accuracy_at_5": ""
    },
    {
      "system": "structural_all_bge_d20_single_chunk",
      "denominator": 10,
      "card_precision_at_3": 0.5666666666666667,
      "card_recall_at_3": 0.55,
      "card_precision_at_5": 0.5,
      "card_recall_at_5": 0.7966666666666667,
      "single_representative_supported_card_recall_at_5": 0.66,
      "single_representative_evidence_accuracy_at_5": 0.42000000000000004,
      "bundle_supported_card_recall_at_5": "",
      "bundle_evidence_accuracy

## Follow-up 2 — paraphrase + independent AND-combination generalization preflight

Codex coder agent가 추가한 append-only 개발 preflight입니다. Paraphrase 10개와 독립 AND 조합형 10개를 결과 확인 전에 고정합니다. P04의 canonical intent는 `direct discount OR cashback OR point accrual`입니다. 비교 계약은 old D20 selective-BGE 대 structural D20 frozen 1-hop card bundle + all-BGE이며, 두 cohort를 별도 집계합니다. 이번 셀은 질의·gold·원문 span·외부 전송 manifest만 동결합니다. API, network, query embedding, GPU, model scoring, Chroma는 모두 0이며 사용자 승인 전 외부 실행은 fail-closed입니다.


In [1]:
from pathlib import Path
import csv, hashlib, json, os, unicodedata
import tiktoken

cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError(f'fail-closed unexpected cwd: {cwd}')
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL','0')=='0'
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
canon=lambda x: json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()
norm=lambda x: ' '.join(unicodedata.normalize('NFKC',str(x)).lower().split())
def read_csv(path):
    with Path(path).open(encoding='utf-8',newline='') as f: return list(csv.DictReader(f))
def write_csv(path,rows):
    with Path(path).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
def write_json(path,obj): Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def tree_state(path):
    files={str(p.relative_to(path)):sha(p) for p in sorted(Path(path).rglob('*')) if p.is_file()}
    return {'file_count':len(files),'total_bytes':sum((Path(path)/n).stat().st_size for n in files),'digest':hashlib.sha256(canon(files)).hexdigest(),'files':files}

PARAPHRASES=[
('P01','Q01','카드를 처음 만든 직후에만 주는 예외는 빼고, 안내서에 포함된 상품 종류 가운데 국내 일반 가맹점 결제에 지난달 이용액 기준 없이 적립이나 할인을 주는 카드를 빠짐없이 찾아줘'),
('P02','Q02','쿠팡 또는 G마켓에서 온라인으로 상품을 살 때 별도 할인·포인트·캐시백을 주는 카드를 전부 찾아줘'),
('P03','Q03','배민·요기요·쿠팡이츠 앱에서 주문을 결제하면 할인이나 캐시백을 받는 카드를 빠짐없이 알려줘'),
('P04','Q04','휴대전화 요금 결제에 별도 할인·적립·캐시백을 주는 카드를 전부 알려줘'),
('P05','Q05','전기료 또는 도시가스비를 카드로 낼 때 결제금액을 직접 깎아 주는 카드를 빠짐없이 찾아줘'),
('P06','Q06','커피 전문 매장에서 카드로 결제했을 때 할인이나 캐시백이 바로 적용되는 카드를 전부 알려줘'),
('P07','Q07','영상·음악 서비스 정기구독료나 유료 멤버십 요금을 결제하면 별도 혜택을 주는 카드를 전부 찾아줘'),
('P08','Q08','일반 식당에서 카드로 결제할 때 직접 할인되거나 포인트가 적립되는 카드를 모두 알려줘'),
('P09','Q09','대형 할인점 또는 슈퍼마켓에서 결제할 때 별도 할인이나 캐시백을 주는 카드를 전부 찾아줘'),
('P10','Q10','버스·지하철·택시 요금을 카드로 낼 때 직접 할인 또는 캐시백을 받는 카드를 모두 알려줘')]
COMBINATIONS=[
('CMB01','Q01','Q04','전월 실적 없이 국내 일반 가맹점 혜택을 받고 이동통신 요금에도 별도 혜택이 있는 카드를 모두 알려줘',['C03','C08']),
('CMB02','Q02','Q04','쿠팡·G마켓 온라인 쇼핑과 이동통신 요금 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C03','C04','C09']),
('CMB03','Q02','Q05','쿠팡·G마켓 온라인 쇼핑과 전기·도시가스 요금 납부에 모두 직접 혜택이 있는 카드를 알려줘',['C03','C09']),
('CMB04','Q02','Q06','쿠팡·G마켓 온라인 쇼핑과 커피전문점 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C09']),
('CMB05','Q02','Q07','쿠팡·G마켓 온라인 쇼핑과 영상·음악 구독 또는 유료 멤버십에 모두 혜택이 있는 카드를 알려줘',['C02','C04']),
('CMB06','Q02','Q08','쿠팡·G마켓 온라인 쇼핑과 일반 음식점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C04','C09']),
('CMB07','Q03','Q06','배달앱과 커피전문점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C02','C07']),
('CMB08','Q04','Q09','이동통신 요금과 대형마트·슈퍼마켓 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C08','C09']),
('CMB09','Q06','Q08','커피전문점과 일반 음식점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C07','C09']),
('CMB10','Q07','Q08','영상·음악 구독 또는 유료 멤버십과 일반 음식점 결제에 모두 혜택이 있는 카드를 알려줘',['C04','C07'])]
query_rows=[]
for q,parent,text in PARAPHRASES: query_rows.append({'request_order':len(query_rows)+1,'query_id':q,'cohort':'paraphrase','parent_query_ids_json':json.dumps([parent],ensure_ascii=False),'boolean_operator':'REFERENCE','query_text':text,'query_sha256':hashlib.sha256(text.encode()).hexdigest(),'canonical_intent':'direct discount OR cashback OR point accrual' if q=='P04' else parent})
for q,a,b,text,_ in COMBINATIONS: query_rows.append({'request_order':len(query_rows)+1,'query_id':q,'cohort':'and_combination','parent_query_ids_json':json.dumps([a,b],ensure_ascii=False),'boolean_operator':'AND','query_text':text,'query_sha256':hashlib.sha256(text.encode()).hexdigest(),'canonical_intent':f'{a} AND {b}'})
assert len(query_rows)==20 and [r['request_order'] for r in query_rows]==list(range(1,21)) and len({r['query_id'] for r in query_rows})==20
assert len({norm(r['query_text']) for r in query_rows})==20
write_csv(OUT/'followup2_queries.csv',query_rows)

base_inputs=[OUT/'queries.csv',OUT/'gold_card_labels.csv',OUT/'atomic_claims.csv',OUT/'source_manifest.csv',ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl',ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl',ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl',ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json',OUT/'followup1_contract.json']
input_before={str(p.relative_to(ROOT)):sha(p) for p in base_inputs}
assert input_before[str((OUT/'followup1_contract.json').relative_to(ROOT))]=='022ab18a8a6e2c5c398299d608a8baca7ccb391f323f67b7c341e7710984b58c'
parent_queries=read_csv(OUT/'queries.csv'); parent_gold=read_csv(OUT/'gold_card_labels.csv'); claims=read_csv(OUT/'atomic_claims.csv'); sources=read_csv(OUT/'source_manifest.csv')
assert len(parent_queries)==10 and len(parent_gold)==100 and len(claims)==33 and len(sources)==10
assert not ({norm(r['query_text']) for r in query_rows}&{norm(r['query_text']) for r in parent_queries})
cards=sorted({(r['card_id'],r['card_key']) for r in parent_gold}); assert len(cards)==10
gold_by={(r['query_id'],r['card_id']):r for r in parent_gold}; claim_by={(r['query_id'],r['card_id']):r for r in claims}
source_by={r['card_id']:r for r in sources}
for r in sources: assert sha(ROOT/r['source_path'])==r['sha256']
positive_counts=[sum(gold_by[(f'Q{i:02d}',card)]['label']=='positive' for card,_ in cards) for i in range(1,11)]
assert positive_counts==[5,5,2,5,2,3,3,3,3,2]
labels=[]; combination_evidence=[]; positive_sets=[]; predicate_pairs=[]
for q,parent,_ in PARAPHRASES:
    for card,card_key in cards:
        src=gold_by[(parent,card)]; claim_ids=[src['claim_id']] if src['label']=='positive' else []
        labels.append({'query_id':q,'cohort':'paraphrase','parent_query_ids_json':json.dumps([parent]),'card_id':card,'card_key':card_key,'label':src['label'],'claim_ids_json':json.dumps(claim_ids),'failed_predicate_ids_json':json.dumps([] if claim_ids else [parent]),'gold_source':'frozen_parent_gold_reference'})
for q,a,b,_,expected in COMBINATIONS:
    pair=tuple(sorted([a,b])); predicate_pairs.append(pair)
    actual=[]
    for card,card_key in cards:
        passed=[p for p in [a,b] if gold_by[(p,card)]['label']=='positive']; is_pos=len(passed)==2
        claim_ids=[claim_by[(p,card)]['claim_id'] for p in [a,b]] if is_pos else []
        labels.append({'query_id':q,'cohort':'and_combination','parent_query_ids_json':json.dumps([a,b]),'card_id':card,'card_key':card_key,'label':'positive' if is_pos else 'negative','claim_ids_json':json.dumps(claim_ids),'failed_predicate_ids_json':json.dumps([p for p in [a,b] if p not in passed]),'gold_source':'frozen_atomic_intersection_plus_raw_span_audit'})
        if is_pos:
            actual.append(card)
            for p in [a,b]:
                c=claim_by[(p,card)]; raw=(ROOT/c['source_path']).read_text(encoding='utf-8')
                spans=json.loads(c['predicate_spans_json']); assert spans
                assert all(raw[int(s['start']):int(s['end'])]==s['term'] for s in spans)
                assert raw[int(c['span_start']):int(c['span_end'])]==c['anchor']
                combination_evidence.append({'query_id':q,'card_id':card,'card_key':card_key,'predicate_id':p,'claim_id':c['claim_id'],'source_path':c['source_path'],'source_page':c['source_page'],'anchor':c['anchor'],'predicate_terms_json':c['predicate_terms_json'],'predicate_spans_json':c['predicate_spans_json'],'raw_span_audit':'PASS','source_sha256':source_by[card]['sha256']})
    assert actual==expected; positive_sets.append(tuple(actual))
assert len(labels)==200 and len(predicate_pairs)==len(set(predicate_pairs))==10 and len(positive_sets)==len(set(positive_sets))==10
assert sum(r['label']=='positive' for r in labels)==56 and sum(r['label']=='negative' for r in labels)==144 and not any(r['label']=='insufficient' for r in labels)
assert len(combination_evidence)==46
assert all(sum(r['query_id']==q and r['card_id']==c for r in combination_evidence)==2 for q,_,_,_,expected in COMBINATIONS for c in expected)
write_csv(OUT/'followup2_gold_labels.csv',labels); write_csv(OUT/'followup2_combination_evidence.csv',combination_evidence)
query_freeze={'declared_before_retrieval_result_reads':True,'restricted_result_reads_before_freeze':0,'ordered_query_sha256':[(r['query_id'],r['query_sha256']) for r in query_rows],'queries_csv_sha256':sha(OUT/'followup2_queries.csv'),'exact_normalized_duplicate_count':0,'parent_exact_normalized_overlap_count':0,'p04_canonical_intent':'direct discount OR cashback OR point accrual'}
gold_freeze={'declared_before_retrieval_result_reads':True,'restricted_result_reads_before_freeze':0,'label_rows':200,'positive':56,'negative':144,'insufficient':0,'paraphrase_parent_positive_counts':positive_counts,'combination_positive_counts':[len(x) for x in positive_sets],'combination_evidence_rows':46,'gold_labels_sha256':sha(OUT/'followup2_gold_labels.csv'),'combination_evidence_sha256':sha(OUT/'followup2_combination_evidence.csv'),'atomic_claims_sha256':sha(OUT/'atomic_claims.csv'),'source_manifest_sha256':sha(OUT/'source_manifest.csv')}
write_json(OUT/'followup2_query_freeze.json',query_freeze); write_json(OUT/'followup2_gold_freeze.json',gold_freeze)
gold_freeze_digest=hashlib.sha256(canon({'query':query_freeze,'gold':gold_freeze})).hexdigest()

# Only after query/gold freeze may prior ranking, metric, and pair-score artifacts be read for preservation hashes.
restricted_names={'ranking_freeze.jsonl','per_query_metrics.csv','pair_scores.csv','followup1_pair_scores.csv'}
assert all((OUT/n).is_file() for n in restricted_names)
existing_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup2_'))
existing_before={n:sha(OUT/n) for n in existing_names}

f1=json.loads((OUT/'followup1_contract.json').read_text()); c21=json.loads((ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json').read_text())
assert f1['relation_order']==['best_rrf_seed','same_node_adjacent_parts','non_root_immediate_parent_direct_body','heading_only_parent_context_and_same_parent_direct_body_children_by_heading_distance_then_earlier','seed_node_direct_children','same_card_other_top20_seeds_rrf_order']
assert f1['max_unique_chunks_per_bundle']==5 and f1['bundle_document_token_cap']==4096 and f1['bge']['revision']=='953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
assert c21['phase_a']['classifier_rule_sha256']=='e4e2c0b33dc228382e269a3edb399b1b5b4d0c4f8d41149e96d4d06d64d55909'
contract={'name':'Follow-up 2 — paraphrase + independent AND-combination generalization preflight','provenance':'Codex coder agent','declared_before_results':True,'scope':'same frozen 10-card development corpus only','p04_canonical_intent':'direct discount OR cashback OR point accrual','systems':{'baseline':'old chunks + RRF vector0.4/BM250.6 k60 depth50 + D20 + frozen regex selective BGE','candidate':'structural direct-body + same RRF/D20 + frozen Follow-up1 same-card 1-hop bundle max5/doc4096 + all BGE'},'retrieval':{'bm25_k1':1.5,'bm25_b':0.75,'vector_weight':0.4,'bm25_weight':0.6,'rrf_k':60,'component_depth':50,'candidate_depth':20},'bundle_binding':{'followup1_contract_sha256':sha(OUT/'followup1_contract.json'),'relation_order':f1['relation_order'],'same_card_only':True,'max_unique_chunks':5,'bundle_document_token_cap':4096,'scorer_text':f1['scorer_text'],'rule_changes_for_new_queries':0},'classifier_binding':{'rule_sha256':c21['phase_a']['classifier_rule_sha256'],'runtime_input_fields':['query_text']},'bge':f1['bge'],'metrics_per_cohort':['Card Precision@3','Card Precision@5','Card Recall@3','Card Recall@5','Evidence-supported Card Recall@5','Evidence Accuracy@3','Evidence Accuracy@5','Candidate Card Recall@20','Bundle-reachable Evidence Recall@20','zero-hit','raw Top20 duplicate rate','raw Top20 unique cards','paired delta','W/L/T'],'gate':{'combination_supported_recall_at_5':'strict improvement and W>L','paraphrase_supported_recall_at_5':'non-regression and W>=L','both_cohorts_nonregression':['Card Recall@5','Precision@5','Evidence Accuracy@5','Recall@3'],'zero_hit_increase':0,'candidate_recall_at_20':'macro non-regression and no new zero-ceiling query','catastrophic_loss':'baseline Card Recall@5 > 0 and candidate == 0, or query delta <= -0.5'},'dispositions':['query_and_compositional_generalization_dev_signal','robustness_observed_but_no_candidate_advantage','generalization_not_supported'],'always':['dev_signal_only','not_eligible_for_promotion'],'cross_corpus_causal_claim_forbidden':True,'execution_preflight':{'api':0,'network':0,'gpu':0,'model_load_or_scoring':0,'chroma':0,'new_embedding':0,'package_install':0},'external_guard':'exact approval manifest digest required; otherwise fail closed'}
write_json(OUT/'followup2_contract.json',contract)
enc=tiktoken.get_encoding('cl100k_base'); per_tokens=[len(enc.encode(r['query_text'])) for r in query_rows]; token_total=sum(per_tokens)
request_items=[{'order':r['request_order'],'query_id':r['query_id'],'text':r['query_text'],'sha256':r['query_sha256'],'estimated_tokens':tok} for r,tok in zip(query_rows,per_tokens)]
embedding_plan={'model':'text-embedding-3-small','ordered_items':request_items,'item_count':20,'estimated_token_count':token_total,'character_count':sum(len(r['query_text']) for r in query_rows),'max_items':20,'max_tokens':token_total,'max_requests':1,'expected_shape':[20,1536],'expected_dtype':'float32','documents_or_gold_or_evaluation_fields_transmitted':False,'price_usd':None,'price_note':'official price not queried in offline preflight','network_api_new_embedding_current_run':0,'approval_status':'awaiting_user_approval'}
write_json(OUT/'followup2_embedding_plan.json',embedding_plan)
approval_core={'experiment':'24_followup2','provider':'OpenAI embeddings API','model':'text-embedding-3-small','ordered_items':request_items,'max_items':20,'max_tokens':token_total,'max_requests':1,'transmit_only':'ordered query text','forbidden_transmission':['documents','gold','atomic claims','source spans','labels','evaluation fields'],'gold_freeze_digest':gold_freeze_digest,'mismatch_policy':'fail before call','approval_status':'awaiting_user_approval'}
approval_digest=hashlib.sha256(canon(approval_core)).hexdigest(); approval={**approval_core,'approval_manifest_sha256':approval_digest}
write_json(OUT/'followup2_approval_manifest.json',approval)
BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'; bge_before=tree_state(BGE_ROOT)
source_manifest={'inputs':input_before,'old_chunks':{'rows':sum(1 for _ in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').open(encoding='utf-8')),'sha256':sha(ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl')},'structural_chunks':{'rows':sum(1 for _ in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').open(encoding='utf-8')),'sha256':sha(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl')},'hierarchy_sha256':sha(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'),'bge':bge_before,'followup1_bundle_contract_sha256':sha(OUT/'followup1_contract.json'),'classifier_contract_sha256':sha(ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json')}
write_json(OUT/'followup2_source_manifest.json',source_manifest)
input_after={str(p.relative_to(ROOT)):sha(p) for p in base_inputs}; bge_after=tree_state(BGE_ROOT); existing_after={n:sha(OUT/n) for n in existing_names}
assert input_after==input_before and bge_after==bge_before and existing_after==existing_before
output_names=['followup2_contract.json','followup2_queries.csv','followup2_query_freeze.json','followup2_gold_labels.csv','followup2_combination_evidence.csv','followup2_gold_freeze.json','followup2_embedding_plan.json','followup2_approval_manifest.json','followup2_source_manifest.json']
integrity={'status':'PASS','query_rows':20,'normalized_duplicate_count':0,'parent_normalized_overlap_count':0,'label_rows':200,'positive':56,'negative':144,'insufficient':0,'combination_evidence_rows':46,'combination_predicate_pair_duplicates':0,'combination_positive_set_duplicates':0,'positive_two_claim_source_span_audit':True,'restricted_result_reads_before_query_gold_freeze':0,'query_gold_frozen_before_restricted_reads':True,'api_network_gpu_model_chroma_new_embedding_package_install':0,'approval_status':'awaiting_user_approval','approval_manifest_sha256':approval_digest,'input_hashes_before':input_before,'input_hashes_after':input_after,'bge_before':bge_before,'bge_after':bge_after,'existing_outputs_before':existing_before,'existing_outputs_after':existing_after,'existing_outputs_preserved':True,'output_hashes':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','run_manifest_sha256':'FINALIZE_AFTER_SAVE'}
write_json(OUT/'followup2_integrity.json',integrity)
run_manifest={'phase':'followup2_offline_preflight','execution':'fresh kernel; only followup2 preflight cell executed','current_external_calls':0,'current_gpu_model_scoring':0,'approval_status':'awaiting_user_approval','approval_manifest_sha256':approval_digest,'inputs_before':input_before,'inputs_after':input_after,'existing_outputs_preserved':True,'outputs':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','self_hash_policy':'followup2_integrity.json and followup2_preflight_run_manifest.json excluded'}
write_json(OUT/'followup2_preflight_run_manifest.json',run_manifest)
print(json.dumps({'status':'PASS','queries':20,'labels':200,'positive':56,'negative':144,'combination_evidence_rows':46,'estimated_tokens':token_total,'max_requests':1,'approval_manifest_sha256':approval_digest,'external_calls':0,'approval_status':'awaiting_user_approval'},ensure_ascii=False,indent=2))

{
  "status": "PASS",
  "queries": 20,
  "labels": 200,
  "positive": 56,
  "negative": 144,
  "combination_evidence_rows": 46,
  "estimated_tokens": 1115,
  "max_requests": 1,
  "approval_manifest_sha256": "18dc2ef69c112fd46e20642c6d87c112ec11bd2d36b63594b0498298778b524b",
  "external_calls": 0,
  "approval_status": "awaiting_user_approval"
}


In [ ]:
# Intentionally not executed in preflight. Any future external stage must bind to the exact approved digest.
import json, os
from pathlib import Path
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL','0')=='1', 'fail-closed: explicit approval required'
approval=json.loads(Path('notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup2_approval_manifest.json').read_text())
assert approval['approval_status']=='approved', 'fail-closed: manifest is awaiting approval'
assert os.environ.get('APPROVED_24_FOLLOWUP2_MANIFEST_SHA256')==approval['approval_manifest_sha256'], 'fail-closed: approval digest mismatch'
raise RuntimeError('preflight-only notebook state: external embedding/scoring implementation is intentionally absent')

### Follow-up 2 preflight v3 — final pre-result freeze

이 v3가 앞선 승인 불가 초안과 v2를 supersede합니다. CMB07의 Q03 범위, cohort별 exact gate, contract/query/gold pre-result guard와 Q01_C06의 implicit 조건 해석을 최종 동결합니다. 엄격 개선도 개발 신호일 뿐이며 인과·운영 승격을 뜻하지 않습니다.


In [1]:
from pathlib import Path
import csv, hashlib, json, os, re, unicodedata
import tiktoken

cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError(f'fail-closed unexpected cwd: {cwd}')
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL','0')=='0'
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
canon=lambda x: json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()
norm=lambda x: ' '.join(unicodedata.normalize('NFKC',str(x)).lower().split())
def read_csv(path):
    with Path(path).open(encoding='utf-8',newline='') as f: return list(csv.DictReader(f))
def write_csv(path,rows):
    with Path(path).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
def write_json(path,obj): Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def write_jsonl(path,rows): Path(path).write_text(''.join(json.dumps(r,ensure_ascii=False,sort_keys=True,separators=(',',':'))+'\n' for r in rows),encoding='utf-8')
def tree_state(path):
    files={str(p.relative_to(path)):sha(p) for p in sorted(Path(path).rglob('*')) if p.is_file()}
    return {'file_count':len(files),'total_bytes':sum((Path(path)/n).stat().st_size for n in files),'digest':hashlib.sha256(canon(files)).hexdigest(),'files':files}

PARAPHRASES=[
('P01','Q01','카드를 처음 만든 직후에만 주는 예외는 빼고, 안내서에 포함된 상품 종류 가운데 국내 일반 가맹점 결제에 지난달 이용액 기준 없이 적립이나 할인을 주는 카드를 빠짐없이 찾아줘'),
('P02','Q02','쿠팡 또는 G마켓에서 온라인으로 상품을 살 때 별도 할인·포인트·캐시백을 주는 카드를 전부 찾아줘'),
('P03','Q03','배민·요기요·쿠팡이츠 중 하나에서 주문을 결제하면 할인이나 캐시백을 받는 카드를 빠짐없이 알려줘'),
('P04','Q04','휴대전화 요금 결제에 별도 할인·적립·캐시백을 주는 카드를 전부 알려줘'),
('P05','Q05','전기료 또는 도시가스비를 카드로 낼 때 결제금액을 직접 깎아 주는 카드를 빠짐없이 찾아줘'),
('P06','Q06','커피 전문 매장에서 카드로 결제했을 때 할인이나 캐시백이 바로 적용되는 카드를 전부 알려줘'),
('P07','Q07','영상·음악 서비스 정기구독료나 유료 멤버십 요금을 결제하면 별도 혜택을 주는 카드를 전부 찾아줘'),
('P08','Q08','일반 식당에서 카드로 결제할 때 직접 할인되거나 포인트가 적립되는 카드를 모두 알려줘'),
('P09','Q09','대형 할인점 또는 슈퍼마켓에서 결제할 때 별도 할인이나 캐시백을 주는 카드를 전부 찾아줘'),
('P10','Q10','버스·지하철·택시 중 하나의 요금을 카드로 낼 때 직접 할인 또는 캐시백을 받는 카드를 모두 알려줘')]
COMBINATIONS=[
('CMB01','Q01','Q04','신규 발급 직후에만 주는 예외는 제외하고, 전월 실적 없이 국내 일반 가맹점 혜택을 받으며 이동통신 요금에도 별도 혜택이 있는 카드를 모두 알려줘',['C03','C08']),
('CMB02','Q02','Q04','쿠팡 또는 G마켓 온라인 쇼핑과 이동통신 요금 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C03','C04','C09']),
('CMB03','Q02','Q05','쿠팡 또는 G마켓 온라인 쇼핑과 전기·도시가스 요금 납부에 모두 직접 혜택이 있는 카드를 알려줘',['C03','C09']),
('CMB04','Q02','Q06','쿠팡 또는 G마켓 온라인 쇼핑과 커피전문점 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C09']),
('CMB05','Q02','Q07','쿠팡 또는 G마켓 온라인 쇼핑과 영상·음악 구독 또는 유료 멤버십에 모두 혜택이 있는 카드를 알려줘',['C02','C04']),
('CMB06','Q02','Q08','쿠팡 또는 G마켓 온라인 쇼핑과 일반 음식점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C04','C09']),
('CMB07','Q03','Q06','배민·요기요·쿠팡이츠 중 하나와 커피전문점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C02','C07']),
('CMB08','Q04','Q09','이동통신 요금과 대형마트 또는 슈퍼마켓 결제에 모두 별도 혜택이 있는 카드를 알려줘',['C02','C08','C09']),
('CMB09','Q06','Q08','커피전문점과 일반 음식점 결제에 모두 직접 혜택이 있는 카드를 알려줘',['C07','C09']),
('CMB10','Q07','Q08','영상·음악 구독 또는 유료 멤버십과 일반 음식점 결제에 모두 혜택이 있는 카드를 알려줘',['C04','C07'])]
assert '중 하나' in PARAPHRASES[2][2] and '중 하나' in PARAPHRASES[9][2]
assert all('쿠팡 또는 G마켓' in r[3] for r in COMBINATIONS[1:6]) and '대형마트 또는 슈퍼마켓' in COMBINATIONS[7][3]
assert '신규 발급 직후에만 주는 예외는 제외하고' in COMBINATIONS[0][3]
query_rows=[]
for q,parent,text in PARAPHRASES: query_rows.append({'request_order':len(query_rows)+1,'query_id':q,'cohort':'paraphrase','parent_query_ids_json':json.dumps([parent]),'boolean_operator':'REFERENCE','query_text':text,'query_sha256':hashlib.sha256(text.encode()).hexdigest(),'canonical_intent':'direct discount OR cashback OR point accrual' if q=='P04' else parent})
for q,a,b,text,_ in COMBINATIONS: query_rows.append({'request_order':len(query_rows)+1,'query_id':q,'cohort':'and_combination','parent_query_ids_json':json.dumps([a,b]),'boolean_operator':'AND','query_text':text,'query_sha256':hashlib.sha256(text.encode()).hexdigest(),'canonical_intent':f'{a} AND {b}'})
assert len(query_rows)==20 and len({r['query_id'] for r in query_rows})==20 and len({norm(r['query_text']) for r in query_rows})==20
write_csv(OUT/'followup2_queries.csv',query_rows)
assert '배민·요기요·쿠팡이츠 중 하나' in COMBINATIONS[6][3]
meaning_review={'version':3,'status':'PASS','p04_canonical_intent':'direct discount OR cashback OR point accrual','cmb01_new_issuance_exception_excluded':True,'or_semantics':{'P03':'one of three delivery apps','P10':'one of bus/subway/taxi','CMB02_CMB06':'Coupang OR Gmarket','CMB07':'one of Baemin/Yogiyo/Coupang Eats','CMB08':'large mart OR supermarket'},'all_20_reviewed_before_retrieval_results':True}
write_json(OUT/'followup2_semantic_review.json',meaning_review)

ROLE_LINES={
'Q01_C01':{'target':[53],'benefit':[49],'condition':[51]},'Q01_C03':{'target':[89],'benefit':[91],'condition':[79]},'Q01_C06':{'target':[13],'benefit':[13],'condition':[]},'Q01_C08':{'target':[65],'benefit':[65],'condition':[65]},'Q01_C10':{'target':[136],'benefit':[132],'condition':[139]},
'Q02_C02':{'target':[97],'benefit':[94],'condition':[93]},'Q02_C03':{'target':[40],'benefit':[28],'condition':[30,32,33,34]},'Q02_C04':{'target':[58],'benefit':[55],'condition':[65,66,67,68]},'Q02_C05':{'target':[17],'benefit':[14],'condition':[19]},'Q02_C09':{'target':[173],'benefit':[169],'condition':[178,180]},
'Q03_C02':{'target':[97],'benefit':[94],'condition':[93]},'Q03_C07':{'target':[25],'benefit':[25],'condition':[27,41]},
'Q04_C02':{'target':[99],'benefit':[94],'condition':[93]},'Q04_C03':{'target':[67],'benefit':[52],'condition':[54,56,57,58,71,72]},'Q04_C04':{'target':[61],'benefit':[55],'condition':[65,74]},'Q04_C08':{'target':[44],'benefit':[39],'condition':[47,49,51,53]},'Q04_C09':{'target':[123],'benefit':[113],'condition':[115,117,125,126]},
'Q05_C03':{'target':[68],'benefit':[52],'condition':[54,56,57,58,74,75,76]},'Q05_C09':{'target':[121,122],'benefit':[113],'condition':[115,117]},
'Q06_C02':{'target':[101],'benefit':[94],'condition':[93]},'Q06_C07':{'target':[31],'benefit':[31],'condition':[33,41]},'Q06_C09':{'target':[175],'benefit':[169],'condition':[177,178]},
'Q07_C02':{'target':[99],'benefit':[94],'condition':[93]},'Q07_C04':{'target':[61],'benefit':[55],'condition':[65,71,72,73]},'Q07_C07':{'target':[37],'benefit':[37],'condition':[41]},
'Q08_C04':{'target':[59],'benefit':[55],'condition':[65]},'Q08_C07':{'target':[17,19],'benefit':[19],'condition':[41]},'Q08_C09':{'target':[175],'benefit':[169],'condition':[177,178]},
'Q09_C02':{'target':[98],'benefit':[94],'condition':[93]},'Q09_C08':{'target':[18,19],'benefit':[13],'condition':[21,23,25,27]},'Q09_C09':{'target':[140],'benefit':[132],'condition':[134,136,143]},
'Q10_C02':{'target':[100],'benefit':[94],'condition':[93]},'Q10_C09':{'target':[174],'benefit':[169],'condition':[178]}}

base_inputs=[OUT/'queries.csv',OUT/'gold_card_labels.csv',OUT/'atomic_claims.csv',OUT/'source_manifest.csv',ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl',ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl',ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl',ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json',OUT/'followup1_contract.json']
input_before={str(p.relative_to(ROOT)):sha(p) for p in base_inputs}
parent_queries=read_csv(OUT/'queries.csv'); parent_gold=read_csv(OUT/'gold_card_labels.csv'); claims=read_csv(OUT/'atomic_claims.csv'); sources=read_csv(OUT/'source_manifest.csv')
hier=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl').read_text(encoding='utf-8').splitlines() if x]
assert len(parent_queries)==10 and len(parent_gold)==100 and len(claims)==33 and len(ROLE_LINES)==33 and set(ROLE_LINES)=={r['claim_id'] for r in claims}
assert not ({norm(r['query_text']) for r in query_rows}&{norm(r['query_text']) for r in parent_queries})
source_by={r['card_id']:r for r in sources}; cards=sorted((r['card_id'],r['card_key']) for r in sources)
for r in sources: assert sha(ROOT/r['source_path'])==r['sha256']
node_by_heading={(r['card_key'],int(r['heading_line_number'])):r for r in hier if r.get('heading_line_number') is not None and int(r['heading_line_number'])>0}
def source_lines(path):
    raw=Path(path).read_text(encoding='utf-8'); kept=raw.splitlines(keepends=True); starts=[]; pos=0
    for line in kept: starts.append(pos); pos+=len(line)
    plain=[x.rstrip('\r\n') for x in kept]; stacks=[]; stack=[]
    for n,text in enumerate(plain,1):
        m=re.match(r'^(#{1,6})\s+(.*)$',text.strip())
        if m:
            level=len(m.group(1)); stack=[x for x in stack if x[0]<level]; stack.append((level,m.group(2).strip(),n))
        stacks.append(list(stack))
    return raw,plain,starts,stacks
source_cache={}
claim_audits=[]
for c in claims:
    path=ROOT/c['source_path']; source_cache.setdefault(c['source_path'],source_lines(path)); raw,plain,starts,stacks=source_cache[c['source_path']]
    spec=ROLE_LINES[c['claim_id']]; assert spec['target'] and spec['benefit']
    role_spans=[]
    for role in ['target','benefit','condition']:
        for line_no in spec[role]:
            text=plain[line_no-1]; start=starts[line_no-1]; end=start+len(text); assert text and raw[start:end]==text
            stack=stacks[line_no-1]; heading_path=[x[1] for x in stack]; heading_line=stack[-1][2] if stack else 0; node=node_by_heading.get((c['card_key'],heading_line))
            role_spans.append({'role':role,'line':line_no,'start':start,'end':end,'text':text,'heading_path':heading_path,'heading_line_number':heading_line,'node_id':node['node_id'] if node else None})
    paths=[r['heading_path'] for r in role_spans]; common=[]
    for vals in zip(*paths):
        if len(set(vals))==1: common.append(vals[0])
        else: break
    declared_common_standard=(c['card_id']=='C07' and any(r['role']=='condition' and r['line']==41 and r['heading_path']==['할인 공통기준'] for r in role_spans))
    assert common or declared_common_standard, (c['claim_id'],paths)
    relation='same-card explicitly declared common-standard sibling' if declared_common_standard and not common else 'same semantic heading subtree; ancestor direct-body roles allowed'
    condition_mode='implicit_no_stated_requirement_in_document' if c['claim_id']=='Q01_C06' else ('direct_condition_spans' if spec['condition'] else 'not_applicable')
    limitation='No direct no-spend-condition span exists; positivity follows the frozen interpretation that the documented mileage benefit states no requirement, not proof of absence.' if c['claim_id']=='Q01_C06' else None
    audit={'claim_id':c['claim_id'],'predicate_id':c['query_id'],'card_id':c['card_id'],'card_key':c['card_key'],'source_path':c['source_path'],'source_sha256':source_by[c['card_id']]['sha256'],'roles':role_spans,'common_heading_path':common,'relation':relation,'unrelated_section_combination':False,'target_role_present':True,'benefit_role_present':True,'condition_role_count':len(spec['condition']),'condition_evidence_mode':condition_mode,'limitation':limitation,'audit_status':'PASS'}
    claim_audits.append(audit)
audit_by={r['claim_id']:r for r in claim_audits}; assert len(audit_by)==33
assert any(r['line']==79 and r['role']=='condition' for r in audit_by['Q01_C03']['roles'])
special={'Q04_C03','Q04_C09','Q02_C02','Q04_C02','Q07_C02','Q09_C02','Q06_C02','Q01_C08','Q02_C04','Q04_C04','Q07_C04','Q08_C04'}
assert all(audit_by[x]['audit_status']=='PASS' for x in special)
write_jsonl(OUT/'followup2_atomic_claim_audit.jsonl',claim_audits)
gold_by={(r['query_id'],r['card_id']):r for r in parent_gold}; claim_by={(r['query_id'],r['card_id']):r for r in claims}
positive_counts=[sum(gold_by[(f'Q{i:02d}',card)]['label']=='positive' for card,_ in cards) for i in range(1,11)]; assert positive_counts==[5,5,2,5,2,3,3,3,3,2]
labels=[]; combo_evidence=[]; positive_provenance=[]; positive_sets=[]; predicate_pairs=[]
def compact_claim(claim_id):
    a=audit_by[claim_id]; return {'claim_id':claim_id,'predicate_id':a['predicate_id'],'source_path':a['source_path'],'source_sha256':a['source_sha256'],'common_heading_path':a['common_heading_path'],'roles':a['roles'],'condition_evidence_mode':a['condition_evidence_mode'],'limitation':a['limitation']}
for q,parent,_ in PARAPHRASES:
    for card,card_key in cards:
        src=gold_by[(parent,card)]; passed=[parent] if src['label']=='positive' else []; claim_ids=[src['claim_id']] if passed else []; claims_compact=[compact_claim(x) for x in claim_ids]
        row={'query_id':q,'cohort':'paraphrase','parent_query_ids_json':json.dumps([parent]),'card_id':card,'card_key':card_key,'label':src['label'],'claim_ids_json':json.dumps(claim_ids),'passed_predicate_ids_json':json.dumps(passed),'failed_predicate_ids_json':json.dumps([] if passed else [parent]),'passed_claim_sources_json':json.dumps(claims_compact,ensure_ascii=False,separators=(',',':')),'gold_source':'frozen_parent_gold_plus_reaudited_atomic_provenance'}; labels.append(row)
        if passed: positive_provenance.append({'query_id':q,'cohort':'paraphrase','card_id':card,'card_key':card_key,'claims':claims_compact,'all_required_claims_audited':True})
for q,a,b,_,expected in COMBINATIONS:
    predicate_pairs.append(tuple(sorted([a,b]))); actual=[]
    for card,card_key in cards:
        passed=[p for p in [a,b] if gold_by[(p,card)]['label']=='positive']; failed=[p for p in [a,b] if p not in passed]; claim_ids=[claim_by[(p,card)]['claim_id'] for p in passed]; claims_compact=[compact_claim(x) for x in claim_ids]; is_pos=len(passed)==2
        labels.append({'query_id':q,'cohort':'and_combination','parent_query_ids_json':json.dumps([a,b]),'card_id':card,'card_key':card_key,'label':'positive' if is_pos else 'negative','claim_ids_json':json.dumps(claim_ids if is_pos else []),'passed_predicate_ids_json':json.dumps(passed),'failed_predicate_ids_json':json.dumps(failed),'passed_claim_sources_json':json.dumps(claims_compact,ensure_ascii=False,separators=(',',':')),'gold_source':'frozen_atomic_intersection_plus_role_span_reaudit'})
        if is_pos:
            actual.append(card); positive_provenance.append({'query_id':q,'cohort':'and_combination','card_id':card,'card_key':card_key,'claims':claims_compact,'all_required_claims_audited':True})
            for p,claim_id in zip([a,b],claim_ids):
                au=audit_by[claim_id]; combo_evidence.append({'query_id':q,'card_id':card,'card_key':card_key,'predicate_id':p,'claim_id':claim_id,'source_path':au['source_path'],'source_sha256':au['source_sha256'],'common_heading_path_json':json.dumps(au['common_heading_path'],ensure_ascii=False),'role_spans_json':json.dumps(au['roles'],ensure_ascii=False,separators=(',',':')),'unrelated_section_combination':False,'raw_span_audit':'PASS'})
    assert actual==expected; positive_sets.append(tuple(actual))
assert len(labels)==200 and sum(r['label']=='positive' for r in labels)==56 and sum(r['label']=='negative' for r in labels)==144 and not any(r['label']=='insufficient' for r in labels)
assert len(positive_provenance)==56 and all(len(r['claims'])==(1 if r['cohort']=='paraphrase' else 2) for r in positive_provenance)
assert len(combo_evidence)==46 and len(predicate_pairs)==len(set(predicate_pairs))==10 and len(positive_sets)==len(set(positive_sets))==10
assert all(r['passed_predicate_ids_json'] and r['failed_predicate_ids_json'] and r['passed_claim_sources_json'] for r in labels if r['label']=='negative')
write_csv(OUT/'followup2_gold_labels.csv',labels); write_csv(OUT/'followup2_combination_evidence.csv',combo_evidence); write_jsonl(OUT/'followup2_positive_provenance.jsonl',positive_provenance)
query_freeze={'version':3,'declared_before_retrieval_result_reads':True,'restricted_result_reads_before_freeze':0,'ordered_query_sha256':[(r['query_id'],r['query_sha256']) for r in query_rows],'queries_csv_sha256':sha(OUT/'followup2_queries.csv'),'semantic_review_sha256':sha(OUT/'followup2_semantic_review.json'),'exact_normalized_duplicate_count':0,'parent_exact_normalized_overlap_count':0,'p04_canonical_intent':'direct discount OR cashback OR point accrual'}
gold_freeze={'version':3,'declared_before_retrieval_result_reads':True,'restricted_result_reads_before_freeze':0,'label_rows':200,'positive':56,'negative':144,'insufficient':0,'unique_atomic_claims_reaudited':33,'positive_provenance_rows':56,'combination_evidence_rows':46,'q01_c06_condition_evidence_mode':'implicit_no_stated_requirement_in_document','gold_labels_sha256':sha(OUT/'followup2_gold_labels.csv'),'atomic_claim_audit_sha256':sha(OUT/'followup2_atomic_claim_audit.jsonl'),'positive_provenance_sha256':sha(OUT/'followup2_positive_provenance.jsonl'),'combination_evidence_sha256':sha(OUT/'followup2_combination_evidence.csv')}
write_json(OUT/'followup2_query_freeze.json',query_freeze); write_json(OUT/'followup2_gold_freeze.json',gold_freeze)
gold_freeze_digest=hashlib.sha256(canon({'query':query_freeze,'gold':gold_freeze})).hexdigest()

# Retrieval result artifacts are first read only after v2 query/gold freeze.
restricted_names={'ranking_freeze.jsonl','per_query_metrics.csv','pair_scores.csv','followup1_pair_scores.csv'}; assert all((OUT/n).is_file() for n in restricted_names)
existing_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup2_')); existing_before={n:sha(OUT/n) for n in existing_names}
f1=json.loads((OUT/'followup1_contract.json').read_text()); c21=json.loads((ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json').read_text())
assert sha(OUT/'followup1_contract.json')=='022ab18a8a6e2c5c398299d608a8baca7ccb391f323f67b7c341e7710984b58c' and f1['max_unique_chunks_per_bundle']==5 and f1['bundle_document_token_cap']==4096
contract={'version':3,'name':'Follow-up 2 — paraphrase + independent AND-combination generalization','provenance':'Codex coder agent','declared_before_results':True,'scope':'same frozen 10-card development corpus only','systems':{'baseline':'old chunks + RRF 0.4/0.6 k60 depth50 + D20 + frozen regex selective BGE','candidate':'structural direct-body + same RRF/D20 + frozen Follow-up1 same-card 1-hop bundle max5/doc4096 + all BGE'},'bundle_binding':{'followup1_contract_sha256':sha(OUT/'followup1_contract.json'),'relation_order':f1['relation_order'],'model_revision':f1['bge']['revision'],'scorer_input_allowlist':f1['scorer_text'],'rule_changes_for_new_queries':0},'classifier_rule_sha256':c21['phase_a']['classifier_rule_sha256'],'metrics_per_cohort':['Card Precision@3','Card Precision@5','Card Recall@3','Card Recall@5','Evidence-supported Card Recall@5','Evidence Accuracy@3','Evidence Accuracy@5','Candidate Card Recall@20','Bundle-reachable Evidence Recall@20','zero-hit','raw Top20 duplicate rate','raw Top20 unique cards','paired delta','W/L/T'],'gate':{'Candidate Card Recall@20':{'paraphrase':'candidate >= same-cohort baseline AND candidate >= 0.90','and_combination':'candidate >= same-cohort baseline AND candidate >= 0.90'},'Bundle-reachable Evidence Recall@20':{'paraphrase':'candidate >= same-cohort baseline AND candidate >= 0.90','and_combination':'candidate >= same-cohort baseline AND candidate >= 0.90'},'nonregression_each_cohort':{'paraphrase':['Card Recall@3','Card Recall@5','Card Precision@5','Evidence Accuracy@5'],'and_combination':['Card Recall@3','Card Recall@5','Card Precision@5','Evidence Accuracy@5']},'Evidence-supported Card Recall@5':{'and_combination':'strict improvement and W>L','paraphrase':'non-regression and W>=L'},'zero_hit_increase_each_cohort':{'paraphrase':0,'and_combination':0},'catastrophic_loss_count_all_queries':0,'catastrophic_loss_count_each_cohort':{'paraphrase':0,'and_combination':0},'catastrophic_loss_definition':'baseline Card Recall@5 > 0 and candidate == 0, or query Card Recall@5 delta <= -0.5'},'dispositions':['query_and_compositional_generalization_dev_signal','robustness_observed_but_no_candidate_advantage','generalization_not_supported'],'always':['dev_signal_only','not_eligible_for_promotion','strict_improvement_is_not_causal_or_operational_evidence'],'limitations':['Q01_C06 uses implicit_no_stated_requirement_in_document; it has no direct no-spend-condition span and does not prove absence of a requirement.'],'cross_corpus_causal_claim_forbidden':True,'execution_preflight':{'api':0,'network':0,'gpu':0,'model_load_or_scoring':0,'chroma':0,'new_embedding':0,'package_install':0},'external_guard':'before API/model/result access, recompute approval core plus contract/query freeze/gold freeze canonical and raw SHA; all must match frozen and user expected values'}
write_json(OUT/'followup2_contract.json',contract)
enc=tiktoken.get_encoding('cl100k_base'); per_tokens=[len(enc.encode(r['query_text'])) for r in query_rows]; token_total=sum(per_tokens)
request_items=[{'order':r['request_order'],'query_id':r['query_id'],'text':r['query_text'],'sha256':r['query_sha256'],'estimated_tokens':tok} for r,tok in zip(query_rows,per_tokens)]
embedding_plan={'version':3,'model':'text-embedding-3-small','ordered_items':request_items,'item_count':20,'estimated_token_count':token_total,'character_count':sum(len(r['query_text']) for r in query_rows),'max_items':20,'max_tokens':token_total,'max_requests':1,'expected_shape':[20,1536],'expected_dtype':'float32','documents_or_gold_or_evaluation_fields_transmitted':False,'price_usd':None,'network_api_new_embedding_current_run':0,'approval_status':'awaiting_user_approval'}; write_json(OUT/'followup2_embedding_plan.json',embedding_plan)
frozen_artifacts={}
for name in ['followup2_contract.json','followup2_query_freeze.json','followup2_gold_freeze.json']:
    obj=json.loads((OUT/name).read_text()); frozen_artifacts[name]={'raw_sha256':sha(OUT/name),'canonical_sha256':hashlib.sha256(canon(obj)).hexdigest()}
approval_core={'schema_version':3,'experiment':'24_followup2','provider':'OpenAI embeddings API','model':'text-embedding-3-small','ordered_items':request_items,'max_items':20,'max_tokens':token_total,'max_requests':1,'transmit_only':'ordered query text','prohibited_fields':['documents','gold','atomic claims','source spans','labels','evaluation fields'],'gold_freeze_digest':gold_freeze_digest,'frozen_pre_result_artifacts':frozen_artifacts,'mismatch_policy':'fail before API, model, scoring, ranking, metric, or result access'}
core_sha=hashlib.sha256(canon(approval_core)).hexdigest(); approval={'approval_core':approval_core,'approval_core_sha256':core_sha,'status':{'state':'awaiting_user_approval','mutable_outside_approval_core':True}}; write_json(OUT/'followup2_approval_manifest.json',approval)
BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'; bge_before=tree_state(BGE_ROOT)
source_manifest={'version':3,'inputs':input_before,'raw_sources':{r['source_path']:r['sha256'] for r in sources},'old_chunks':{'rows':sum(1 for _ in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').open(encoding='utf-8')),'sha256':sha(ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl')},'structural_chunks':{'rows':sum(1 for _ in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').open(encoding='utf-8')),'sha256':sha(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl')},'hierarchy_sha256':sha(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'),'bge':bge_before,'followup1_bundle_contract_sha256':sha(OUT/'followup1_contract.json')}; write_json(OUT/'followup2_source_manifest.json',source_manifest)
input_after={str(p.relative_to(ROOT)):sha(p) for p in base_inputs}; bge_after=tree_state(BGE_ROOT); existing_after={n:sha(OUT/n) for n in existing_names}; assert input_after==input_before and bge_after==bge_before and existing_after==existing_before
output_names=['followup2_contract.json','followup2_queries.csv','followup2_semantic_review.json','followup2_atomic_claim_audit.jsonl','followup2_positive_provenance.jsonl','followup2_combination_evidence.csv','followup2_gold_labels.csv','followup2_query_freeze.json','followup2_gold_freeze.json','followup2_embedding_plan.json','followup2_approval_manifest.json','followup2_source_manifest.json']
integrity={'version':3,'status':'PASS','query_rows':20,'normalized_duplicate_count':0,'label_rows':200,'positive':56,'negative':144,'insufficient':0,'unique_atomic_claims_reaudited':33,'positive_provenance_rows':56,'combination_evidence_rows':46,'q01_c03_condition_line':79,'q01_c06_condition_evidence_mode':'implicit_no_stated_requirement_in_document','role_span_raw_exact':True,'unrelated_section_combinations':0,'combination_predicate_pair_duplicates':0,'combination_positive_set_duplicates':0,'negative_passed_and_failed_provenance_fields':True,'restricted_result_reads_before_query_gold_freeze':0,'api_network_gpu_model_chroma_new_embedding_package_install':0,'approval_status':'awaiting_user_approval','approval_core_sha256':core_sha,'frozen_pre_result_artifacts':frozen_artifacts,'input_hashes_before':input_before,'input_hashes_after':input_after,'bge_before':bge_before,'bge_after':bge_after,'existing_outputs_before':existing_before,'existing_outputs_after':existing_after,'existing_outputs_preserved':True,'output_hashes':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','run_manifest_sha256':'FINALIZE_AFTER_SAVE'}; write_json(OUT/'followup2_integrity.json',integrity)
run_manifest={'version':3,'phase':'followup2_final_pre_result_freeze','execution':'fresh kernel; only corrected followup2 v3 preflight cell executed','current_external_calls':0,'current_gpu_model_scoring':0,'approval_status':'awaiting_user_approval','approval_core_sha256':core_sha,'frozen_pre_result_artifacts':frozen_artifacts,'inputs_before':input_before,'inputs_after':input_after,'existing_outputs_preserved':True,'outputs':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','self_hash_policy':'followup2_integrity.json and followup2_preflight_run_manifest.json excluded'}; write_json(OUT/'followup2_preflight_run_manifest.json',run_manifest)
print(json.dumps({'status':'PASS','version':3,'queries':20,'labels':200,'positive':56,'negative':144,'atomic_claims_reaudited':33,'positive_provenance':56,'combination_evidence':46,'estimated_tokens':token_total,'max_requests':1,'approval_core_sha256':core_sha,'external_calls':0,'approval_status':'awaiting_user_approval'},ensure_ascii=False,indent=2))

{
  "status": "PASS",
  "version": 3,
  "queries": 20,
  "labels": 200,
  "positive": 56,
  "negative": 144,
  "atomic_claims_reaudited": 33,
  "positive_provenance": 56,
  "combination_evidence": 46,
  "estimated_tokens": 1179,
  "max_requests": 1,
  "approval_core_sha256": "fca33efe64c4dfe0f728c9c7e7eeb7d5ea9b61a48b571f75416bca1f2bc9454b",
  "external_calls": 0,
  "approval_status": "awaiting_user_approval"
}


In [ ]:
# Not executed in preflight. Verify every immutable pre-result artifact before any API/model/result access.
import hashlib, json, os
from pathlib import Path
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None)
assert ROOT is not None, 'fail-closed: cwd must be repository root or notebooks/'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
canon=lambda x: json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()
raw_sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
load_json=lambda name: json.loads((OUT/name).read_text(encoding='utf-8'))
manifest=load_json('followup2_approval_manifest.json')
computed=hashlib.sha256(canon(manifest['approval_core'])).hexdigest()
assert computed==manifest['approval_core_sha256'], 'fail-closed: stored approval core digest mismatch'
assert os.environ.get('APPROVED_24_FOLLOWUP2_CORE_SHA256')==computed, 'fail-closed: user-provided expected digest mismatch'
frozen=manifest['approval_core']['frozen_pre_result_artifacts']
for name in ('followup2_contract.json','followup2_query_freeze.json','followup2_gold_freeze.json'):
    obj=load_json(name); actual={'raw_sha256':raw_sha(OUT/name),'canonical_sha256':hashlib.sha256(canon(obj)).hexdigest()}
    assert actual==frozen[name], f'fail-closed: frozen artifact drift: {name}'
query_freeze=load_json('followup2_query_freeze.json')
assert raw_sha(OUT/'followup2_queries.csv')==query_freeze['queries_csv_sha256'], 'fail-closed: query CSV drift'
assert raw_sha(OUT/'followup2_semantic_review.json')==query_freeze['semantic_review_sha256'], 'fail-closed: semantic review drift'
gold_freeze=load_json('followup2_gold_freeze.json')
for key,name in {'gold_labels_sha256':'followup2_gold_labels.csv','atomic_claim_audit_sha256':'followup2_atomic_claim_audit.jsonl','positive_provenance_sha256':'followup2_positive_provenance.jsonl','combination_evidence_sha256':'followup2_combination_evidence.csv'}.items():
    assert raw_sha(OUT/name)==gold_freeze[key], f'fail-closed: gold/source freeze drift: {name}'
assert manifest['status']['state']=='approved', 'fail-closed: mutable approval status is not approved'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL','0')=='1', 'fail-closed: explicit execution approval required'
APPROVAL_CORE=manifest['approval_core']; EXPECTED_APPROVAL_SHA=computed
print({'approval_guard':'PASS','approval_core_sha256':computed,'frozen_artifacts':len(frozen)})

## Follow-up 2 — 승인된 external execution + evaluation

승인 core `fca33efe...9454b`의 ordered 개발 질의 20개만 embedding하며, 기존 fail-closed guard가 API·모델·결과 접근보다 먼저 실행됩니다. 검색 ranking과 BGE score를 gold 의미 해석 전에 동결하고, 결과는 개발 진단 신호일 뿐 운영 승격 근거가 아닙니다.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from decimal import Decimal
import csv, hashlib, json, math, os, re, resource, time, unicodedata
import numpy as np

cwd=Path.cwd().resolve()
ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None)
assert ROOT is not None
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
OLD_CHUNKS=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl'
NEW_CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'
HIERARCHY=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'
OLD_CACHE=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/embedding_cache/text-embedding-3-small'
NEW_CACHE=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/embedding_cache/text-embedding-3-small'
BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'
assert EXPECTED_APPROVAL_SHA=='fca33efe64c4dfe0f728c9c7e7eeb7d5ea9b61a48b571f75416bca1f2bc9454b'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL')=='1'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def canonical(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x]
def tree_state(root):
    files=sorted(p for p in root.rglob('*') if p.is_file()); mapping={str(p.relative_to(root)):sha(p) for p in files}
    return {'file_count':len(files),'total_bytes':sum(p.stat().st_size for p in files),'files':mapping,'digest':hashlib.sha256(canonical(mapping).encode()).hexdigest()}
query_rows=list(csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline='')))
assert len(query_rows)==20 and [int(r['request_order']) for r in query_rows]==list(range(1,21))
query_map={r['query_id']:r['query_text'] for r in query_rows}
ordered_items=APPROVAL_CORE['ordered_items']
assert [r['query_id'] for r in query_rows]==[r['query_id'] for r in ordered_items]
assert all(hashlib.sha256(r['query_text'].encode()).hexdigest()==item['sha256'] and r['query_text']==item['text'] for r,item in zip(query_rows,ordered_items))
assert len(ordered_items)==APPROVAL_CORE['max_items']==20 and sum(r['estimated_tokens'] for r in ordered_items)==APPROVAL_CORE['max_tokens']==1179 and APPROVAL_CORE['max_requests']==1
assert APPROVAL_CORE['transmit_only']=='ordered query text' and set(APPROVAL_CORE['prohibited_fields'])=={'documents','gold','atomic claims','source spans','labels','evaluation fields'}

old_usage=json.loads((ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json').read_text())
new_usage=json.loads((ROOT/'notebooks/data/22_structural_heading_chunking_ablation/embedding_usage.json').read_text())
old_cache_files=[OLD_CACHE/(b['batch_fingerprint']+'.npz') for b in old_usage['batches']]
new_cache_files=[ROOT/b['cache_path'] for b in new_usage['batches']]
assert len(old_cache_files)==6 and len(new_cache_files)==3 and all(p.is_file() for p in old_cache_files+new_cache_files)
immutable_paths=[OLD_CHUNKS,NEW_CHUNKS,HIERARCHY,*old_cache_files,*new_cache_files]
immutable_before={str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}
bge_before_f2=tree_state(BGE_ROOT)
preflight_integrity=json.loads((OUT/'followup2_integrity.json').read_text())
protected_before=preflight_integrity['existing_outputs_after']
assert all(sha(OUT/path)==digest for path,digest in protected_before.items())
preflight_names=['followup2_contract.json','followup2_queries.csv','followup2_semantic_review.json','followup2_atomic_claim_audit.jsonl','followup2_positive_provenance.jsonl','followup2_combination_evidence.csv','followup2_gold_labels.csv','followup2_query_freeze.json','followup2_gold_freeze.json','followup2_embedding_plan.json','followup2_source_manifest.json','followup2_preflight_run_manifest.json']
preflight_before={name:sha(OUT/name) for name in preflight_names}
approval_manifest_before=sha(OUT/'followup2_approval_manifest.json')

cache_dir=OUT/'embedding_cache/text-embedding-3-small'; cache_dir.mkdir(parents=True,exist_ok=True)
query_cache=cache_dir/(EXPECTED_APPROVAL_SHA+'.npz'); attempt_path=OUT/'followup2_api_attempt_state.json'
current_api_requests=0; api_started=time.perf_counter()
if query_cache.is_file():
    assert attempt_path.is_file()
    attempt=json.loads(attempt_path.read_text())
    assert attempt['state']=='completed' and attempt['approval_core_sha256']==EXPECTED_APPROVAL_SHA and attempt['request_count']==1
else:
    assert not attempt_path.exists(), 'fail-closed: prior started/failed attempt exists; retry is outside the one-request approval'
    write_json(attempt_path,{'state':'started','approval_core_sha256':EXPECTED_APPROVAL_SHA,'request_count':1,'item_count':20,'estimated_tokens':1179,'retry_allowed':False})
    from dotenv import dotenv_values
    from openai import OpenAI
    api_key=dotenv_values(ROOT/'.env').get('OPENAI_API_KEY')
    assert api_key, 'OPENAI_API_KEY is required but never printed or stored'
    client=OpenAI(api_key=api_key,max_retries=0,timeout=120.0)
    response=client.embeddings.create(model='text-embedding-3-small',input=[r['text'] for r in ordered_items],encoding_format='float')
    current_api_requests=1
    assert len(response.data)==20 and sorted(x.index for x in response.data)==list(range(20))
    query_vectors=np.asarray([x.embedding for x in sorted(response.data,key=lambda x:x.index)],dtype=np.float32)
    actual_tokens=int(response.usage.prompt_tokens)
    assert actual_tokens<=1179
    np.savez_compressed(query_cache,embeddings=query_vectors,hashes=np.asarray([r['sha256'] for r in ordered_items]),approval_core_sha256=np.asarray(EXPECTED_APPROVAL_SHA),creation_api_requests=np.asarray(1),creation_input_tokens=np.asarray(actual_tokens))
    write_json(attempt_path,{'state':'completed','approval_core_sha256':EXPECTED_APPROVAL_SHA,'request_count':1,'item_count':20,'estimated_tokens':1179,'actual_input_tokens':actual_tokens,'retry_allowed':False,'cache_sha256':sha(query_cache),'documents_or_gold_transmitted':False})
    del client,response,api_key
with np.load(query_cache,allow_pickle=False) as z:
    query_vectors=z['embeddings']; cached_hashes=z['hashes'].tolist(); cached_core=str(z['approval_core_sha256']); historical_requests=int(z['creation_api_requests']); historical_tokens=int(z['creation_input_tokens'])
assert query_vectors.shape==(20,1536) and query_vectors.dtype==np.float32 and np.isfinite(query_vectors).all()
assert cached_hashes==[r['sha256'] for r in ordered_items] and cached_core==EXPECTED_APPROVAL_SHA and historical_requests==1 and historical_tokens<=1179
api_seconds=time.perf_counter()-api_started
write_json(OUT/'followup2_embedding_usage.json',{'approval_core_sha256':EXPECTED_APPROVAL_SHA,'items':20,'planned_tokens':1179,'actual_input_tokens':historical_tokens,'creation_api_requests_total':1,'current_run_api_requests':current_api_requests,'current_run_network_requests':current_api_requests,'cache_path':str(query_cache.relative_to(ROOT)),'cache_sha256':sha(query_cache),'dimension':1536,'dtype':'float32','finite':True,'api_seconds':api_seconds,'documents_or_gold_or_evaluation_fields_transmitted':False,'retry_count':0,'max_requests':1,'cost_usd':None,'cost_reason':'official price was not newly verified in this run'})

def cache_rows(paths):
    hashes=[]; matrices=[]
    for path in paths:
        with np.load(path,allow_pickle=False) as z:
            matrix=z['embeddings']; batch_hashes=z['hashes'].tolist()
            assert matrix.dtype==np.float32 and matrix.shape==(len(batch_hashes),1536) and np.isfinite(matrix).all()
            hashes.extend(batch_hashes); matrices.append(matrix.copy())
    return hashes,np.vstack(matrices)
old_hashes,old_matrix=cache_rows(old_cache_files); new_hashes,new_matrix=cache_rows(new_cache_files)
old=load_jsonl(OLD_CHUNKS); new=load_jsonl(NEW_CHUNKS); hierarchy=load_jsonl(HIERARCHY)
assert len(old)==327 and len(new)==147
old_by_id={r['id']:r for r in old}; new_by_id={r['chunk_id']:r for r in new}; node_by_id={r['node_id']:r for r in hierarchy}
expected_old_hashes=[hashlib.sha256(r['document'].encode()).hexdigest() for r in old]
expected_new_hashes=[hashlib.sha256(r['retrieval_text'].encode()).hexdigest() for r in new]
assert len(old_hashes)==357 and old_hashes[:327]==expected_old_hashes and len(new_hashes)==147 and new_hashes==expected_new_hashes
old_vectors={r['id']:old_matrix[i] for i,r in enumerate(old)}; new_vectors={r['chunk_id']:new_matrix[i] for i,r in enumerate(new)}
query_vectors_by_id={r['query_id']:query_vectors[i] for i,r in enumerate(query_rows)}

RAW_TOKEN=re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*',re.IGNORECASE)
def normalized_text(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
def canonical_decimal(value):
    rendered=format(Decimal(str(value).replace(',','')).normalize(),'f'); rendered=rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered
    return '0' if rendered in {'','-0'} else rendered
def search_tokens(value):
    text=normalized_text(value); tokens=list(RAW_TOKEN.findall(text))
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?',text):
        joined=run.replace(' ','')
        for size in (2,3,4): tokens.extend('ko'+str(size)+'_'+joined[i:i+size] for i in range(max(0,len(joined)-size+1)))
    consumed=[]
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원',text):
        tokens.append('money_krw_'+canonical_decimal(Decimal(match.group(1).replace(',',''))*10000+Decimal(match.group(2).replace(',',''))*1000)); consumed.append(match.span())
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원',text):
        if any(a<=match.start() and match.end()<=b for a,b in consumed): continue
        tokens.append('money_krw_'+canonical_decimal(Decimal(match.group(1).replace(',',''))*{'만':10000,'천':1000,None:1}[match.group(2)]))
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%',text): tokens.append('percent_'+canonical_decimal(match.group(1)))
    for match in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)',text): tokens.append('period_'+(match.group(1) or 'none')+'_'+canonical_decimal(match.group(2))+'_'+match.group(3))
    return tokens
def bm25_rank(query,documents,k1=1.5,b=.75):
    qt=search_tokens(query); docs={k:search_tokens(v) for k,v in documents.items()}; df=Counter(t for ts in docs.values() for t in set(ts)); avg=sum(map(len,docs.values()))/len(docs); scores={}
    for cid,tokens in docs.items():
        freq=Counter(tokens); score=0.0
        for token in qt:
            f=freq[token]
            if f:
                idf=math.log(1+(len(docs)-df[token]+.5)/(df[token]+.5)); score+=idf*f*(k1+1)/(f+k1*(1-b+b*len(tokens)/avg))
        scores[cid]=score
    return sorted(scores,key=lambda cid:(-scores[cid],cid)),scores
def vector_rank(qv,vectors):
    scores={cid:float(np.sum((v-qv)**2,dtype=np.float64)) for cid,v in vectors.items()}
    assert all(math.isfinite(x) for x in scores.values())
    return sorted(scores,key=lambda cid:(scores[cid],cid)),scores
def rrf(vector,bm25):
    score=defaultdict(float)
    for weight,ranked in ((.4,vector[:50]),(.6,bm25[:50])):
        for rank,cid in enumerate(ranked,1): score[cid]+=weight/(60+rank)
    fused=sorted(score,key=lambda cid:(-score[cid],cid))[:50]
    assert len(fused)==len(set(fused))==50 and set(fused)<=set(vector[:50])|set(bm25[:50])
    return fused,score

PROPER=[('proper_issuer_product',re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$')),('proper_issuer_direct',re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$')),('proper_issuer_noun',re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$')),('proper_where_action',re.compile(r'어디서\s*(?:발급|발행|출시)'))]
NUMERIC=[('numeric_direct_amount',re.compile(r'얼마(?:인가|나)?$')),('numeric_direct_count',re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$')),('numeric_direct_how_much',re.compile(r'얼마나\s*(?:할인|적립|차감|청구)')),('numeric_target_end',re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$'))]
def classify(value):
    text=re.sub(r'[?!。？！.]+$','',normalized_text(value)).strip()
    for rid,p in PROPER:
        m=p.search(text)
        if m:return 'proper_noun',rid,m.group(0)
    if '연회비 면제 조건' not in text:
        for rid,p in NUMERIC:
            m=p.search(text)
            if m:return 'numeric_condition',rid,m.group(0)
    return 'semantic','semantic_fallback',''

search_started=time.perf_counter(); cpu_started=time.process_time(); rss_started=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
ranking_rows=[]; classifications=[]
old_docs={cid:r['document'] for cid,r in old_by_id.items()}; new_docs={cid:r['retrieval_text'] for cid,r in new_by_id.items()}
for row in query_rows:
    q=row['query_id']; query=row['query_text']; category,rule,span=classify(query)
    classifications.append({'query_id':q,'cohort':row['cohort'],'category':category,'matched_rule_id':rule,'matched_span':span,'route':'reranker' if category=='semantic' else 'no_reranker'})
    for corpus,docs,vectors,by_id in [('old',old_docs,old_vectors,old_by_id),('structural',new_docs,new_vectors,new_by_id)]:
        vr,_=vector_rank(query_vectors_by_id[q],vectors); br,_=bm25_rank(query,docs); fused,scores=rrf(vr,br)
        eligible=[cid for cid in fused if corpus=='structural' or by_id[cid]['metadata']['level'] in {'section','benefit'}]
        assert len(eligible)>=20
        top20=eligible[:20]
        ranking_rows.append({'corpus':corpus,'query_id':q,'cohort':row['cohort'],'vector_top50':vr[:50],'bm25_top50':br[:50],'rrf_top50':fused,'eligible_top20':top20,'eligible_available_in_rrf_top50':len(eligible),'rrf_scores_top20':[scores[cid] for cid in top20]})
assert len(ranking_rows)==40 and Counter(r['category'] for r in classifications)=={'semantic':20}
rank_path=OUT/'followup2_ranking_freeze.jsonl'; rank_path.write_text(''.join(canonical(r)+'\n' for r in ranking_rows),encoding='utf-8')
with (OUT/'followup2_query_classification.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(classifications[0])); w.writeheader(); w.writerows(classifications)

children=defaultdict(list)
for node in hierarchy:
    if node['parent_id'] is not None: children[node['parent_id']].append(node)
for rows in children.values(): rows.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
bundles=[]; traces=[]
for ranking in [r for r in ranking_rows if r['corpus']=='structural']:
    top20=ranking['eligible_top20']; by_card=defaultdict(list)
    for rank,cid in enumerate(top20,1): by_card[new_by_id[cid]['metadata']['card_key']].append((rank,cid))
    for card_key,seeds in sorted(by_card.items(),key=lambda x:(x[1][0][0],x[0])):
        best_rank,seed_id=seeds[0]; seed=new_by_id[seed_id]; seed_node=node_by_id[seed['metadata']['node_id']]; selected=[]; relation=[]; parent_context=None
        def add(cid,kind,source_node):
            if len(selected)>=5 or cid in selected: return
            candidate=new_by_id[cid]; assert candidate['metadata']['card_key']==card_key
            selected.append(cid); relation.append({'selection_order':len(selected),'chunk_id':cid,'relation':kind,'node_id':candidate['metadata']['node_id'],'parent_id':candidate['metadata']['parent_id'],'part_index':candidate['metadata']['part_index'],'part_count':candidate['metadata']['part_count'],'source_relation_node_id':source_node})
        add(seed_id,'best_rrf_seed',seed_node['node_id'])
        same_node=[new_by_id[c] for c in seed_node['search_chunk_ids'] if c!=seed_id]
        same_node.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
        for c in same_node: add(c['chunk_id'],'same_node_adjacent_part',seed_node['node_id'])
        parent=node_by_id.get(seed_node['parent_id'])
        if parent is not None and parent['parent_id'] is not None:
            if not parent['heading_only']:
                for cid in parent['search_chunk_ids']: add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
            else:
                parent_context=parent['heading_text']; seed_line=seed_node['heading_line_number'] if seed_node['heading_line_number'] is not None else 10**12
                siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']]
                siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-seed_line),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                for n in siblings:
                    for cid in n['search_chunk_ids']: add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
        for child in children[seed_node['node_id']]:
            if not child['heading_only']:
                for cid in child['search_chunk_ids']: add(cid,'seed_node_direct_child',seed_node['node_id'])
        for _,cid in seeds[1:]: add(cid,'same_card_other_top20_seed',seed_node['node_id'])
        assert 1<=len(selected)<=5 and len(selected)==len(set(selected))
        sections=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
        if parent_context: sections.append('[상위 제목]\n'+parent_context)
        for i,cid in enumerate(selected,1):
            c=new_by_id[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'
            sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
        bundle_text='\n\n'.join(sections)
        bundle={'query_id':ranking['query_id'],'cohort':ranking['cohort'],'card_key':card_key,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'candidate_top20_card_set':sorted(by_card),'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':bundle_text,'bundle_sha256':hashlib.sha256(bundle_text.encode()).hexdigest(),'bundle_characters':len(bundle_text),'relation_trace':relation}
        bundles.append(bundle)
        for item in relation: traces.append({'query_id':ranking['query_id'],'cohort':ranking['cohort'],'card_key':card_key,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'optional_parent_heading':parent_context or '',**item})
assert len({(r['query_id'],r['card_key']) for r in bundles})==len(bundles)
for ranking in [r for r in ranking_rows if r['corpus']=='structural']:
    assert {new_by_id[c]['metadata']['card_key'] for c in ranking['eligible_top20']}=={r['card_key'] for r in bundles if r['query_id']==ranking['query_id']}
bundle_path=OUT/'followup2_bundles.jsonl'; bundle_path.write_text(''.join(canonical(r)+'\n' for r in bundles),encoding='utf-8')
with (OUT/'followup2_bundle_trace.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(traces[0])); w.writeheader(); w.writerows(traces)
search_wall=time.perf_counter()-search_started; search_cpu=time.process_time()-cpu_started; rss_end=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
write_json(OUT/'followup2_search_resources.json',{'wall_seconds':search_wall,'cpu_seconds':search_cpu,'peak_rss_mib':rss_end/1024,'incremental_ru_maxrss_mib':max(0,rss_end-rss_started)/1024,'api_seconds':api_seconds,'current_api_requests':current_api_requests})
freeze={'created_after_frozen_query_gold_and_before_semantic_gold_read':True,'approval_guard_passed':True,'approval_core_sha256':EXPECTED_APPROVAL_SHA,'ranking_rows':40,'ranking_sha256':sha(rank_path),'query_cache_sha256':sha(query_cache),'search_contract':{'bm25_k1':1.5,'bm25_b':.75,'vector_weight':.4,'bm25_weight':.6,'rrf_k':60,'component_depth':50,'candidate_depth':20,'vector_distance':'numpy_squared_l2','tie':'chunk_id lexical'},'all_routes_semantic':True,'immutable_inputs_before':immutable_before,'bge_before':bge_before_f2}
write_json(OUT/'followup2_ranking_freeze.json',freeze)
build_freeze={'created_before_semantic_gold_read':True,'rule_hash_binding':sha(OUT/'followup1_contract.json'),'query_card_pairs':len(bundles),'bundle_rows':len(bundles),'trace_rows':len(traces),'bundle_sha256':sha(bundle_path),'trace_sha256':sha(OUT/'followup2_bundle_trace.csv'),'max_unique_chunks':max(len(r['selected_chunk_ids']) for r in bundles),'candidate_cards_only':True}
write_json(OUT/'followup2_bundle_build_freeze.json',build_freeze)
assert {str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}==immutable_before and tree_state(BGE_ROOT)==bge_before_f2
assert all(sha(OUT/path)==digest for path,digest in protected_before.items())
assert {name:sha(OUT/name) for name in preflight_names}==preflight_before and sha(OUT/'followup2_approval_manifest.json')==approval_manifest_before
print({'followup2_search_bundle':'PASS','current_api_requests':current_api_requests,'historical_api_requests':historical_requests,'ranking_rows':40,'bundle_pairs':len(bundles),'all_routes':'semantic'})

In [ ]:
# Local-only BGE scoring. Gold/labels/claims are not read in this cell.
import gc, statistics, subprocess
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL')=='1'
assert os.environ.get('RUN_APPROVED_24_GPU')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0'
assert os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
def gpu_snapshot():
    lines=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip().splitlines()
    line=next(x for x in lines if x.split(',')[0].strip()=='0'); uuid=line.split(',')[1].strip()
    raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip(); processes=[]
    for x in raw.splitlines():
        parts=[v.strip() for v in x.split(',',3)]
        if len(parts)==4 and parts[0]==uuid: processes.append({'gpu_uuid':parts[0],'pid':int(parts[1]),'process_name':parts[2],'used_memory_mib':float(parts[3])})
    return {'gpu_line':line,'processes':processes,'captured_at_unix':time.time()}

rankings=load_jsonl(OUT/'followup2_ranking_freeze.jsonl'); bundles=load_jsonl(OUT/'followup2_bundles.jsonl')
assert len(rankings)==40 and len(bundles)==json.loads((OUT/'followup2_bundle_build_freeze.json').read_text())['query_card_pairs']
heading_re=re.compile(r'(?m)^#{1,6}\s+(.+?)\s*$')
def old_augmented(chunk):
    meta=chunk['metadata']; m=heading_re.search(chunk['document']); heading=m.group(1).strip() if m else ''
    body=chunk['document'][:m.start()]+chunk['document'][m.end():] if m else chunk['document']
    path=[meta['issuer'],meta['card_name'],meta['level']]+([heading] if heading else [])
    return '[문서 경로]\n'+' > '.join(path)+'\n\n[본문]\n'+body.strip(),path
pairs=[]; audit=[]
for row in [r for r in rankings if r['corpus']=='old']:
    query=query_map[row['query_id']]
    for rank,cid in enumerate(row['eligible_top20'],1):
        text,path=old_augmented(old_by_id[cid])
        pairs.append({'pair_type':'old_chunk','query_id':row['query_id'],'cohort':row['cohort'],'candidate_id':cid,'card_key':old_by_id[cid]['metadata']['card_key'],'original_rank':rank,'query':query,'document':text})
        audit.append({'pair_type':'old_chunk','query_id':row['query_id'],'candidate_id':cid,'query_sha256':hashlib.sha256(query.encode()).hexdigest(),'document_sha256':hashlib.sha256(text.encode()).hexdigest(),'construction_allowlist':'query_text + issuer + card_name + level + first Markdown heading + document body','gold_or_evaluation_fields_used':False})
for row in bundles:
    query=query_map[row['query_id']]
    pairs.append({'pair_type':'structural_bundle','query_id':row['query_id'],'cohort':row['cohort'],'candidate_id':row['card_key'],'card_key':row['card_key'],'original_rank':int(row['best_seed_rrf_rank']),'query':query,'document':row['bundle_text']})
    audit.append({'pair_type':'structural_bundle','query_id':row['query_id'],'candidate_id':row['card_key'],'query_sha256':hashlib.sha256(query.encode()).hexdigest(),'document_sha256':row['bundle_sha256'],'construction_allowlist':'query_text + frozen Follow-up1 bundle text only','gold_or_evaluation_fields_used':False})
assert len(pairs)==400+len(bundles) and len({(r['pair_type'],r['query_id'],r['candidate_id']) for r in pairs})==len(pairs)
assert all(not r['gold_or_evaluation_fields_used'] for r in audit)
gpu_before=gpu_snapshot()
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
def score_all(batch_size):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); load_start=time.perf_counter()
    tokenizer=AutoTokenizer.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False)
    model=AutoModelForSequenceClassification.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0')
    load_seconds=time.perf_counter()-load_start; token_rows=[]
    for row in pairs:
        qn=len(tokenizer.encode(row['query'],add_special_tokens=False)); dn=len(tokenizer.encode(row['document'],add_special_tokens=False)); encoded=tokenizer(row['query'],row['document'],truncation='only_second',max_length=8192)
        truncated=max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192)
        if row['pair_type']=='structural_bundle':
            assert dn<=4096 and truncated==0
        token_rows.append({'query_tokens':qn,'document_tokens':dn,'input_tokens':len(encoded['input_ids']),'document_truncated_tokens':truncated})
    warm=tokenizer(['준비'],['준비'],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0'); warm_start=time.perf_counter()
    with torch.inference_mode(): assert torch.isfinite(model(**warm).logits).all()
    torch.cuda.synchronize(); warm_seconds=time.perf_counter()-warm_start; del warm
    scores=[]; started=time.perf_counter()
    try:
        for start in range(0,len(pairs),batch_size):
            batch=pairs[start:start+batch_size]; inputs=tokenizer([x['query'] for x in batch],[x['document'] for x in batch],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
            with torch.inference_mode(): logits=model(**inputs).logits.reshape(-1).float().cpu().numpy()
            assert len(logits)==len(batch) and np.isfinite(logits).all(); scores.extend(map(float,logits)); del inputs,logits
        torch.cuda.synchronize(); seconds=time.perf_counter()-started
        result=(scores,token_rows,{'load_seconds':load_seconds,'warmup_seconds':warm_seconds,'scoring_seconds':seconds,'pairs_per_second':len(pairs)/seconds,'batch_size':batch_size,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30})
    except torch.cuda.OutOfMemoryError:
        result=None
    del model,tokenizer,scores; gc.collect(); torch.cuda.empty_cache()
    return result
scored=score_all(2); oom_batch2=scored is None
if scored is None: scored=score_all(1)
assert scored is not None
scores,token_rows,model_resource=scored
assert len(scores)==len(pairs) and np.isfinite(np.asarray(scores)).all()
pair_rows=[]
for row,score,tokens in zip(pairs,scores,token_rows):
    pair_rows.append({k:row[k] for k in ('pair_type','query_id','cohort','candidate_id','card_key','original_rank')}|{'raw_logit':score,**tokens})
with (OUT/'followup2_pair_scores.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(pair_rows[0])); w.writeheader(); w.writerows(pair_rows)
for a,t in zip(audit,token_rows): a.update(t)
with (OUT/'followup2_scorer_input_audit.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0])); w.writeheader(); w.writerows(audit)
rank_rows=[]
for q in query_map:
    old_rows=sorted((r for r in pair_rows if r['query_id']==q and r['pair_type']=='old_chunk'),key=lambda r:(-r['raw_logit'],int(r['original_rank']),r['candidate_id']))
    bundle_rows=sorted((r for r in pair_rows if r['query_id']==q and r['pair_type']=='structural_bundle'),key=lambda r:(-r['raw_logit'],int(r['original_rank']),r['card_key']))
    assert len(old_rows)==20 and len(bundle_rows)==len({r['card_key'] for r in bundle_rows})
    rank_rows.append({'query_id':q,'cohort':next(r['cohort'] for r in old_rows),'old_chunk_ids':[r['candidate_id'] for r in old_rows],'old_raw_logits':[r['raw_logit'] for r in old_rows],'old_original_ranks':[int(r['original_rank']) for r in old_rows],'structural_card_keys':[r['card_key'] for r in bundle_rows],'structural_raw_logits':[r['raw_logit'] for r in bundle_rows],'structural_best_seed_ranks':[int(r['original_rank']) for r in bundle_rows]})
ranked_path=OUT/'followup2_scored_rankings.jsonl'; ranked_path.write_text(''.join(canonical(r)+'\n' for r in rank_rows),encoding='utf-8')
gpu_after=gpu_snapshot(); input_lengths=[r['input_tokens'] for r in pair_rows]; bundle_lengths=[r['document_tokens'] for r in pair_rows if r['pair_type']=='structural_bundle']
resources={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','trust_remote_code':False,'local_files_only':True,'dtype':'float16','max_length':8192,'truncation':'only_second','quality_pairs':len(pair_rows),'old_chunk_pairs':400,'structural_bundle_pairs':len(bundles),'oom_batch2':oom_batch2,**model_resource,'token_p50':statistics.median(input_lengths),'token_p95':float(np.percentile(input_lengths,95)),'token_max':max(input_lengths),'bundle_document_token_p50':statistics.median(bundle_lengths),'bundle_document_token_p95':float(np.percentile(bundle_lengths,95)),'bundle_document_token_max':max(bundle_lengths),'truncated_pairs':sum(r['document_truncated_tokens']>0 for r in pair_rows),'bundle_truncated_pairs':sum(r['document_truncated_tokens']>0 for r in pair_rows if r['pair_type']=='structural_bundle'),'gpu_before':gpu_before,'gpu_after':gpu_after,'physical_gpu':0,'shared_gpu_measurement_with_ollama_allowed':True,'network_api_download_new_embedding_chroma_package_install_during_scoring':0}
write_json(OUT/'followup2_model_resources.json',resources)
scoring_freeze={'created_before_semantic_gold_read':True,'pair_rows':len(pair_rows),'pair_scores_sha256':sha(OUT/'followup2_pair_scores.csv'),'rankings_sha256':sha(ranked_path),'input_audit_sha256':sha(OUT/'followup2_scorer_input_audit.csv'),'ranking_freeze_sha256':sha(OUT/'followup2_ranking_freeze.jsonl'),'bundle_sha256':sha(OUT/'followup2_bundles.jsonl'),'query_cache_sha256':sha(query_cache),'source_hashes_before':immutable_before,'bge_before':bge_before_f2}
write_json(OUT/'followup2_scoring_freeze.json',scoring_freeze)
assert tree_state(BGE_ROOT)==bge_before_f2 and {str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}==immutable_before and sha(query_cache)==json.loads((OUT/'followup2_embedding_usage.json').read_text())['cache_sha256']
assert all(sha(OUT/path)==digest for path,digest in protected_before.items()) and {name:sha(OUT/name) for name in preflight_names}==preflight_before
print({'followup2_scoring':'PASS','pairs':len(pair_rows),'bundle_pairs':len(bundles),'batch':model_resource['batch_size'],'oom_batch2':oom_batch2,'truncated':resources['truncated_pairs'],'peak_gib':resources['peak_allocated_gib']})

In [1]:
# Gold is first interpreted here, after ranking and scoring hashes are frozen.
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, json, math, os, re, statistics, time, unicodedata
import numpy as np

cwd=Path.cwd().resolve()
ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None)
assert ROOT is not None
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
OLD_CHUNKS=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl'
NEW_CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'
HIERARCHY=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'
BGE_ROOT=ROOT/'.cache/reranker/bge-reranker-v2-m3'
EXPECTED_APPROVAL_SHA='fca33efe64c4dfe0f728c9c7e7eeb7d5ea9b61a48b571f75416bca1f2bc9454b'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP2_EXTERNAL','0')=='0', 'API hard-disabled for CPU resume'
assert os.environ.get('RUN_APPROVED_24_GPU','0')=='0' and os.environ.get('CUDA_VISIBLE_DEVICES','')==''
assert os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def canonical(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x]
def tree_state(root):
    files=sorted(p for p in root.rglob('*') if p.is_file()); mapping={str(p.relative_to(root)):sha(p) for p in files}
    return {'file_count':len(files),'total_bytes':sum(p.stat().st_size for p in files),'files':mapping,'digest':hashlib.sha256(canonical(mapping).encode()).hexdigest()}

# Same fail-closed approval/freeze guard runs before reading saved rankings, pair scores, or semantic gold.
approval=json.loads((OUT/'followup2_approval_manifest.json').read_text())
computed=hashlib.sha256(canonical(approval['approval_core']).encode()).hexdigest()
assert computed==approval['approval_core_sha256']==EXPECTED_APPROVAL_SHA
assert os.environ.get('APPROVED_24_FOLLOWUP2_CORE_SHA256')==computed
for name,expected in approval['approval_core']['frozen_pre_result_artifacts'].items():
    obj=json.loads((OUT/name).read_text()); actual={'raw_sha256':sha(OUT/name),'canonical_sha256':hashlib.sha256(canonical(obj).encode()).hexdigest()}
    assert actual==expected
qfreeze=json.loads((OUT/'followup2_query_freeze.json').read_text())
assert sha(OUT/'followup2_queries.csv')==qfreeze['queries_csv_sha256'] and sha(OUT/'followup2_semantic_review.json')==qfreeze['semantic_review_sha256']
gfreeze=json.loads((OUT/'followup2_gold_freeze.json').read_text())
for key,name in {'gold_labels_sha256':'followup2_gold_labels.csv','atomic_claim_audit_sha256':'followup2_atomic_claim_audit.jsonl','positive_provenance_sha256':'followup2_positive_provenance.jsonl','combination_evidence_sha256':'followup2_combination_evidence.csv'}.items(): assert sha(OUT/name)==gfreeze[key]
assert approval['status']['state']=='approved'
attempt=json.loads((OUT/'followup2_api_attempt_state.json').read_text()); embedding_usage=json.loads((OUT/'followup2_embedding_usage.json').read_text())
assert attempt['state']=='completed' and attempt['request_count']==1 and not attempt['retry_allowed'] and attempt['approval_core_sha256']==EXPECTED_APPROVAL_SHA
query_cache=ROOT/embedding_usage['cache_path']
with np.load(query_cache,allow_pickle=False) as z:
    assert z['embeddings'].shape==(20,1536) and z['embeddings'].dtype==np.float32 and np.isfinite(z['embeddings']).all() and len(z['hashes'])==20
    historical_tokens=int(z['creation_input_tokens']); assert int(z['creation_api_requests'])==1 and str(z['approval_core_sha256'])==EXPECTED_APPROVAL_SHA
assert historical_tokens==1179 and sha(query_cache)==attempt['cache_sha256']==embedding_usage['cache_sha256']
current_api_requests=0

query_rows=list(csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline=''))); query_map={r['query_id']:r['query_text'] for r in query_rows}
old=load_jsonl(OLD_CHUNKS); new=load_jsonl(NEW_CHUNKS); old_by_id={r['id']:r for r in old}; new_by_id={r['chunk_id']:r for r in new}
rankings=load_jsonl(OUT/'followup2_ranking_freeze.jsonl'); bundles=load_jsonl(OUT/'followup2_bundles.jsonl'); ranked=load_jsonl(OUT/'followup2_scored_rankings.jsonl')
pair_rows=list(csv.DictReader((OUT/'followup2_pair_scores.csv').open(encoding='utf-8',newline='')))
assert len(query_rows)==20 and len(old)==327 and len(new)==147 and len(rankings)==40 and len(ranked)==20 and len(pair_rows)==553
assert all(math.isfinite(float(r['raw_logit'])) for r in pair_rows)
scoring_freeze=json.loads((OUT/'followup2_scoring_freeze.json').read_text()); model_resources=json.loads((OUT/'followup2_model_resources.json').read_text())
assert scoring_freeze['pair_scores_sha256']==sha(OUT/'followup2_pair_scores.csv') and scoring_freeze['rankings_sha256']==sha(OUT/'followup2_scored_rankings.jsonl')
immutable_before=scoring_freeze['source_hashes_before']; immutable_paths=[ROOT/path for path in immutable_before]; assert all(sha(path)==immutable_before[str(path.relative_to(ROOT))] for path in immutable_paths)
bge_before_f2=scoring_freeze['bge_before']; assert tree_state(BGE_ROOT)==bge_before_f2
protected_before=json.loads((OUT/'followup2_integrity.json').read_text())['existing_outputs_after']; assert all(sha(OUT/path)==digest for path,digest in protected_before.items())
preflight_names=['followup2_contract.json','followup2_queries.csv','followup2_semantic_review.json','followup2_atomic_claim_audit.jsonl','followup2_positive_provenance.jsonl','followup2_combination_evidence.csv','followup2_gold_labels.csv','followup2_query_freeze.json','followup2_gold_freeze.json','followup2_embedding_plan.json','followup2_source_manifest.json','followup2_preflight_run_manifest.json']
preflight_before={name:sha(OUT/name) for name in preflight_names}; approval_manifest_before=sha(OUT/'followup2_approval_manifest.json')
oom_batch2=bool(model_resources['oom_batch2'])
import statistics
labels=list(csv.DictReader((OUT/'followup2_gold_labels.csv').open(encoding='utf-8',newline='')))
audits=load_jsonl(OUT/'followup2_atomic_claim_audit.jsonl')
ranked=load_jsonl(OUT/'followup2_scored_rankings.jsonl')
assert len(labels)==200 and Counter(r['label'] for r in labels)=={'negative':144,'positive':56} and len(audits)==33 and len(ranked)==20
claim_by={r['claim_id']:r for r in audits}; label_by={(r['query_id'],r['card_key']):r for r in labels}
positives=defaultdict(set)
for row in labels:
    if row['label']=='positive': positives[row['query_id']].add(row['card_key'])
assert all(2<=len(positives[q])<=5 for q in query_map)
def norm(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
def role_supported(role,evidence):
    raw=norm(role['text']); stripped=norm(re.sub(r'^#{1,6}\s*','',role['text']))
    hay=norm(evidence)
    return raw in hay or (stripped and stripped in hay)
def claim_supported(claim_id,evidence):
    claim=claim_by[claim_id]
    return all(role_supported(role,evidence) for role in claim['roles'])
def label_supported(query_id,card_key,evidence):
    row=label_by[(query_id,card_key)]
    if row['label']!='positive': return False
    return all(claim_supported(cid,evidence) for cid in json.loads(row['claim_ids_json']))
bundles_by={(r['query_id'],r['card_key']):r for r in bundles}
old_ranking_by={(r['query_id']):r for r in rankings if r['corpus']=='old'}
new_ranking_by={(r['query_id']):r for r in rankings if r['corpus']=='structural'}
ranked_by={r['query_id']:r for r in ranked}
per_query=[]; ceiling=[]; support_audit=[]
for qrow in query_rows:
    q=qrow['query_id']; cohort=qrow['cohort']; pos=positives[q]; rr=ranked_by[q]
    old_card_order=[]; old_rep={}
    for cid in rr['old_chunk_ids']:
        card=old_by_id[cid]['metadata']['card_key']
        if card not in old_rep: old_rep[card]=cid; old_card_order.append(card)
    new_card_order=rr['structural_card_keys']
    assert len(old_card_order)==len(set(old_card_order)) and len(new_card_order)==len(set(new_card_order))
    old_top20=old_ranking_by[q]['eligible_top20']; new_top20=new_ranking_by[q]['eligible_top20']
    old_candidate_cards={old_by_id[c]['metadata']['card_key'] for c in old_top20}; new_candidate_cards={new_by_id[c]['metadata']['card_key'] for c in new_top20}
    assert len(rr['old_chunk_ids'])==20 and len(set(rr['old_chunk_ids']))==20 and set(rr['old_chunk_ids'])==set(old_top20)
    assert set(old_card_order)==old_candidate_cards and set(new_card_order)==new_candidate_cards and len(new_card_order)==len(new_candidate_cards)
    old_reachable={}
    for card in old_candidate_cards:
        evidence='\n'.join(old_by_id[c]['document'] for c in old_top20 if old_by_id[c]['metadata']['card_key']==card)
        old_reachable[card]=label_supported(q,card,evidence) if card in pos else False
    new_reachable={card:(label_supported(q,card,bundles_by[(q,card)]['bundle_text']) if card in pos else False) for card in new_candidate_cards}
    for system,cards in [('old_selective_bge_d20',old_card_order),('structural_bundle_all_bge_d20',new_card_order)]:
        top3=cards[:3]; top5=cards[:5]; rep_ids=[]; supported=[]
        for slot,card in enumerate(top5,1):
            if system=='old_selective_bge_d20':
                cid=old_rep[card]; evidence=old_by_id[cid]['document']; rep_ids.append(cid)
            else:
                row=bundles_by[(q,card)]; evidence=row['bundle_text']; rep_ids.append(row['bundle_sha256'])
            ok=label_supported(q,card,evidence) if card in pos else False; supported.append(ok)
            support_audit.append({'system':system,'query_id':q,'cohort':cohort,'rank':slot,'missing_slot':False,'card_key':card,'positive':card in pos,'evidence_supported':ok,'claim_ids_json':label_by[(q,card)]['claim_ids_json'] if card in pos else '[]','evidence_id':rep_ids[-1]})
        for slot in range(len(top5)+1,6): support_audit.append({'system':system,'query_id':q,'cohort':cohort,'rank':slot,'missing_slot':True,'card_key':'','positive':False,'evidence_supported':False,'claim_ids_json':'[]','evidence_id':''})
        supported.extend([False]*(5-len(supported)))
        hit3=sum(c in pos for c in top3); hit5=sum(c in pos for c in top5)
        candidate_cards=old_candidate_cards if system=='old_selective_bge_d20' else new_candidate_cards
        reachable=old_reachable if system=='old_selective_bge_d20' else new_reachable
        raw_top20=old_top20 if system=='old_selective_bge_d20' else new_top20
        per_query.append({'system':system,'query_id':q,'cohort':cohort,'output_card_count':len(top5),'missing_slots_at_5':5-len(top5),'card_precision_at_3':hit3/3,'card_recall_at_3':hit3/len(pos),'card_precision_at_5':hit5/5,'card_recall_at_5':hit5/len(pos),'evidence_supported_card_recall_at_5':sum(supported)/len(pos),'evidence_accuracy_at_3':sum(supported[:3])/3,'evidence_accuracy_at_5':sum(supported)/5,'candidate_card_recall_at_20':len(pos&candidate_cards)/len(pos),'bundle_reachable_evidence_recall_at_20':sum(bool(reachable.get(c,False)) for c in pos)/len(pos),'card_zero_hit_at_5':int(hit5==0),'supported_zero_hit_at_5':int(sum(supported)==0),'raw_top20_duplicate_count':20-len(candidate_cards),'raw_top20_unique_cards':len(candidate_cards),'output_card_keys_json':json.dumps(top5,ensure_ascii=False),'representative_evidence_ids_json':json.dumps(rep_ids,ensure_ascii=False),'supported_json':json.dumps(supported),'positive_count':len(pos)})
    ceiling.extend([{'system':'old_selective_bge_d20','query_id':q,'cohort':cohort,'candidate_depth':20,'candidate_card_count':len(old_candidate_cards),'candidate_card_recall_at_20':len(pos&old_candidate_cards)/len(pos),'bundle_reachable_evidence_recall_at_20':sum(bool(old_reachable.get(c,False)) for c in pos)/len(pos),'missing_positive_cards_json':json.dumps(sorted(pos-old_candidate_cards),ensure_ascii=False),'unreachable_positive_cards_json':json.dumps(sorted(c for c in pos if not old_reachable.get(c,False)),ensure_ascii=False)},{'system':'structural_bundle_all_bge_d20','query_id':q,'cohort':cohort,'candidate_depth':20,'candidate_card_count':len(new_candidate_cards),'candidate_card_recall_at_20':len(pos&new_candidate_cards)/len(pos),'bundle_reachable_evidence_recall_at_20':sum(bool(new_reachable.get(c,False)) for c in pos)/len(pos),'missing_positive_cards_json':json.dumps(sorted(pos-new_candidate_cards),ensure_ascii=False),'unreachable_positive_cards_json':json.dumps(sorted(c for c in pos if not new_reachable.get(c,False)),ensure_ascii=False)}])
assert len(per_query)==40 and len(ceiling)==40 and len(support_audit)==200
metrics=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','candidate_card_recall_at_20','bundle_reachable_evidence_recall_at_20','card_zero_hit_at_5','supported_zero_hit_at_5','raw_top20_duplicate_count','raw_top20_unique_cards']
summary=[]
for cohort in ('paraphrase','and_combination'):
    for system in ('old_selective_bge_d20','structural_bundle_all_bge_d20'):
        rows=[r for r in per_query if r['cohort']==cohort and r['system']==system]; assert len(rows)==10
        summary.append({'cohort':cohort,'system':system,'denominator':10,**{m:sum(float(r[m]) for r in rows)/10 for m in metrics}})
base={(r['query_id']):r for r in per_query if r['system']=='old_selective_bge_d20'}; cand={(r['query_id']):r for r in per_query if r['system']=='structural_bundle_all_bge_d20'}
paired=[]
delta_metrics=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','candidate_card_recall_at_20','bundle_reachable_evidence_recall_at_20','card_zero_hit_at_5']
for qrow in query_rows:
    q=qrow['query_id']; paired.append({'query_id':q,'cohort':qrow['cohort'],**{'delta_'+m:float(cand[q][m])-float(base[q][m]) for m in delta_metrics}})
wlt=[]
for cohort in ('paraphrase','and_combination'):
    rows=[r for r in paired if r['cohort']==cohort]
    for metric in delta_metrics:
        values=[r['delta_'+metric] for r in rows]; wins=sum(v>1e-12 for v in values); losses=sum(v< -1e-12 for v in values)
        wlt.append({'cohort':cohort,'metric':metric,'denominator':10,'wins':wins,'losses':losses,'ties':10-wins-losses,'mean_delta':sum(values)/10})
summary_by={(r['cohort'],r['system']):r for r in summary}
checks={}; catastrophic_by={}
for cohort in ('paraphrase','and_combination'):
    b=summary_by[(cohort,'old_selective_bge_d20')]; c=summary_by[(cohort,'structural_bundle_all_bge_d20')]
    checks[cohort]={
      'candidate_card_recall_at_20_nonreg_and_min_0_90':c['candidate_card_recall_at_20']>=b['candidate_card_recall_at_20']-1e-12 and c['candidate_card_recall_at_20']>=.90-1e-12,
      'bundle_reachable_evidence_recall_at_20_nonreg_and_min_0_90':c['bundle_reachable_evidence_recall_at_20']>=b['bundle_reachable_evidence_recall_at_20']-1e-12 and c['bundle_reachable_evidence_recall_at_20']>=.90-1e-12,
      'card_recall_at_3_nonreg':c['card_recall_at_3']>=b['card_recall_at_3']-1e-12,
      'card_recall_at_5_nonreg':c['card_recall_at_5']>=b['card_recall_at_5']-1e-12,
      'card_precision_at_5_nonreg':c['card_precision_at_5']>=b['card_precision_at_5']-1e-12,
      'evidence_accuracy_at_5_nonreg':c['evidence_accuracy_at_5']>=b['evidence_accuracy_at_5']-1e-12,
      'zero_hit_increase_zero':c['card_zero_hit_at_5']<=b['card_zero_hit_at_5']+1e-12}
    sw=next(r for r in wlt if r['cohort']==cohort and r['metric']=='evidence_supported_card_recall_at_5')
    if cohort=='and_combination': checks[cohort]['supported_recall_strict_and_wins_gt_losses']=c['evidence_supported_card_recall_at_5']>b['evidence_supported_card_recall_at_5']+1e-12 and sw['wins']>sw['losses']
    else: checks[cohort]['supported_recall_nonreg_and_wins_ge_losses']=c['evidence_supported_card_recall_at_5']>=b['evidence_supported_card_recall_at_5']-1e-12 and sw['wins']>=sw['losses']
    cohort_pairs=[r for r in paired if r['cohort']==cohort]
    catastrophic_by[cohort]=sum(float(base[r['query_id']]['card_recall_at_5'])>0 and (float(cand[r['query_id']]['card_recall_at_5'])==0 or r['delta_card_recall_at_5']<=-.5) for r in cohort_pairs)
    checks[cohort]['catastrophic_loss_count_zero']=catastrophic_by[cohort]==0
catastrophic_all=sum(catastrophic_by.values())
all_nonreg=all(all(v for k,v in rows.items() if 'strict' not in k) for rows in checks.values())
strict_signal=checks['and_combination']['supported_recall_strict_and_wins_gt_losses']
gate_pass=all(all(rows.values()) for rows in checks.values()) and catastrophic_all==0
if gate_pass: disposition='query_and_compositional_generalization_dev_signal'
elif all_nonreg and not strict_signal: disposition='robustness_observed_but_no_candidate_advantage'
else: disposition='generalization_not_supported'
decision={'gate_checks':checks,'catastrophic_loss_count_each_cohort':catastrophic_by,'catastrophic_loss_count_all_queries':catastrophic_all,'gate_pass':gate_pass,'disposition':disposition,'always':['dev_signal_only','not_eligible_for_promotion','not_causal_or_operational_evidence'],'baseline':'old_selective_bge_d20','candidate':'structural_bundle_all_bge_d20'}
def write_csv(path,rows):
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
write_csv(OUT/'followup2_per_query.csv',per_query); write_csv(OUT/'followup2_summary.csv',summary); write_csv(OUT/'followup2_paired_deltas.csv',paired); write_csv(OUT/'followup2_wlt.csv',wlt); write_csv(OUT/'followup2_candidate_ceiling.csv',ceiling); write_csv(OUT/'followup2_evidence_support_audit.csv',support_audit)
write_json(OUT/'followup2_summary.json',{'summary':summary,'decision':decision,'metric_notes':{'bundle_reachable_evidence_recall_at_20':'baseline: union of same-card old D20 chunks; candidate: frozen same-card 1-hop structural bundle','evidence_supported_card_recall_at_5':'old: first-ranked representative chunk; candidate: scored frozen bundle','zero_hit':'card_zero_hit_at_5'}})
write_json(OUT/'followup2_decision.json',decision)
resources={'embedding_creation':json.loads((OUT/'followup2_embedding_usage.json').read_text()),'evaluation_resume':{'current_api_requests':0,'current_network_requests':0,'gpu_used':False,'model_loaded':False,'stored_pair_scores_only':True},'search':json.loads((OUT/'followup2_search_resources.json').read_text()),'model':json.loads((OUT/'followup2_model_resources.json').read_text()),'api_network_policy':'one approved embedding request only; model/cache local-only','api_cost_usd':None,'new_document_embeddings':0,'chroma_queries':0,'package_installs':0}
write_json(OUT/'followup2_resources.json',resources)
immutable_after={str(p.relative_to(ROOT)):sha(p) for p in immutable_paths}; bge_after_f2=tree_state(BGE_ROOT); protected_after={path:sha(OUT/path) for path in protected_before}; preflight_after={name:sha(OUT/name) for name in preflight_names}
assert immutable_after==immutable_before and bge_after_f2==bge_before_f2 and protected_after==protected_before and preflight_after==preflight_before
assert sha(OUT/'followup2_approval_manifest.json')==approval_manifest_before and sha(query_cache)==json.loads((OUT/'followup2_embedding_usage.json').read_text())['cache_sha256']
output_names=['followup2_api_attempt_state.json','followup2_embedding_usage.json','followup2_query_classification.csv','followup2_ranking_freeze.jsonl','followup2_ranking_freeze.json','followup2_search_resources.json','followup2_bundles.jsonl','followup2_bundle_trace.csv','followup2_bundle_build_freeze.json','followup2_pair_scores.csv','followup2_scorer_input_audit.csv','followup2_scored_rankings.jsonl','followup2_model_resources.json','followup2_scoring_freeze.json','followup2_per_query.csv','followup2_summary.csv','followup2_summary.json','followup2_paired_deltas.csv','followup2_wlt.csv','followup2_candidate_ceiling.csv','followup2_evidence_support_audit.csv','followup2_decision.json','followup2_resources.json']
execution_integrity={'status':'PASS','approval_core_sha256':EXPECTED_APPROVAL_SHA,'approval_guard_passed_before_api_model_results':True,'query_count':20,'label_rows':200,'positive':56,'negative':144,'ranking_rows':40,'bundle_rows':len(bundles),'pair_score_rows':len(pair_rows),'pair_scores_finite':True,'per_query_rows':40,'summary_rows':4,'paired_rows':20,'wlt_rows':len(wlt),'candidate_ceiling_rows':40,'support_audit_rows':200,'all_routes_semantic':True,'ranking_frozen_before_semantic_gold_read':True,'scoring_frozen_before_semantic_gold_read':True,'current_api_requests':current_api_requests,'historical_creation_api_requests':1,'api_item_count':20,'api_input_tokens':historical_tokens,'gpu_physical':0,'oom_batch2':oom_batch2,'truncated_pairs':resources['model']['truncated_pairs'],'bundle_truncated_pairs':resources['model']['bundle_truncated_pairs'],'immutable_inputs_before':immutable_before,'immutable_inputs_after':immutable_after,'bge_before':bge_before_f2,'bge_after':bge_after_f2,'protected_existing_outputs_before':protected_before,'protected_existing_outputs_after':protected_after,'preflight_outputs_before':preflight_before,'preflight_outputs_after':preflight_after,'output_hashes':{name:sha(OUT/name) for name in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','run_manifest_sha256':'FINALIZE_AFTER_SAVE'}
write_json(OUT/'followup2_execution_integrity.json',execution_integrity)
run_manifest={'phase':'followup2_external_execution_and_evaluation','approval_core_sha256':EXPECTED_APPROVAL_SHA,'fresh_kernel':True,'current_api_requests':current_api_requests,'historical_creation_api_requests':1,'gpu_physical':0,'inputs_before':immutable_before,'inputs_after':immutable_after,'bge_before':bge_before_f2,'bge_after':bge_after_f2,'protected_outputs_preserved':True,'outputs':{name:sha(OUT/name) for name in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','self_hash_policy':'followup2_run_manifest.json and followup2_execution_integrity.json excluded'}
write_json(OUT/'followup2_run_manifest.json',run_manifest)
execution_integrity['run_manifest_sha256']=sha(OUT/'followup2_run_manifest.json'); write_json(OUT/'followup2_execution_integrity.json',execution_integrity)
print(json.dumps({'followup2_evaluation':'PASS','summary':summary,'decision':decision,'resources':{'api_requests':current_api_requests,'api_tokens':historical_tokens,'pairs':len(pair_rows),'scoring_seconds':resources['model']['scoring_seconds'],'peak_allocated_gib':resources['model']['peak_allocated_gib']}},ensure_ascii=False,indent=2))

{
  "followup2_evaluation": "PASS",
  "summary": [
    {
      "cohort": "paraphrase",
      "system": "old_selective_bge_d20",
      "denominator": 10,
      "card_precision_at_3": 0.6666666666666667,
      "card_recall_at_3": 0.64,
      "card_precision_at_5": 0.5399999999999999,
      "card_recall_at_5": 0.8366666666666667,
      "evidence_supported_card_recall_at_5": 0.45,
      "evidence_accuracy_at_3": 0.3666666666666667,
      "evidence_accuracy_at_5": 0.27999999999999997,
      "candidate_card_recall_at_20": 0.9400000000000001,
      "bundle_reachable_evidence_recall_at_20": 0.66,
      "card_zero_hit_at_5": 0.0,
      "supported_zero_hit_at_5": 0.0,
      "raw_top20_duplicate_count": 13.9,
      "raw_top20_unique_cards": 6.1
    },
    {
      "cohort": "paraphrase",
      "system": "structural_bundle_all_bge_d20",
      "denominator": 10,
      "card_precision_at_3": 0.7666666666666667,
      "card_recall_at_3": 0.7433333333333334,
      "card_precision_at_5": 0.5599999999999

## Follow-up 3 — oracle atomic decomposition retrieval diagnostic

Frozen parent 질의를 아는 사후 oracle이며 운영 분해기가 아닙니다. Atomic union 후보가 combined-query 후보 누락을 복구할 수 있는지만 진단합니다.


In [1]:
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, os, resource, time
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' else None)
assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP3_GPU','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'; CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'; HIER=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'; BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def canon(v):return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(p,v):p.write_text(json.dumps(v,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def jl(p):return [json.loads(x) for x in Path(p).read_text(encoding='utf-8').splitlines() if x]
def tree(root):
 fs=sorted(p for p in root.rglob('*') if p.is_file());m={str(p.relative_to(root)):sha(p) for p in fs};return {'file_count':len(fs),'total_bytes':sum(p.stat().st_size for p in fs),'files':m,'digest':hashlib.sha256(canon(m).encode()).hexdigest()}
prior_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup3_'));prior_before={n:sha(OUT/n) for n in prior_names};bge_before=tree(BGE)
inputs=[CHUNKS,HIER,OUT/'ranking_freeze.jsonl',OUT/'ranking_freeze.json',OUT/'queries.csv',OUT/'followup2_queries.csv',OUT/'followup1_contract.json',OUT/'followup2_per_query.csv',OUT/'followup2_summary.json',OUT/'followup2_bundles.jsonl',OUT/'followup2_pair_scores.csv',OUT/'followup2_execution_integrity.json'];input_before={str(p.relative_to(ROOT)):sha(p) for p in inputs}
f1=json.loads((OUT/'followup1_contract.json').read_text())
contract={'version':1,'name':'Follow-up3 — oracle atomic decomposition retrieval diagnostic','declared_before_results':True,'scope':'F2 AND-combination 10 only','oracle':'frozen parent_query_ids; not an operational decomposition/router','adaptive_dev_diagnostic':True,'promotion_eligible':False,'search':{'source':'frozen original Q01-Q10 structural rankings','vector_weight':.4,'bm25_weight':.6,'bm25_k1':1.5,'bm25_b':.75,'rrf_k':60,'component_depth':50,'atomic_candidate_depth':20},'candidate_policy':{'primary':'atomic union','intersection':'ceiling diagnostic only','expected_union_query_card_pairs':82,'expected_intersection_query_card_pairs':52,'expected_positive_recall_each_query':1.0},'bundle':{'relation_order':f1['relation_order'],'seed_sort':['atomic_rrf_rank','parent_query_id','chunk_id'],'parent_order':'frozen parent_query_ids','max_unique_chunks':5,'document_token_cap':4096,'forbidden':['cross_card','root_fanout','recursive_descendants','whole_document','gold_driven_selection']},'scoring':{'query':'original combined CMB text','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','local_files_only':True,'trust_remote_code':False,'batch_size':2,'dtype':'float16','max_length':8192,'truncation':'only_second','ranking':'raw_logit desc then atomic seed tuple then card_key','rrf_score_mixing':False,'reuse':'same query_id + exact bundle_sha256 only'},'gate':{'union_candidate_recall_each_query':1.0,'union_missing_positive_cards':0,'cmb05_cmb08_catastrophic_resolved':True,'card_recall_at_5_gte_old':True,'card_precision_at_5_gte_better_baseline':True,'evidence_accuracy_at_5_gte_better_baseline':True,'supported_recall_at_5_gt_current_structural':True,'supported_wins_gt_losses_vs_current':True,'catastrophic_loss_count':0,'truncation_or_contract_violation_count':0},'execution':{'api':0,'network':0,'new_embedding':0,'chroma':0,'index_change':0,'package_install':0}}
write_json(OUT/'followup3_contract.json',contract)
chunks=jl(CHUNKS);hier=jl(HIER);cb={r['chunk_id']:r for r in chunks};nb={r['node_id']:r for r in hier};children=defaultdict(list)
for n in hier:
 if n['parent_id'] is not None:children[n['parent_id']].append(n)
for x in children.values():x.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
rf=json.loads((OUT/'ranking_freeze.json').read_text());assert rf['search_contract']=={'bm25_k1':1.5,'bm25_b':.75,'vector_weight':.4,'bm25_weight':.6,'rrf_k':60,'component_depth':50,'vector_distance':'numpy_squared_l2','tie':'chunk_id lexical'}
atomic={r['query_id']:r for r in jl(OUT/'ranking_freeze.jsonl') if r['corpus']=='structural'};assert len(atomic)==10 and all(len(r['eligible_top20'])==20 for r in atomic.values())
f2q=[r for r in csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8')) if r['cohort']=='and_combination'];assert len(f2q)==10
sets=[];bundles=[];traces=[];start=time.perf_counter();cpu=time.process_time()
for qr in f2q:
 q=qr['query_id'];parents=json.loads(qr['parent_query_ids_json']);seed_map=defaultdict(list);parent_cards=[]
 for p in parents:
  cards=[]
  for rank,cid in enumerate(atomic[p]['eligible_top20'],1):
   card=cb[cid]['metadata']['card_key']
   if card not in cards:cards.append(card)
   seed_map[card].append({'atomic_rrf_rank':rank,'parent_query_id':p,'chunk_id':cid})
  parent_cards.append(cards)
 union=sorted(set(parent_cards[0])|set(parent_cards[1]));inter=sorted(set(parent_cards[0])&set(parent_cards[1]));sets.append({'query_id':q,'parent_query_ids':parents,'parent_card_sets':parent_cards,'union_card_keys':union,'intersection_card_keys':inter,'union_count':len(union),'intersection_count':len(inter)})
 for card in union:
  seeds=sorted(seed_map[card],key=lambda x:(x['atomic_rrf_rank'],x['parent_query_id'],x['chunk_id']));first=seeds[0];sid=first['chunk_id'];seed=cb[sid];node=nb[seed['metadata']['node_id']];selected=[];rel=[];parent_context=None
  def add(cid,kind,source,atomic_parent=''):
   if len(selected)>=5 or cid in selected:return
   c=cb[cid];assert c['metadata']['card_key']==card;selected.append(cid);rel.append({'selection_order':len(selected),'chunk_id':cid,'relation':kind,'node_id':c['metadata']['node_id'],'parent_id':c['metadata']['parent_id'],'part_index':c['metadata']['part_index'],'part_count':c['metadata']['part_count'],'source_relation_node_id':source,'atomic_parent_query_id':atomic_parent})
  add(sid,'best_atomic_rrf_seed',node['node_id'],first['parent_query_id'])
  same=[cb[c] for c in node['search_chunk_ids'] if c!=sid];same.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
  for c in same:add(c['chunk_id'],'same_node_adjacent_part',node['node_id'])
  pn=nb.get(node['parent_id'])
  if pn is not None and pn['parent_id'] is not None:
   if not pn['heading_only']:
    for cid in pn['search_chunk_ids']:add(cid,'non_root_immediate_parent_direct_body',pn['node_id'])
   else:
    parent_context=pn['heading_text'];line=node['heading_line_number'] if node['heading_line_number'] is not None else 10**12;sib=[n for n in children[pn['node_id']] if not n['heading_only'] and n['search_chunk_ids']];sib.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-line),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
    for n in sib:
     for cid in n['search_chunk_ids']:add(cid,'heading_only_parent_same_parent_direct_body_child',pn['node_id'])
  for child in children[node['node_id']]:
   if not child['heading_only']:
    for cid in child['search_chunk_ids']:add(cid,'seed_node_direct_child',node['node_id'])
  for x in seeds[1:]:add(x['chunk_id'],'same_card_other_atomic_top20_seed',node['node_id'],x['parent_query_id'])
  sec=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
  if parent_context:sec.append('[상위 제목]\n'+parent_context)
  for i,cid in enumerate(selected,1):
   c=cb[cid];heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)';sec.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
  text='\n\n'.join(sec);row={'query_id':q,'combined_query_text':qr['query_text'],'parent_query_ids':parents,'card_key':card,'best_seed':first,'all_atomic_seeds':seeds,'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':text,'bundle_sha256':hashlib.sha256(text.encode()).hexdigest(),'bundle_characters':len(text),'relation_trace':rel};bundles.append(row)
  for x in rel:traces.append({'query_id':q,'card_key':card,'bundle_sha256':row['bundle_sha256'],**x})
assert sum(r['union_count'] for r in sets)==len(bundles)==82 and sum(r['intersection_count'] for r in sets)==52 and len({(r['query_id'],r['card_key']) for r in bundles})==82
(OUT/'followup3_candidate_sets.jsonl').write_text(''.join(canon(r)+'\n' for r in sets),encoding='utf-8');(OUT/'followup3_bundles.jsonl').write_text(''.join(canon(r)+'\n' for r in bundles),encoding='utf-8')
with (OUT/'followup3_bundle_trace.csv').open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=list(traces[0]));w.writeheader();w.writerows(traces)
input_after={str(p.relative_to(ROOT)):sha(p) for p in inputs};prior_after={n:sha(OUT/n) for n in prior_names};assert input_after==input_before and prior_after==prior_before and tree(BGE)==bge_before
freeze={'status':'PASS','created_before_gold_read':True,'gold_or_f2_metric_rows_read_before_freeze':0,'contract_sha256':sha(OUT/'followup3_contract.json'),'candidate_sets_sha256':sha(OUT/'followup3_candidate_sets.jsonl'),'bundles_sha256':sha(OUT/'followup3_bundles.jsonl'),'trace_sha256':sha(OUT/'followup3_bundle_trace.csv'),'query_count':10,'union_pairs':82,'intersection_pairs':52,'max_bundle_chunks':max(len(r['selected_chunk_ids']) for r in bundles),'input_before':input_before,'input_after':input_after,'prior_before':prior_before,'prior_after':prior_after,'bge_before':bge_before,'resources':{'wall_seconds':time.perf_counter()-start,'cpu_seconds':time.process_time()-cpu,'peak_rss_mib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,'api_network_gpu_model_scoring':0}}
write_json(OUT/'followup3_preflight_freeze.json',freeze);write_json(OUT/'followup3_preflight_integrity.json',{'status':'PASS','query_count':10,'union_pairs':82,'intersection_pairs':52,'candidate_unique':True,'bundle_max_chunks':freeze['max_bundle_chunks'],'no_gold_before_freeze':True,'source_unchanged':True,'prior_24_f1_f2_unchanged':True,'api_network_embedding_chroma_gpu_model':0,'output_hashes':{n:sha(OUT/n) for n in ['followup3_contract.json','followup3_candidate_sets.jsonl','followup3_bundles.jsonl','followup3_bundle_trace.csv','followup3_preflight_freeze.json']}})
print({'followup3_preflight':'PASS','queries':10,'union_pairs':82,'intersection_pairs':52,'bundle_max_chunks':freeze['max_bundle_chunks'],'gold_reads':0})

{'followup3_preflight': 'PASS', 'queries': 10, 'union_pairs': 82, 'intersection_pairs': 52, 'bundle_max_chunks': 5, 'gold_reads': 0}


In [1]:
from pathlib import Path
import csv, gc, hashlib, json, math, os, statistics, subprocess, time
import numpy as np
cwd=Path.cwd().resolve();ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent;OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation';BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP3_GPU')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP3_EXTERNAL','0')=='0'
assert os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def canon(v):return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def jl(p):return [json.loads(x) for x in Path(p).read_text(encoding='utf-8').splitlines() if x]
def write_json(p,v):p.write_text(json.dumps(v,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def tree(root):
 fs=sorted(p for p in root.rglob('*') if p.is_file());m={str(p.relative_to(root)):sha(p) for p in fs};return {'file_count':len(fs),'total_bytes':sum(p.stat().st_size for p in fs),'files':m,'digest':hashlib.sha256(canon(m).encode()).hexdigest()}
def snap():
 lines=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.splitlines();line=next(x for x in lines if x.split(',')[0].strip()=='0');uuid=line.split(',')[1].strip()
 raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout;ps=[]
 for x in raw.splitlines():
  z=[v.strip() for v in x.split(',',3)]
  if len(z)==4 and z[0]==uuid:ps.append({'pid':int(z[1]),'process_name':z[2],'used_memory_mib':float(z[3])})
 return {'gpu_line':line,'processes':ps,'captured_at_unix':time.time()}
contract=json.loads((OUT/'followup3_contract.json').read_text());freeze=json.loads((OUT/'followup3_preflight_freeze.json').read_text())
assert freeze['status']=='PASS' and freeze['contract_sha256']==sha(OUT/'followup3_contract.json') and freeze['candidate_sets_sha256']==sha(OUT/'followup3_candidate_sets.jsonl') and freeze['bundles_sha256']==sha(OUT/'followup3_bundles.jsonl')
assert all(sha(ROOT/p)==h for p,h in freeze['input_before'].items()) and all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and tree(BGE)==freeze['bge_before']
bundles=jl(OUT/'followup3_bundles.jsonl');assert len(bundles)==82
f2b={(r['query_id'],r['card_key']):r for r in jl(OUT/'followup2_bundles.jsonl')}
f2s={(r['query_id'],r['card_key']):r for r in csv.DictReader((OUT/'followup2_pair_scores.csv').open(encoding='utf-8')) if r['pair_type']=='structural_bundle'}
rows=[];todo=[]
for b in bundles:
 key=(b['query_id'],b['card_key']);reuse=key in f2b and f2b[key]['bundle_sha256']==b['bundle_sha256'] and key in f2s
 row={'query_id':b['query_id'],'card_key':b['card_key'],'bundle_sha256':b['bundle_sha256'],'best_atomic_rrf_rank':b['best_seed']['atomic_rrf_rank'],'best_parent_query_id':b['best_seed']['parent_query_id'],'best_seed_chunk_id':b['best_seed']['chunk_id'],'score_reused':reuse}
 if reuse:row['raw_logit']=float(f2s[key]['raw_logit'])
 else:todo.append((len(rows),b))
 rows.append(row)
assert len(rows)==82 and len(todo)<=82
gpu_before=snap()
import torch
from transformers import AutoModelForSequenceClassification,AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
tokenizer=AutoTokenizer.from_pretrained(BGE,local_files_only=True,trust_remote_code=False)
audit=[]
for b in bundles:
 q=b['combined_query_text'];d=b['bundle_text'];qn=len(tokenizer.encode(q,add_special_tokens=False));dn=len(tokenizer.encode(d,add_special_tokens=False));enc=tokenizer(q,d,truncation='only_second',max_length=8192);tr=max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192)
 assert dn<=4096 and tr==0
 audit.append({'query_id':b['query_id'],'card_key':b['card_key'],'bundle_sha256':b['bundle_sha256'],'query_sha256':hashlib.sha256(q.encode()).hexdigest(),'bundle_tokens':dn,'input_tokens':len(enc['input_ids']),'truncated_tokens':tr,'gold_fields_used':False,'scorer_allowlist':'combined query text + frozen bundle text'})
def score(batch):
 torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();t=time.perf_counter();model=AutoModelForSequenceClassification.from_pretrained(BGE,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0');load=time.perf_counter()-t
 warm=tokenizer(['준비'],['준비'],return_tensors='pt').to('cuda:0');t=time.perf_counter()
 with torch.inference_mode():assert torch.isfinite(model(**warm).logits).all()
 torch.cuda.synchronize();warm_s=time.perf_counter()-t;out=[];t=time.perf_counter()
 try:
  for i in range(0,len(todo),batch):
   x=todo[i:i+batch];z=tokenizer([b['combined_query_text'] for _,b in x],[b['bundle_text'] for _,b in x],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
   with torch.inference_mode():v=model(**z).logits.reshape(-1).float().cpu().numpy()
   assert len(v)==len(x) and np.isfinite(v).all();out.extend(map(float,v));del z,v
  torch.cuda.synchronize();sec=time.perf_counter()-t;meta={'load_seconds':load,'warmup_seconds':warm_s,'scoring_seconds':sec,'pairs_per_second':len(todo)/sec if todo else None,'batch_size':batch,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30};ok=True
 except torch.cuda.OutOfMemoryError:out=[];meta={};ok=False
 del model;gc.collect();torch.cuda.empty_cache();return ok,out,meta
ok,scores,meta=score(2);oom=not ok
if not ok:ok,scores,meta=score(1)
assert ok and len(scores)==len(todo)
for (idx,b),v in zip(todo,scores):rows[idx]['raw_logit']=v
assert all(math.isfinite(r['raw_logit']) for r in rows)
with (OUT/'followup3_pair_scores.csv').open('w',newline='',encoding='utf-8') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
with (OUT/'followup3_scorer_input_audit.csv').open('w',newline='',encoding='utf-8') as f:w=csv.DictWriter(f,fieldnames=list(audit[0]));w.writeheader();w.writerows(audit)
ranks=[]
for q in sorted({r['query_id'] for r in rows}):
 x=sorted([r for r in rows if r['query_id']==q],key=lambda r:(-r['raw_logit'],int(r['best_atomic_rrf_rank']),r['best_parent_query_id'],r['best_seed_chunk_id'],r['card_key']))
 ranks.append({'query_id':q,'card_keys':[r['card_key'] for r in x],'bundle_sha256s':[r['bundle_sha256'] for r in x],'raw_logits':[r['raw_logit'] for r in x]})
(OUT/'followup3_rankings.jsonl').write_text(''.join(canon(r)+'\n' for r in ranks),encoding='utf-8')
gpu_after=snap();tok=[r['bundle_tokens'] for r in audit];res={'physical_gpu':0,'shared_ollama_allowed':True,'model':'bge-reranker-v2-m3','revision':contract['scoring']['revision'],'local_files_only':True,'trust_remote_code':False,'dtype':'float16','max_length':8192,'document_token_cap':4096,'total_pairs':82,'reused_pairs':sum(r['score_reused'] for r in rows),'new_scored_pairs':len(todo),'oom_batch2':oom,'truncated_pairs':0,**meta,'bundle_tokens':{'p50':statistics.median(tok),'p95':float(np.percentile(tok,95)),'max':max(tok)},'gpu_before':gpu_before,'gpu_after':gpu_after,'api_network_embedding_chroma_package_install':0}
write_json(OUT/'followup3_resources.json',res)
sc={'status':'PASS','created_before_gold_read':True,'gold_reads_before_score_freeze':0,'pair_rows':82,'pair_scores_sha256':sha(OUT/'followup3_pair_scores.csv'),'audit_sha256':sha(OUT/'followup3_scorer_input_audit.csv'),'rankings_sha256':sha(OUT/'followup3_rankings.jsonl'),'bundles_sha256':sha(OUT/'followup3_bundles.jsonl'),'resources_sha256':sha(OUT/'followup3_resources.json'),'input_before':freeze['input_before'],'prior_before':freeze['prior_before'],'bge_before':freeze['bge_before']}
write_json(OUT/'followup3_scoring_freeze.json',sc)
assert all(sha(ROOT/p)==h for p,h in freeze['input_before'].items()) and all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and tree(BGE)==freeze['bge_before']
print({'followup3_scoring':'PASS','pairs':82,'reused':res['reused_pairs'],'new':len(todo),'oom':oom,'truncated':0})

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'followup3_scoring': 'PASS', 'pairs': 82, 'reused': 11, 'new': 71, 'oom': False, 'truncated': 0}


In [1]:
# Follow-up3 evaluation: self-contained, CPU/offline, score freeze before gold read.
import csv, hashlib, json, math, re, unicodedata
from collections import defaultdict
from pathlib import Path
import numpy as np

cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError('repository root or notebooks cwd required')
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'
canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
def jl(path): return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x.strip()]
def write_json(path,obj): Path(path).write_text(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_csv(path,rows):
    with Path(path).open('w',encoding='utf-8',newline='') as f:
        fields=list(dict.fromkeys(k for row in rows for k in row)); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def tree(path):
    files=sorted(x for x in Path(path).rglob('*') if x.is_file()); file_map={str(p.relative_to(path)):sha(p) for p in files}
    return {'file_count':len(files),'total_bytes':sum(p.stat().st_size for p in files),'files':file_map,'digest':hashlib.sha256(canon(file_map).encode()).hexdigest()}
assert __import__('os').environ.get('RUN_APPROVED_24_FOLLOWUP3_EXTERNAL','0')=='0' and __import__('os').environ.get('RUN_APPROVED_24_FOLLOWUP3_GPU','0')=='0'
freeze=json.loads((OUT/'followup3_preflight_freeze.json').read_text()); scoring=json.loads((OUT/'followup3_scoring_freeze.json').read_text())
assert freeze['status']=='PASS' and scoring['status']=='PASS' and scoring['created_before_gold_read'] and scoring['gold_reads_before_score_freeze']==0
assert scoring['pair_scores_sha256']==sha(OUT/'followup3_pair_scores.csv') and scoring['rankings_sha256']==sha(OUT/'followup3_rankings.jsonl') and scoring['bundles_sha256']==sha(OUT/'followup3_bundles.jsonl')
assert all(sha(ROOT/p)==h for p,h in freeze['input_before'].items()) and all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and tree(BGE)==freeze['bge_before']
f2i=json.loads((OUT/'followup2_execution_integrity.json').read_text()); assert sha(OUT/'followup2_per_query.csv')==f2i['output_hashes']['followup2_per_query.csv']
contract=json.loads((OUT/'followup3_contract.json').read_text()); candidates=jl(OUT/'followup3_candidate_sets.jsonl'); bundles=jl(OUT/'followup3_bundles.jsonl'); ranks=jl(OUT/'followup3_rankings.jsonl')
scores=list(csv.DictReader((OUT/'followup3_pair_scores.csv').open(encoding='utf-8',newline=''))); assert len(scores)==82 and all(math.isfinite(float(r['raw_logit'])) for r in scores)
assert len(candidates)==len(ranks)==10 and sum(r['union_count'] for r in candidates)==82 and sum(r['intersection_count'] for r in candidates)==52
assert all(len(r['card_keys'])==len(set(r['card_keys'])) for r in ranks)
bundle_by={(r['query_id'],r['card_key']):r for r in bundles}; candidate_by={r['query_id']:r for r in candidates}; rank_by={r['query_id']:r for r in ranks}
for r in ranks:
    expected=sorted([x for x in scores if x['query_id']==r['query_id']],key=lambda x:(-float(x['raw_logit']),int(x['best_atomic_rrf_rank']),x['best_parent_query_id'],x['best_seed_chunk_id'],x['card_key']))
    assert r['card_keys']==[x['card_key'] for x in expected] and r['bundle_sha256s']==[x['bundle_sha256'] for x in expected]

# Gold and prior metrics are read only after candidate, bundle, score and ranking freeze verification.
labels=list(csv.DictReader((OUT/'followup2_gold_labels.csv').open(encoding='utf-8',newline=''))); audits=jl(OUT/'followup2_atomic_claim_audit.jsonl')
labels=[r for r in labels if r['cohort']=='and_combination']; assert len(labels)==100
label_by={(r['query_id'],r['card_key']):r for r in labels}; claim_by={r['claim_id']:r for r in audits}; positives=defaultdict(set)
for r in labels:
    if r['label']=='positive': positives[r['query_id']].add(r['card_key'])
assert set(positives)==set(rank_by) and all(2<=len(v)<=4 for v in positives.values())
norm=lambda v:' '.join(unicodedata.normalize('NFKC',str(v)).lower().split())
def role_supported(role,evidence):
    raw=norm(role['text']); stripped=norm(re.sub(r'^#{1,6}\s*','',role['text'])); hay=norm(evidence)
    return raw in hay or bool(stripped and stripped in hay)
def label_supported(q,card,evidence):
    row=label_by[(q,card)]
    return row['label']=='positive' and all(all(role_supported(role,evidence) for role in claim_by[cid]['roles']) for cid in json.loads(row['claim_ids_json']))
f2=list(csv.DictReader((OUT/'followup2_per_query.csv').open(encoding='utf-8',newline=''))); f2=[r for r in f2 if r['cohort']=='and_combination' and r['system'] in {'old_selective_bge_d20','structural_bundle_all_bge_d20'}]; assert len(f2)==20
f2_by={(r['system'],r['query_id']):r for r in f2}; qids=sorted(rank_by); assert qids==[f'CMB{i:02d}' for i in range(1,11)]
metrics=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','candidate_card_recall_at_20','bundle_reachable_evidence_recall_at_20','card_zero_hit_at_5','supported_zero_hit_at_5']
per=[]; evidence_audit=[]
for source,system in [('old_selective_bge_d20','old_selective_bge_d20'),('structural_bundle_all_bge_d20','structural_combined_query_bundle_all_bge_d20')]:
    for q in qids:
        r=f2_by[(source,q)]; per.append({'system':system,'query_id':q,'cohort':'and_combination',**{m:float(r[m]) for m in metrics},'output_card_count':int(r['output_card_count']),'missing_slots_at_5':int(r['missing_slots_at_5']),'raw_top20_unique_cards':int(r['raw_top20_unique_cards']),'raw_top20_duplicate_count':int(r['raw_top20_duplicate_count']),'output_card_keys_json':r['output_card_keys_json'],'supported_json':r['supported_json'],'positive_count':int(r['positive_count']),'source_sha256':f2i['output_hashes']['followup2_per_query.csv']})
for q in qids:
    pos=positives[q]; cards=rank_by[q]['card_keys']; top3=cards[:3]; top5=cards[:5]; supported=[]
    for slot in range(1,6):
        if slot<=len(top5):
            card=top5[slot-1]; b=bundle_by[(q,card)]; ok=label_supported(q,card,b['bundle_text']) if card in pos else False
            evidence_audit.append({'query_id':q,'rank':slot,'missing_slot':False,'card_key':card,'positive':card in pos,'evidence_supported':ok,'bundle_sha256':b['bundle_sha256'],'claim_ids_json':label_by[(q,card)]['claim_ids_json'] if card in pos else '[]'}); supported.append(ok)
        else:
            evidence_audit.append({'query_id':q,'rank':slot,'missing_slot':True,'card_key':'','positive':False,'evidence_supported':False,'bundle_sha256':'','claim_ids_json':'[]'}); supported.append(False)
    candidate=set(candidate_by[q]['union_card_keys']); intersection=set(candidate_by[q]['intersection_card_keys']); missing=pos-candidate
    reachable=sum(label_supported(q,c,bundle_by[(q,c)]['bundle_text']) for c in pos if c in candidate)/len(pos)
    hit3=sum(c in pos for c in top3); hit5=sum(c in pos for c in top5)
    per.append({'system':'oracle_atomic_union_bundle_all_bge','query_id':q,'cohort':'and_combination','card_precision_at_3':hit3/3,'card_recall_at_3':hit3/len(pos),'card_precision_at_5':hit5/5,'card_recall_at_5':hit5/len(pos),'evidence_supported_card_recall_at_5':sum(supported)/len(pos),'evidence_accuracy_at_3':sum(supported[:3])/3,'evidence_accuracy_at_5':sum(supported)/5,'candidate_card_recall_at_20':len(pos&candidate)/len(pos),'bundle_reachable_evidence_recall_at_20':reachable,'card_zero_hit_at_5':int(hit5==0),'supported_zero_hit_at_5':int(sum(supported)==0),'output_card_count':len(top5),'missing_slots_at_5':5-len(top5),'raw_top20_unique_cards':len(candidate),'raw_top20_duplicate_count':40-len(candidate_by[q]['parent_card_sets'][0])-len(candidate_by[q]['parent_card_sets'][1]),'output_card_keys_json':json.dumps(top5,ensure_ascii=False),'supported_json':json.dumps(supported),'positive_count':len(pos),'source_sha256':scoring['rankings_sha256'],'intersection_card_recall':len(pos&intersection)/len(pos),'candidate_inflation_vs_current':len(candidate)-int(f2_by[('structural_bundle_all_bge_d20',q)]['raw_top20_unique_cards']),'missing_positive_cards_json':json.dumps(sorted(missing),ensure_ascii=False)})
assert len(per)==30 and len(evidence_audit)==50
new_by={r['query_id']:r for r in per if r['system']=='oracle_atomic_union_bundle_all_bge'}; assert all(abs(r['candidate_card_recall_at_20']-1)<1e-12 and json.loads(r['missing_positive_cards_json'])==[] for r in new_by.values())
summary=[]
for system in ['old_selective_bge_d20','structural_combined_query_bundle_all_bge_d20','oracle_atomic_union_bundle_all_bge']:
    rows=[r for r in per if r['system']==system]; assert len(rows)==10
    summary.append({'system':system,'cohort':'and_combination','denominator':10,**{m:sum(float(r[m]) for r in rows)/10 for m in metrics},'mean_output_card_count':sum(int(r['output_card_count']) for r in rows)/10,'mean_unique_candidate_cards':sum(int(r['raw_top20_unique_cards']) for r in rows)/10,**({'mean_intersection_card_recall':sum(float(r['intersection_card_recall']) for r in rows)/10,'mean_candidate_inflation_vs_current':sum(float(r['candidate_inflation_vs_current']) for r in rows)/10} if system=='oracle_atomic_union_bundle_all_bge' else {})})
summary_by={r['system']:r for r in summary}; delta_metrics=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','card_zero_hit_at_5','supported_zero_hit_at_5']
paired=[]; wlt=[]
for ref in ['old_selective_bge_d20','structural_combined_query_bundle_all_bge_d20']:
    ref_by={r['query_id']:r for r in per if r['system']==ref}
    for q in qids: paired.append({'reference':ref,'candidate':'oracle_atomic_union_bundle_all_bge','query_id':q,**{'delta_'+m:float(new_by[q][m])-float(ref_by[q][m]) for m in delta_metrics}})
    for m in delta_metrics:
        vals=[r['delta_'+m] for r in paired if r['reference']==ref]; wins=sum(v>1e-12 for v in vals); losses=sum(v< -1e-12 for v in vals)
        wlt.append({'reference':ref,'candidate':'oracle_atomic_union_bundle_all_bge','metric':m,'denominator':10,'wins':wins,'losses':losses,'ties':10-wins-losses,'mean_delta':sum(vals)/10})
old=summary_by['old_selective_bge_d20']; current=summary_by['structural_combined_query_bundle_all_bge_d20']; new=summary_by['oracle_atomic_union_bundle_all_bge']
current_wlt=next(r for r in wlt if r['reference']=='structural_combined_query_bundle_all_bge_d20' and r['metric']=='evidence_supported_card_recall_at_5')
old_by={r['query_id']:r for r in per if r['system']=='old_selective_bge_d20'}
catastrophic={q:(old_by[q]['card_recall_at_5']>0 and (new_by[q]['card_recall_at_5']==0 or new_by[q]['card_recall_at_5']-old_by[q]['card_recall_at_5']<=-.5)) for q in qids}
checks={'union_candidate_recall_each_query_1':all(abs(new_by[q]['candidate_card_recall_at_20']-1)<1e-12 for q in qids),'union_missing_positive_cards_0':all(json.loads(new_by[q]['missing_positive_cards_json'])==[] for q in qids),'cmb05_cmb08_catastrophic_resolved':not catastrophic['CMB05'] and not catastrophic['CMB08'],'card_recall_at_5_gte_old':new['card_recall_at_5']>=old['card_recall_at_5']-1e-12,'card_precision_at_5_gte_better_baseline':new['card_precision_at_5']>=max(old['card_precision_at_5'],current['card_precision_at_5'])-1e-12,'evidence_accuracy_at_5_gte_better_baseline':new['evidence_accuracy_at_5']>=max(old['evidence_accuracy_at_5'],current['evidence_accuracy_at_5'])-1e-12,'supported_recall_at_5_gt_current_structural':new['evidence_supported_card_recall_at_5']>current['evidence_supported_card_recall_at_5']+1e-12,'supported_wins_gt_losses_vs_current':current_wlt['wins']>current_wlt['losses'],'catastrophic_loss_count_0':sum(catastrophic.values())==0,'truncation_or_contract_violation_count_0':json.loads((OUT/'followup3_resources.json').read_text())['truncated_pairs']==0}
gate_pass=all(checks.values()); decision={'gate_checks':checks,'gate_pass':gate_pass,'catastrophic_loss_count':sum(catastrophic.values()),'catastrophic_by_query':catastrophic,'disposition':'oracle_decomposition_diagnostic_success' if gate_pass else 'oracle_decomposition_diagnostic_not_supported','always':['development_only','adaptive_oracle_diagnostic','not_an_operational_decomposer','not_eligible_for_promotion'],'comparison_baselines':['old_selective_bge_d20','structural_combined_query_bundle_all_bge_d20']}
write_csv(OUT/'followup3_per_query.csv',per); write_csv(OUT/'followup3_summary.csv',summary); write_csv(OUT/'followup3_paired_deltas.csv',paired); write_csv(OUT/'followup3_wlt.csv',wlt); write_csv(OUT/'followup3_evidence_audit.csv',evidence_audit)
write_json(OUT/'followup3_summary.json',{'summary':summary,'decision':decision,'metric_note':'fixed K uses missing slots as failures; evidence support requires all frozen atomic claim roles in one automatic bundle'})
write_json(OUT/'followup3_decision.json',decision)
(OUT/'followup3_README.md').write_text('# Follow-up3 — oracle atomic decomposition retrieval diagnostic\n\n정답 유형의 두 원자 질의를 미리 아는 개발셋 진단입니다. 운영 분해기가 아니며 승격 근거로 사용할 수 없습니다. 기존 복합 질의 검색 대신 두 parent의 구조 청크 Top20 카드 합집합을 만들고, 고정된 1-hop 번들 규칙과 로컬 BGE로 재정렬했습니다. Precision은 상위 슬롯 중 정답 비율, Recall은 전체 정답 중 찾은 비율, Evidence Accuracy는 상위 슬롯 중 동결된 두 원자 근거를 모두 담은 비율입니다.\n',encoding='utf-8')
assert all(0<=float(r[m])<=1 for r in per for m in metrics[:11]) and all(r['wins']+r['losses']+r['ties']==10 for r in wlt)
assert all(sha(ROOT/p)==h for p,h in freeze['input_before'].items()) and all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and tree(BGE)==freeze['bge_before']
output_names=['followup3_contract.json','followup3_candidate_sets.jsonl','followup3_bundles.jsonl','followup3_bundle_trace.csv','followup3_preflight_freeze.json','followup3_preflight_integrity.json','followup3_pair_scores.csv','followup3_scorer_input_audit.csv','followup3_rankings.jsonl','followup3_resources.json','followup3_scoring_freeze.json','followup3_per_query.csv','followup3_summary.csv','followup3_summary.json','followup3_paired_deltas.csv','followup3_wlt.csv','followup3_evidence_audit.csv','followup3_decision.json','followup3_README.md']
integrity={'status':'PASS','compile_checked_separately':True,'execution_errors':0,'preflight_queries':10,'union_pairs':82,'intersection_pairs':52,'pair_scores_rows':82,'pair_scores_finite':True,'per_query_rows':30,'summary_rows':3,'paired_rows':20,'wlt_rows':18,'evidence_audit_rows':50,'fixed_k_missing_slots_are_failures':True,'ranking_reconstructed_exact':True,'candidate_positive_recall_each_query_1':True,'gold_read_after_search_and_score_freeze':True,'api_network_new_embedding_chroma_index_package_install':0,'gpu_scoring_only':True,'truncated_pairs':0,'prior_outputs_preserved':True,'input_before':freeze['input_before'],'input_after':freeze['input_before'],'prior_before':freeze['prior_before'],'prior_after':freeze['prior_before'],'bge_before':freeze['bge_before'],'bge_after':freeze['bge_before'],'output_hashes':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','run_manifest_sha256':'FINALIZE_AFTER_SAVE'}
write_json(OUT/'followup3_integrity.json',integrity)
manifest={'phase':'followup3_oracle_atomic_decomposition','preflight_fresh_kernel':True,'gpu_scoring_fresh_kernel':True,'evaluation_fresh_kernel':True,'api_network_new_embedding_chroma':0,'gpu_physical':0,'inputs':freeze['input_before'],'prior_outputs':freeze['prior_before'],'outputs':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','self_hash_policy':'followup3_run_manifest.json and followup3_integrity.json excluded'}
write_json(OUT/'followup3_run_manifest.json',manifest); integrity['run_manifest_sha256']=sha(OUT/'followup3_run_manifest.json'); write_json(OUT/'followup3_integrity.json',integrity)
print(json.dumps({'followup3_evaluation':'PASS','summary':summary,'decision':decision},ensure_ascii=False,indent=2))


{
  "followup3_evaluation": "PASS",
  "summary": [
    {
      "system": "old_selective_bge_d20",
      "cohort": "and_combination",
      "denominator": 10,
      "card_precision_at_3": 0.6333333333333334,
      "card_recall_at_3": 0.8416666666666668,
      "card_precision_at_5": 0.42000000000000004,
      "card_recall_at_5": 0.925,
      "evidence_supported_card_recall_at_5": 0.38333333333333336,
      "evidence_accuracy_at_3": 0.3,
      "evidence_accuracy_at_5": 0.18,
      "candidate_card_recall_at_20": 0.95,
      "bundle_reachable_evidence_recall_at_20": 0.7666666666666667,
      "card_zero_hit_at_5": 0.0,
      "supported_zero_hit_at_5": 0.3,
      "mean_output_card_count": 4.9,
      "mean_unique_candidate_cards": 5.4
    },
    {
      "system": "structural_combined_query_bundle_all_bge_d20",
      "cohort": "and_combination",
      "denominator": 10,
      "card_precision_at_3": 0.5666666666666667,
      "card_recall_at_3": 0.7583333333333333,
      "card_precision_at_5": 0.

## Follow-up4 Phase A — query-text-only deterministic benefit parser

Codex coder agent가 작성한 CPU/offline preflight입니다. 기존 CMB01~10은 parser 동결 후 sanity에만 사용합니다. Blind 문장, query ID, parent ID, gold, card, ranking은 parser 입력에 들어가지 않으며 Phase B는 아직 만들지 않습니다. 출력은 서로 다른 canonical predicate 두 개 또는 고정 fail code입니다.

In [1]:
# Follow-up4 Phase A: parser freeze, then dev-only sanity. No blind evaluation.
import csv, hashlib, inspect, json, os
from pathlib import Path

cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError('repository root or notebooks cwd required')
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
SOURCE=OUT/'followup2_queries.csv'
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP4_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_GPU','0')=='0'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
canonical=lambda v:json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(path,obj): Path(path).write_text(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_csv(path,rows):
    with Path(path).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
prior_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup4_parser_'))
prior_before={n:sha(OUT/n) for n in prior_names}; source_before=sha(SOURCE)

PARSER_SOURCE=r'''import re
import unicodedata

FAIL_CODES = (
    'FAIL_TOO_FEW', 'FAIL_TOO_MANY', 'FAIL_CROSS_PREDICATE_OR',
    'FAIL_NEGATION', 'FAIL_OVERLAP_AMBIGUITY', 'FAIL_NO_AND_CONNECTOR',
)
SEPARATOR_RE = re.compile(r'[·ㆍ/|,&+]')
TRAILING_RE = re.compile(r'[?!.,;:]+$')
NEGATION_RE = re.compile(r'제외(?:하고)?|빼고|말고|아닌|않는|않은|\bnot\b')
ALLOWED_EXCEPTION_RE = re.compile(r'신규\s*발급.{0,30}?예외는?\s*제외하고')
OR_RE = re.compile(r'또는|혹은|\bor\b')
BRIDGE_AND_RE = re.compile(r'(?:과|와)\s|및|그리고|동시에|함께|뿐\s*아니라|받으며|있고|이며|에도')
GLOBAL_AND_RE = re.compile(r'모두|둘\s*다|양쪽')

def normalize_query(value):
    text = unicodedata.normalize('NFKC', str(value)).lower()
    text = SEPARATOR_RE.sub(' ', text)
    return ' '.join(TRAILING_RE.sub('', text.strip()).split())

def _blank(text, pattern):
    return re.sub(pattern, lambda m: ' ' * (m.end() - m.start()), text)

def _combined_span(*matches):
    return min(m.start() for m in matches), max(m.end() for m in matches)

def parse_benefit_pair(query_text):
    text = normalize_query(query_text)
    negation_text = _blank(text, ALLOWED_EXCEPTION_RE)
    if NEGATION_RE.search(negation_text):
        return 'FAIL_NEGATION'
    hits = []
    def add(qid, *matches):
        if all(matches):
            hits.append((qid, *_combined_span(*matches)))

    q01_scope = re.search(r'(?:국내\s*)?(?:일반\s*)?가맹점', text)
    q01_no_spend = re.search(r'전월\s*실적(?:\s*조건)?\s*없이|지난달\s*이용액\s*기준\s*없이|무실적|실적\s*(?:제한\s*없음|에\s*관계없이)', text)
    add('Q01', q01_scope, q01_no_spend)

    q02_text = _blank(text, re.compile(r'쿠팡\s*(?:이츠|로켓\s*와우)|네이버플러스\s*멤버십'))
    q02_merchant = re.search(r'쿠팡|g\s*마켓|지마켓', q02_text)
    q02_context = re.search(r'온라인\s*(?:쇼핑|상품|결제)|쇼핑|상품을?\s*(?:사|구매|결제)', q02_text)
    add('Q02', q02_merchant, q02_context)

    add('Q03', re.search(r'배달\s*앱|배민|요기요|쿠팡이츠', text))
    add('Q04', re.search(r'이동통신\s*요금|통신\s*요금|휴대(?:전화|폰)\s*요금(?:\s*정기결제)?|통신비', text))
    q05_text = _blank(text, re.compile(r'전기차\s*충전|전기차|lpg(?:\s*충전)?'))
    add('Q05', re.search(r'전기(?:료|\s*요금)|도시가스(?:비|\s*요금)?|가스\s*요금', q05_text))
    q06_direct = re.search(r'커피\s*전문점', text)
    q06_cafe = re.search(r'카페', text); q06_payment = re.search(r'결제', text)
    if q06_direct: add('Q06', q06_direct)
    elif q06_cafe and q06_payment: add('Q06', q06_cafe, q06_payment)
    add('Q07', re.search(r'쿠팡\s*로켓\s*와우|네이버플러스\s*멤버십|구독|멤버십|영상\s*(?:서비스|정기)|음악\s*(?:서비스|정기)|넷플릭스|유튜브\s*프리미엄|디즈니\s*플러스', text))
    add('Q08', re.search(r'일반\s*음식점|식당|외식|다이닝', text))
    add('Q09', re.search(r'대형\s*마트|슈퍼마켓|슈퍼|이마트|롯데마트|홈플러스|하나로마트', text))
    add('Q10', re.search(r'대중교통|버스|지하철|택시', text))

    hits.sort(key=lambda x: (x[1], x[2], x[0]))
    for i, left in enumerate(hits):
        for right in hits[i + 1:]:
            if max(left[1], right[1]) < min(left[2], right[2]):
                return 'FAIL_OVERLAP_AMBIGUITY'
    qids = sorted({h[0] for h in hits}, key=lambda q: int(q[1:]))
    if len(qids) < 2: return 'FAIL_TOO_FEW'
    if len(qids) > 2: return 'FAIL_TOO_MANY'
    first, second = sorted(hits, key=lambda x: (x[1], x[2], x[0]))
    bridge = text[first[2]:second[1]]
    bridge_and = bool(BRIDGE_AND_RE.search(bridge))
    if OR_RE.search(bridge) and not bridge_and:
        return 'FAIL_CROSS_PREDICATE_OR'
    if not bridge_and and not GLOBAL_AND_RE.search(text):
        return 'FAIL_NO_AND_CONNECTOR'
    return tuple(qids)
'''
parser_ns={}; exec(compile(PARSER_SOURCE,'<followup4_frozen_parser_v1>','exec'),parser_ns)
parse_benefit_pair=parser_ns['parse_benefit_pair']; normalize_query=parser_ns['normalize_query']; FAIL_CODES=parser_ns['FAIL_CODES']
assert list(inspect.signature(parse_benefit_pair).parameters)==['query_text']
parser_sha=hashlib.sha256(PARSER_SOURCE.encode()).hexdigest()
contract_core={'phase':'Follow-up4 Phase A','name':'query-text-only deterministic benefit parser','declared_before_blind_evaluation':True,'runtime_input':['query_text'],'runtime_forbidden':['query_id','parent_query_ids','gold','card','ranking'],'output':'two distinct numeric-sorted Qxx predicates or one fixed fail code','normalization':'NFKC/lower/separators-to-space/whitespace/trailing-punctuation','precedence':['쿠팡이츠→Q03 not Q02','쿠팡 로켓와우/네이버플러스 멤버십→Q07','이동통신/휴대폰 요금 정기결제→Q04 not Q07','전기차 충전/LPG excluded from Q05','long protected phrases before generic terms'],'minimum_conditions':{'Q01':'domestic/general merchant scope AND no-spend condition','Q02':'Coupang/Gmarket AND online-shopping context','Q03':'delivery app or one of three apps','Q04':'telecom/mobile bill','Q05':'electricity/city gas excluding EV/LPG','Q06':'coffee shop, or cafe with payment context','Q07':'subscription/membership/service','Q08':'general restaurant/dining','Q09':'hypermarket/supermarket/named mart','Q10':'public transit/bus/subway/taxi'},'connector':'cross-predicate AND required; cross-predicate OR-only fails','fail_codes':list(FAIL_CODES),'parser_source_sha256':parser_sha,'blind_queries_received':0,'phase_b_present':False,'execution':{'api':0,'network':0,'gpu':0,'model':0,'embedding':0,'chroma':0,'package_install':0}}
contract={'contract_core':contract_core,'contract_core_sha256':hashlib.sha256(canonical(contract_core).encode()).hexdigest()}
write_json(OUT/'followup4_parser_source.json',{'source':PARSER_SOURCE,'source_sha256':parser_sha})
write_json(OUT/'followup4_parser_contract.json',contract)
freeze={'status':'FROZEN_BEFORE_DEV_SANITY','parser_source_sha256':sha(OUT/'followup4_parser_source.json'),'contract_sha256':sha(OUT/'followup4_parser_contract.json'),'runtime_signature':'parse_benefit_pair(query_text)','blind_query_count':0,'source_input_sha256':source_before}
write_json(OUT/'followup4_parser_freeze.json',freeze)

# Dev sanity begins only after the parser/contract freeze above. Parent IDs are expected values only.
cmb=[r for r in csv.DictReader(SOURCE.open(encoding='utf-8',newline='')) if r['cohort']=='and_combination']
assert len(cmb)==10
dev=[]
for row in cmb:
    expected=tuple(sorted(json.loads(row['parent_query_ids_json']),key=lambda q:int(q[1:])))
    actual=parse_benefit_pair(row['query_text'])
    dev.append({'query_id':row['query_id'],'query_text_sha256':hashlib.sha256(row['query_text'].encode()).hexdigest(),'expected_json':json.dumps(expected),'actual_json':json.dumps(actual) if isinstance(actual,tuple) else actual,'pass':actual==expected})
assert len(dev)==10 and all(r['pass'] for r in dev)
edge_cases=[
 ('protected_coupang_eats','쿠팡이츠와 커피전문점 결제에 모두 혜택',('Q03','Q06')),
 ('protected_rocketwow','쿠팡 로켓와우와 일반 음식점 결제에 모두 혜택',('Q07','Q08')),
 ('protected_mobile_bill','휴대폰 요금 정기결제와 대형마트 결제에 모두 혜택',('Q04','Q09')),
 ('exclude_ev','전기차 충전과 커피전문점 결제에 모두 혜택','FAIL_TOO_FEW'),
 ('cross_or','커피전문점 또는 일반 음식점 혜택을 모두 알려줘','FAIL_CROSS_PREDICATE_OR'),
 ('negation','커피전문점은 제외하고 택시 혜택을 알려줘','FAIL_NEGATION'),
 ('one_predicate','커피전문점 혜택을 알려줘','FAIL_TOO_FEW'),
 ('three_predicates','커피전문점과 일반 음식점과 택시에서 모두 혜택','FAIL_TOO_MANY'),
 ('no_and','커피전문점 일반 음식점 혜택','FAIL_NO_AND_CONNECTOR'),
]
edges=[]
for case,text,expected in edge_cases:
    actual=parse_benefit_pair(text); edges.append({'case':case,'query_text_sha256':hashlib.sha256(text.encode()).hexdigest(),'expected_json':json.dumps(expected) if isinstance(expected,tuple) else expected,'actual_json':json.dumps(actual) if isinstance(actual,tuple) else actual,'pass':actual==expected})
assert all(r['pass'] for r in edges)
write_csv(OUT/'followup4_parser_dev_sanity.csv',dev); write_csv(OUT/'followup4_parser_edge_sanity.csv',edges)
prior_after={n:sha(OUT/n) for n in prior_names}; assert prior_after==prior_before and sha(SOURCE)==source_before
preflight={'status':'PASS_PHASE_A_WAITING_FOR_BLIND_QUERIES','parser_frozen_before_dev_sanity':True,'dev_sanity_passed':10,'dev_sanity_total':10,'edge_sanity_passed':len(edges),'edge_sanity_total':len(edges),'blind_query_count':0,'phase_b_created':False,'runtime_signature_exact':True,'query_text_only_no_leak':True,'api_network_gpu_model_embedding_chroma_package_install':0,'source_before':source_before,'source_after':sha(SOURCE),'prior_before':prior_before,'prior_after':prior_after}
write_json(OUT/'followup4_parser_preflight.json',preflight)
outputs=['followup4_parser_source.json','followup4_parser_contract.json','followup4_parser_freeze.json','followup4_parser_dev_sanity.csv','followup4_parser_edge_sanity.csv','followup4_parser_preflight.json']
integrity={'status':'PASS','parser_source_sha256':parser_sha,'contract_core_sha256':contract['contract_core_sha256'],'source_freeze_exact':json.loads((OUT/'followup4_parser_source.json').read_text())['source_sha256']==parser_sha,'contract_frozen_before_sanity':True,'dev_sanity_10_of_10':True,'edge_sanity_all_pass':True,'signature_query_text_only':True,'blind_queries_read_or_created':0,'phase_b_cells_created':0,'existing_24_f1_f2_f3_outputs_preserved':prior_after==prior_before,'api_network_gpu_model_embedding_chroma_package_install':0,'output_hashes':{n:sha(OUT/n) for n in outputs},'notebook_sha256':'FINALIZE_AFTER_SAVE'}
write_json(OUT/'followup4_parser_integrity.json',integrity)
print(json.dumps({'followup4_phase_a':'PASS','parser_sha256':parser_sha,'contract_core_sha256':contract['contract_core_sha256'],'dev_sanity':'10/10','edge_sanity':f'{len(edges)}/{len(edges)}','blind_queries':0,'phase_b_created':False},ensure_ascii=False,indent=2))


{
  "followup4_phase_a": "PASS",
  "parser_sha256": "6df586df89c882b501e0db0876daa7bbb9add8536b972dc0fbc389a3f9ef8ba5",
  "contract_core_sha256": "3a0f8e6405e12a27284dfe005fc87ec670f90e66be3e2ba21374d3bdef8253ea",
  "dev_sanity": "10/10",
  "edge_sanity": "9/9",
  "blind_queries": 0,
  "phase_b_created": false
}


## Follow-up4 Phase B — frozen-parser blind-style evaluation

Phase A parser source와 contract hash는 수정하지 않습니다. 첫 code cell은 ordered query text만 저장·예측·동결하고, 다음 code cell에서만 expected pair를 공개합니다. Parser가 10/10 strong pass하지 않으면 검색·bundle·GPU를 시작하지 않습니다.

In [1]:
# Phase B step 1: query text only -> prediction freeze. Expected pairs are not present/read here.
import csv, hashlib, inspect, json, os
from pathlib import Path
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' else None)
assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_GPU','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
canon=lambda v:json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(p,v):Path(p).write_text(json.dumps(v,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_csv(p,rows):
    with Path(p).open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
EXPECTED_PARSER_SHA='6df586df89c882b501e0db0876daa7bbb9add8536b972dc0fbc389a3f9ef8ba5';EXPECTED_CONTRACT_SHA='3a0f8e6405e12a27284dfe005fc87ec670f90e66be3e2ba21374d3bdef8253ea'
prior_names=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith('followup4_parser_blind_'));prior_before={n:sha(OUT/n) for n in prior_names}
src=json.loads((OUT/'followup4_parser_source.json').read_text());contract=json.loads((OUT/'followup4_parser_contract.json').read_text())
assert src['source_sha256']==EXPECTED_PARSER_SHA==hashlib.sha256(src['source'].encode()).hexdigest() and contract['contract_core']['parser_source_sha256']==EXPECTED_PARSER_SHA and contract['contract_core_sha256']==EXPECTED_CONTRACT_SHA
ns={};exec(compile(src['source'],'<followup4_frozen_parser_v1>','exec'),ns);parse=ns['parse_benefit_pair'];assert list(inspect.signature(parse).parameters)==['query_text']
query_texts=[
 ('B01','평소 국내 어디서나 실적 조건 없이 기본 적립을 받고, 휴대폰 요금을 낼 때도 추가 혜택이 붙는 카드를 찾아줘'),
 ('B02','인터넷으로 쿠팡이나 지마켓에서 물건을 사고 휴대전화 요금도 내려고 해. 두 곳 모두 혜택을 주는 카드가 뭐야?'),
 ('B03','쿠팡·G마켓 상품 구매 혜택과 전기료·도시가스비 납부 할인을 한 카드로 함께 받고 싶어. 해당 카드를 알려줘'),
 ('B04','온라인 쇼핑은 쿠팡이나 G마켓을 쓰고 커피 매장도 자주 가. 두 소비 영역을 모두 챙겨 주는 카드를 찾아줘'),
 ('B05','쿠팡·지마켓에서 물건 살 때와 넷플릭스 같은 정기 구독료를 낼 때 각각 혜택을 받을 수 있는 카드가 있어?'),
 ('B06','쿠팡 또는 G마켓 쇼핑과 일반 식당 결제를 한 카드로 같이 할인받거나 적립하고 싶어. 맞는 카드를 알려줘'),
 ('B07','배민·요기요·쿠팡이츠로 주문할 때도, 커피 전문 매장에서 결제할 때도 혜택이 적용되는 카드를 찾아줘'),
 ('B08','휴대폰 요금과 대형마트나 슈퍼 장보기 비용을 모두 할인·적립해 주는 카드를 알려줘'),
 ('B09','카페 결제와 일반 음식점 결제를 둘 다 챙겨 주는 카드가 무엇인지 알려줘'),
 ('B10','영상이나 음악 정기구독료를 내면서 일반 식당 결제 혜택도 함께 받을 수 있는 카드를 찾아줘'),
]
queries=[{'order':i,'query_id':qid,'query_text':text,'query_text_sha256':hashlib.sha256(text.encode()).hexdigest()} for i,(qid,text) in enumerate(query_texts,1)]
assert len(queries)==10 and [r['query_id'] for r in queries]==[f'B{i:02d}' for i in range(1,11)] and len({r['query_text_sha256'] for r in queries})==10
write_csv(OUT/'followup4_parser_blind_queries.csv',queries)
qfreeze={'status':'FROZEN_BEFORE_PARSER_PREDICTIONS','ordered_query_count':10,'ordered_query_digest':hashlib.sha256(canon([(r['query_id'],r['query_text'],r['query_text_sha256']) for r in queries]).encode()).hexdigest(),'queries_csv_sha256':sha(OUT/'followup4_parser_blind_queries.csv'),'parser_source_sha256':EXPECTED_PARSER_SHA,'contract_core_sha256':EXPECTED_CONTRACT_SHA,'expected_pair_fields_present':False,'gold_reads':0,'prior_before':prior_before}
write_json(OUT/'followup4_parser_blind_query_freeze.json',qfreeze)
pred=[]
for r in queries:
    value=parse(r['query_text']);pred.append({'order':r['order'],'query_id':r['query_id'],'query_text_sha256':r['query_text_sha256'],'prediction_type':'pair' if isinstance(value,tuple) else 'fail_code','prediction_json':json.dumps(value,ensure_ascii=False) if isinstance(value,tuple) else value})
write_csv(OUT/'followup4_parser_blind_predictions.csv',pred)
pfreeze={'status':'FROZEN_BEFORE_EXPECTED_PAIR_READ','prediction_count':10,'predictions_csv_sha256':sha(OUT/'followup4_parser_blind_predictions.csv'),'queries_csv_sha256':qfreeze['queries_csv_sha256'],'query_freeze_sha256':sha(OUT/'followup4_parser_blind_query_freeze.json'),'parser_source_sha256':EXPECTED_PARSER_SHA,'contract_core_sha256':EXPECTED_CONTRACT_SHA,'expected_pair_fields_present':False,'gold_reads_before_prediction_freeze':0,'api_network_gpu_model_embedding_chroma':0}
write_json(OUT/'followup4_parser_blind_prediction_freeze.json',pfreeze)
assert {n:sha(OUT/n) for n in prior_names}==prior_before
print({'followup4_blind_prediction_freeze':'PASS','queries':10,'predictions':10,'expected_reads':0,'gpu_search':0})


{'followup4_blind_prediction_freeze': 'PASS', 'queries': 10, 'predictions': 10, 'expected_reads': 0, 'gpu_search': 0}


In [2]:
# Phase B step 2: expected pairs are revealed only after prediction-freeze verification.
import csv, hashlib, json, math, os
from collections import Counter
from pathlib import Path
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' else None)
assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_GPU','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
def write_json(p,v):Path(p).write_text(json.dumps(v,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_csv(p,rows):
    with Path(p).open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
qfreeze=json.loads((OUT/'followup4_parser_blind_query_freeze.json').read_text());pfreeze=json.loads((OUT/'followup4_parser_blind_prediction_freeze.json').read_text())
assert pfreeze['status']=='FROZEN_BEFORE_EXPECTED_PAIR_READ' and pfreeze['predictions_csv_sha256']==sha(OUT/'followup4_parser_blind_predictions.csv') and pfreeze['queries_csv_sha256']==sha(OUT/'followup4_parser_blind_queries.csv') and pfreeze['gold_reads_before_prediction_freeze']==0
assert pfreeze['parser_source_sha256']=='6df586df89c882b501e0db0876daa7bbb9add8536b972dc0fbc389a3f9ef8ba5' and pfreeze['contract_core_sha256']=='3a0f8e6405e12a27284dfe005fc87ec670f90e66be3e2ba21374d3bdef8253ea'
assert all(sha(OUT/n)==h for n,h in qfreeze['prior_before'].items())
queries=list(csv.DictReader((OUT/'followup4_parser_blind_queries.csv').open(encoding='utf-8',newline='')));pred=list(csv.DictReader((OUT/'followup4_parser_blind_predictions.csv').open(encoding='utf-8',newline='')));assert len(queries)==len(pred)==10
# Expected pairs become available here, after the frozen prediction artifact above.
expected={'B01':('Q01','Q04'),'B02':('Q02','Q04'),'B03':('Q02','Q05'),'B04':('Q02','Q06'),'B05':('Q02','Q07'),'B06':('Q02','Q08'),'B07':('Q03','Q06'),'B08':('Q04','Q09'),'B09':('Q06','Q08'),'B10':('Q07','Q08')}
historical=[r for r in csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination'];assert len(historical)==10
assert all(tuple(json.loads(historical[i]['parent_query_ids_json']))==expected[f'B{i+1:02d}'] for i in range(10))
rows=[]
for q,p in zip(queries,pred):
    assert q['query_id']==p['query_id'] and q['query_text_sha256']==p['query_text_sha256']
    actual=tuple(json.loads(p['prediction_json'])) if p['prediction_type']=='pair' else p['prediction_json'];exp=expected[q['query_id']];covered=isinstance(actual,tuple);exact=actual==exp
    rows.append({'query_id':q['query_id'],'query_text_sha256':q['query_text_sha256'],'expected_json':json.dumps(exp),'prediction_type':p['prediction_type'],'prediction_json':p['prediction_json'],'exact_match':exact,'covered':covered,'wrong_pair':covered and not exact,'fail_code':'' if covered else actual})
write_csv(OUT/'followup4_parser_blind_evaluation.csv',rows)
conf=[]
for i in range(1,11):
    qid=f'Q{i:02d}';tp=fp=fn=tn=0
    for r in rows:
        gold=qid in expected[r['query_id']];predicted=r['prediction_type']=='pair' and qid in json.loads(r['prediction_json'])
        tp+=gold and predicted;fp+=(not gold) and predicted;fn+=gold and not predicted;tn+=(not gold) and not predicted
    precision=tp/(tp+fp) if tp+fp else 0;recall=tp/(tp+fn) if tp+fn else 0;f1=2*precision*recall/(precision+recall) if precision+recall else 0
    conf.append({'predicate_id':qid,'tp':tp,'fp':fp,'fn':fn,'tn':tn,'precision':precision,'recall':recall,'f1':f1})
write_csv(OUT/'followup4_parser_blind_predicate_confusion.csv',conf)
exact_count=sum(r['exact_match'] for r in rows);coverage=sum(r['covered'] for r in rows);wrong=sum(r['wrong_pair'] for r in rows);fails=10-coverage;strong=exact_count==coverage==10 and wrong==fails==0
decision={'strong_pass':strong,'exact_count':exact_count,'total':10,'exact_accuracy':exact_count/10,'coverage_count':coverage,'coverage_rate':coverage/10,'wrong_pair_count':wrong,'fail_count':fails,'fail_code_counts':dict(Counter(r['fail_code'] for r in rows if r['fail_code'])),'predicate_macro_f1':sum(r['f1'] for r in conf)/10,'predicate_micro_f1':sum(r['tp'] for r in conf)/(sum(r['tp'] for r in conf)+.5*(sum(r['fp'] for r in conf)+sum(r['fn'] for r in conf))) if sum(r['tp'] for r in conf) else 0,'disposition':'READY_FOR_CONDITIONAL_SEARCH' if strong else 'deterministic_parser_generalization_failed','gpu_search_started':False,'parser_or_lexicon_modified':False,'always':['blind_style_development_only','not_eligible_for_promotion']}
write_json(OUT/'followup4_parser_blind_summary.json',{'decision':decision,'historical_expected_reference_sha256':sha(OUT/'followup2_queries.csv'),'prediction_freeze_sha256':sha(OUT/'followup4_parser_blind_prediction_freeze.json')})
write_json(OUT/'followup4_parser_blind_decision.json',decision)
assert all(sha(OUT/n)==h for n,h in qfreeze['prior_before'].items())
outputs=['followup4_parser_blind_queries.csv','followup4_parser_blind_query_freeze.json','followup4_parser_blind_predictions.csv','followup4_parser_blind_prediction_freeze.json','followup4_parser_blind_evaluation.csv','followup4_parser_blind_predicate_confusion.csv','followup4_parser_blind_summary.json','followup4_parser_blind_decision.json']
integrity={'status':'PASS_PARSER_EVALUATION','parser_source_sha256':pfreeze['parser_source_sha256'],'contract_core_sha256':pfreeze['contract_core_sha256'],'prediction_frozen_before_expected_read':True,'query_count':10,'prediction_count':10,'evaluation_rows':10,'confusion_rows':10,'strong_pass':strong,'gpu_search_started':False,'search_bundle_pair_scores_created':False,'prior_24_f1_f2_f3_f4a_preserved':True,'api_network_gpu_model_embedding_chroma_package_install':0,'output_hashes':{n:sha(OUT/n) for n in outputs},'notebook_sha256':'FINALIZE_AFTER_SAVE'}
write_json(OUT/'followup4_parser_blind_integrity.json',integrity)
print(json.dumps({'followup4_blind_parser_evaluation':'PASS','decision':decision},ensure_ascii=False,indent=2))


{
  "followup4_blind_parser_evaluation": "PASS",
  "decision": {
    "strong_pass": false,
    "exact_count": 4,
    "total": 10,
    "exact_accuracy": 0.4,
    "coverage_count": 4,
    "coverage_rate": 0.4,
    "wrong_pair_count": 0,
    "fail_count": 6,
    "fail_code_counts": {
      "FAIL_TOO_FEW": 5,
      "FAIL_NO_AND_CONNECTOR": 1
    },
    "predicate_macro_f1": 0.43714285714285717,
    "predicate_micro_f1": 0.5714285714285714,
    "disposition": "deterministic_parser_generalization_failed",
    "gpu_search_started": false,
    "parser_or_lexicon_modified": false,
    "always": [
      "blind_style_development_only",
      "not_eligible_for_promotion"
    ]
  }
}


### Follow-up4 reviewer remediation — terminal failure evidence

Frozen parser와 blind predictions/metrics/disposition은 변경하지 않습니다. 이 parser version은 strong gate 실패로 terminal이며 downstream search/bundle/BGE를 영구 금지합니다. Blind-style은 독립 snapshot/commit이 아닌 self-recorded 실행 순서와 hash에 의존하는 개발 진단입니다.

In [1]:
# Reviewer remediation: evidence/schema only. No parser/result/search/model changes.
import hashlib, json, os
from pathlib import Path
cwd=Path.cwd().resolve();ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' else None)
assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP4_GPU','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation';NB=ROOT/'notebooks/24_multicard_recommendation_retrieval_evaluation.ipynb'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
def read(n):return json.loads((OUT/n).read_text())
def write(n,v):(OUT/n).write_text(json.dumps(v,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
TEXT_SHA='6df586df89c882b501e0db0876daa7bbb9add8536b972dc0fbc389a3f9ef8ba5';ARTIFACT_SHA='f2df3adec8d508df825d51cf109516281d63c34dd647e799d124b75289c48444'
before={'followup4_parser_contract.json':'02253b376cbb9aeebf079fdf075f409f12eade8627ec5bbd21bed914771bb7d9','followup4_parser_freeze.json':'9591a9d02a0a69f405e2fd1d7125b63888ee73636fdfc28edd40936cd75604f9','followup4_parser_integrity.json':'eaf51947f62275093fb63a77c86f5c84b46f35dc402944f6a85a0a7d2cb7b6e2','followup4_parser_blind_query_freeze.json':'c17e2d3b0fe7758649f45d2a54f0a9af27a53a47ced598b6667f2959b9faf487','followup4_parser_blind_prediction_freeze.json':'13d71e235cb06660f8cff14b0e092279b37f696cce0e5d1ada9283c42f89aaf1','followup4_parser_blind_summary.json':'fcc7f3e6a27ad94f4307f69833d47a818b789eba01c38a1700e6f3e8599251ba','followup4_parser_blind_decision.json':'fd2a8521c4bb3a7b0ad52d4d1d90910e4c624f2df163414a34f261e47ff6a52e','followup4_parser_blind_integrity.json':'ae43940c7369834c585672cb0fc0421175fbc4f8a42abf039e72e1b1454a1d50'}
PARTIAL_CONTRACT_SHA='eb716d479b47fb4d130fdbf4ced53c7ab4f2744426843e2985dcbe9a1006c724'
review_v1={'followup4_parser_contract.json':'f7362c37baecedd8f258d05c925a6c16f782fa05754e80dcc7782b2880d25465','followup4_parser_freeze.json':'b6f8f530ab50ba136fc4f51221610de9b29b2ad53407df2acc412f91144169f6','followup4_parser_integrity.json':'0672e75f34e264463cd8fbce43ddb6deb63ba95fdd9af9f3523c2d0c1cb60579','followup4_parser_blind_query_freeze.json':'1873d0f0e4ac9a017cf124165515c051ef27abc635abd406c5c0e52726d681c7','followup4_parser_blind_prediction_freeze.json':'0c23c0c2a2edaa28a571f903d8e10a0fd616cc57a6bd86fd3b4955448fc0e849','followup4_parser_blind_summary.json':'34fb3a7453b457bf5392e4e3e93c3d93d8991c35a91ae33dbc600a99084aa87e','followup4_parser_blind_decision.json':'ed355342b237e138324c8892ffad7d1885135bd9a2e178a3da13850df4d072ac','followup4_parser_blind_integrity.json':'b2a475f6211d947355c5886e90887f8b7cc71a2b20eeb5d0f181ca3b96cadca2'}
assert all(sha(OUT/n) in {before[n],review_v1[n]} for n in before)
protected={'followup4_parser_source.json':ARTIFACT_SHA,'followup4_parser_blind_queries.csv':'cbd66d1a965be25ca3c70c07fbc211865d394260618397be4e96b5715acee52b','followup4_parser_blind_predictions.csv':'2864eb831b131215e81822f5e78eda8b7d71c08e47e2667123b8eda7a0423759','followup4_parser_blind_evaluation.csv':'612b553657634f8b064ee895f8efc4047af9e01ac03e36d8f3544259349be1e2','followup4_parser_blind_predicate_confusion.csv':'682e7078b3698df7d257ad0f95658c88e19d81c71dc0eaa565f7ff0ebc3d02bb'}
assert all(sha(OUT/n)==h for n,h in protected.items()) and read('followup4_parser_source.json')['source_sha256']==TEXT_SHA
hash_semantics={'parser_source_text_sha256':TEXT_SHA,'parser_source_artifact_sha256':ARTIFACT_SHA,'historical_aliases':{'followup4_parser_source.json.source_sha256':'parser_source_text_sha256','contract_core.parser_source_sha256':'parser_source_text_sha256','followup4_parser_freeze.json.parser_source_sha256':'parser_source_artifact_sha256'},'source_artifact_self_hash_excluded_from_source_artifact':True}
terminal={'terminal':True,'downstream_permanently_forbidden_for_this_parser_version':True,'allow_search_bundle_bge':False,'future_parser_change_policy':'new Follow-up/version required; no Phase C for this parser version','blind_limitation':'blind_style_development_only: self-recorded phase order and hashes, not an independent snapshot/commit'}
contract=read('followup4_parser_contract.json');assert contract['contract_core']['parser_source_sha256']==TEXT_SHA;contract['hash_semantics']=hash_semantics;contract['terminal_review_amendment']=terminal;contract.update(terminal);write('followup4_parser_contract.json',contract)
freeze=read('followup4_parser_freeze.json');old=freeze.pop('parser_source_sha256',None);assert old in {None,ARTIFACT_SHA} and (old is not None or (freeze['parser_source_text_sha256']==TEXT_SHA and freeze['parser_source_artifact_sha256']==ARTIFACT_SHA));freeze.update(hash_semantics);freeze['contract_sha256']=sha(OUT/'followup4_parser_contract.json');freeze.update(terminal);write('followup4_parser_freeze.json',freeze)
qfreeze=read('followup4_parser_blind_query_freeze.json');old=qfreeze.pop('parser_source_sha256',None);assert old in {None,TEXT_SHA} and (old is not None or (qfreeze['parser_source_text_sha256']==TEXT_SHA and qfreeze['parser_source_artifact_sha256']==ARTIFACT_SHA));qfreeze.update(hash_semantics);qfreeze['prior_before_semantics']='historical phase-order snapshot captured before predictions; review-amended evidence hashes are recorded separately';qfreeze.update(terminal);write('followup4_parser_blind_query_freeze.json',qfreeze)
pfreeze=read('followup4_parser_blind_prediction_freeze.json');old=pfreeze.pop('parser_source_sha256',None);assert old in {None,TEXT_SHA} and (old is not None or (pfreeze['parser_source_text_sha256']==TEXT_SHA and pfreeze['parser_source_artifact_sha256']==ARTIFACT_SHA));pfreeze.update(hash_semantics);pfreeze['query_freeze_sha256']=sha(OUT/'followup4_parser_blind_query_freeze.json');pfreeze.update(terminal);write('followup4_parser_blind_prediction_freeze.json',pfreeze)
decision=read('followup4_parser_blind_decision.json');numeric_before={k:decision[k] for k in ['exact_count','total','exact_accuracy','coverage_count','coverage_rate','wrong_pair_count','fail_count','predicate_macro_f1','predicate_micro_f1','disposition']}
assert decision['strong_pass'] is False and decision['disposition']=='deterministic_parser_generalization_failed' and decision['gpu_search_started'] is False
decision.update(terminal);decision['two_predicate_output_coverage_count']=decision['coverage_count'];decision['two_predicate_output_coverage_rate']=decision['coverage_rate'];decision['coverage_historical_alias']={'coverage_count':'two_predicate_output_coverage_count','coverage_rate':'two_predicate_output_coverage_rate'};decision['api_gpu_model_search_flags']=0;write('followup4_parser_blind_decision.json',decision)
summary=read('followup4_parser_blind_summary.json');summary['decision']=decision;summary['prediction_freeze_sha256']=sha(OUT/'followup4_parser_blind_prediction_freeze.json');summary['reportable_coverage_name']='two_predicate_output_coverage';summary['two_predicate_output_coverage_count']=decision['coverage_count'];summary['two_predicate_output_coverage_rate']=decision['coverage_rate'];summary['coverage_historical_alias']='coverage_count/coverage_rate';summary['hash_semantics']=hash_semantics;summary['blind_limitation']=terminal['blind_limitation'];write('followup4_parser_blind_summary.json',summary)
conditional=[p.name for p in OUT.iterdir() if p.is_file() and p.name.startswith('followup4_parser_blind_') and any(x in p.name for x in ['candidate','bundle','ranking','pair_score'])]
nb=json.loads(NB.read_text());downstream_cells=[c.get('id') for c in nb['cells'] if c.get('cell_type')=='code' and str(c.get('id','')).startswith('24-followup4') and any(x in str(c.get('id','')) for x in ['search','bundle','bge','ranking','pair-score'])]
assert conditional==[] and downstream_cells==[]
block={**terminal,**hash_semantics,'strong_pass':False,'disposition':'deterministic_parser_generalization_failed','conditional_search_bundle_ranking_pair_score_files':conditional,'conditional_file_count':0,'downstream_execution_cells':downstream_cells,'downstream_execution_cell_count':0,'flags':{'api':0,'gpu':0,'model':0,'search':0,'bundle':0,'ranking':0,'pair_score':0},'parser_source_artifact_unchanged':sha(OUT/'followup4_parser_source.json')==ARTIFACT_SHA,'blind_predictions_metrics_unchanged':all(sha(OUT/n)==h for n,h in protected.items())}
write('followup4_parser_downstream_block.json',block)
phase_a=read('followup4_parser_integrity.json');old=phase_a.pop('parser_source_sha256',None);assert old in {None,TEXT_SHA};phase_a.update(hash_semantics);phase_a.update(terminal);phase_a['historical_notebook_sha256']=phase_a.pop('notebook_sha256',phase_a.get('historical_notebook_sha256'));phase_a['output_hashes']={n:sha(OUT/n) for n in ['followup4_parser_source.json','followup4_parser_contract.json','followup4_parser_freeze.json','followup4_parser_dev_sanity.csv','followup4_parser_edge_sanity.csv','followup4_parser_preflight.json']};write('followup4_parser_integrity.json',phase_a)
blind=read('followup4_parser_blind_integrity.json');old=blind.pop('parser_source_sha256',None);assert old in {None,TEXT_SHA};blind.update(hash_semantics);blind.update(terminal);blind['two_predicate_output_coverage_count']=decision['coverage_count'];blind['two_predicate_output_coverage_rate']=decision['coverage_rate'];blind['coverage_historical_alias']='coverage_count/coverage_rate in decision';blind['conditional_file_count']=0;blind['downstream_execution_cell_count']=0;blind['flags_api_gpu_model_search']=0;blind['historical_notebook_sha256']=blind.pop('notebook_sha256',blind.get('historical_notebook_sha256'));blind['output_hashes']={n:sha(OUT/n) for n in ['followup4_parser_blind_queries.csv','followup4_parser_blind_query_freeze.json','followup4_parser_blind_predictions.csv','followup4_parser_blind_prediction_freeze.json','followup4_parser_blind_evaluation.csv','followup4_parser_blind_predicate_confusion.csv','followup4_parser_blind_summary.json','followup4_parser_blind_decision.json','followup4_parser_downstream_block.json']};write('followup4_parser_blind_integrity.json',blind)
numeric_after={k:decision[k] for k in numeric_before};assert numeric_after==numeric_before and all(sha(OUT/n)==h for n,h in protected.items())
allowed=set(before);historical_prior=qfreeze['prior_before'];assert all(sha(OUT/n)==h for n,h in historical_prior.items() if n not in allowed)
after={n:sha(OUT/n) for n in before};changes={'status':'PASS','reason':'reviewer terminal/hash-semantics/coverage evidence remediation only','failed_attempt':{'contract_sha256_after_partial_write':PARTIAL_CONTRACT_SHA,'failure_reason':'historical freeze parser_source_sha256 was the artifact hash, not the text hash','api_gpu_model_search_flags':0},'changed_evidence_before':before,'changed_evidence_after':after,'byte_exact_protected':protected,'numeric_decision_before':numeric_before,'numeric_decision_after':numeric_after,'parser_or_lexicon_changed':False,'blind_predictions_or_metrics_changed':False}
write('followup4_parser_review_hash_changes.json',changes)
terminal_integrity={'status':'PASS_TERMINAL_FAILURE','terminal':True,'downstream_permanently_forbidden_for_this_parser_version':True,'allow_search_bundle_bge':False,'parser_source_text_sha256':TEXT_SHA,'parser_source_artifact_sha256':ARTIFACT_SHA,'parser_source_artifact_unchanged':True,'blind_prediction_metric_hashes':protected,'decision_numeric_exact':True,'disposition_exact':True,'two_predicate_output_coverage_count':decision['coverage_count'],'two_predicate_output_coverage_rate':decision['coverage_rate'],'conditional_file_count':0,'downstream_execution_cell_count':0,'api_gpu_model_search_flags':0,'blind_limitation':terminal['blind_limitation'],'review_hash_changes_sha256':sha(OUT/'followup4_parser_review_hash_changes.json'),'downstream_block_sha256':sha(OUT/'followup4_parser_downstream_block.json'),'notebook_sha256':'FINALIZE_AFTER_SAVE'}
write('followup4_parser_terminal_integrity.json',terminal_integrity)
print(json.dumps({'followup4_terminal_review':'PASS','terminal':True,'allow_search_bundle_bge':False,'two_predicate_output_coverage':decision['coverage_rate'],'conditional_files':0,'downstream_cells':0,'flags':0},ensure_ascii=False,indent=2))


{
  "followup4_terminal_review": "PASS",
  "terminal": true,
  "allow_search_bundle_bge": false,
  "two_predicate_output_coverage": 0.4,
  "conditional_files": 0,
  "downstream_cells": 0,
  "flags": 0
}


## Follow-up5 — structural candidate depth D20/D30/D50 ablation

F2의 동결된 복합질의 검색 순위를 그대로 사용해 structural 후보 깊이만 20/30/50으로 바꿉니다. 질의 분해·라우팅·검색 재튜닝은 하지 않으며, 기존 old D20은 저장 결과를 그대로 쓰는 고정 기준선입니다. 기술 gate를 통과해도 같은 10개 카드 개발셋의 탐색 신호일 뿐 운영·holdout 승격 근거가 아닙니다.

In [1]:
# Follow-up5 CPU/offline preflight. Gold content is not loaded here.
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, os, resource, time
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None)
assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP5_EXTERNAL','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'; CHUNKS=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'; HIER=ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl'; BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest(); canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def jl(p): return [json.loads(x) for x in Path(p).read_text(encoding='utf-8').splitlines() if x]
def write_json(n,x): (OUT/n).write_text(json.dumps(x,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_jsonl(n,x): (OUT/n).write_text(''.join(canon(r)+'\n' for r in x),encoding='utf-8')
def write_csv(n,x):
    with (OUT/n).open('w',encoding='utf-8',newline='') as f: w=csv.DictWriter(f,fieldnames=list(x[0]));w.writeheader();w.writerows(x)
def tree(root):
    m={str(p.relative_to(root)):sha(p) for p in sorted(root.rglob('*')) if p.is_file()};return {'file_count':len(m),'total_bytes':sum((root/n).stat().st_size for n in m),'digest':hashlib.sha256(canon(m).encode()).hexdigest(),'files':m}
contract={'name':'Follow-up5 — structural candidate depth D20/D30/D50 ablation','declared_before_results':True,'scope':'F2 frozen AND-combination CMB01-CMB10; same 10-card development corpus','search':{'source':'followup2 frozen structural combined-query RRF top50','depths':[20,30,50],'vector_weight':.4,'bm25_weight':.6,'bm25_k1':1.5,'bm25_b':.75,'rrf_k':60,'component_depth':50,'rerun':False},'bundle':{'rule_binding':'Follow-up1 automatic same-card 1-hop','max_unique_chunks':5,'document_token_cap':4096,'root_seed_allowed':True,'root_children_fanout_forbidden':True,'recursive_descendants':False,'cross_card':False,'whole_document':False,'gold_driven_selection':False},'scoring':{'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','local_files_only':True,'trust_remote_code':False,'dtype':'float16','batch_size':2,'max_length':8192,'truncation':'only_second','rank':'raw_logit desc then best seed RRF rank then card_key','score_mixing':False,'reuse':'exact query text + bundle_sha256 from F2/F3'},'systems':['old_selective_bge_d20_fixed','structural_bundle_bge_d20','structural_bundle_bge_d30','structural_bundle_bge_d50'],'expected_candidate_pairs':{'20':68,'30':85,'50':98},'gate':{'candidate_card_recall_at_n_min':.95,'no_new_zero_ceiling':True,'card_recall_at_5_min':.925,'card_precision_at_5_min':.42,'card_recall_at_3_min':.8417,'evidence_supported_card_recall_at_5_min':.6333,'evidence_accuracy_at_5_min':.28,'catastrophic_loss_count':0,'contract_or_truncation_count':0,'selection':'shallowest passing depth; otherwise retain old'},'catastrophic':'old supported recall > 0 and candidate = 0, or candidate-old <= -0.5','always':['same_10_card_development_only','not_eligible_for_promotion','no_holdout_or_operational_claim'],'execution':{'api':0,'network':0,'new_embedding':0,'chroma':0,'index_change':0,'package_install':0}}
write_json('followup5_contract.json',contract)
prior_before={p.name:sha(p) for p in sorted(OUT.iterdir()) if p.is_file() and not p.name.startswith('followup5_')}
usage=json.loads((OUT/'followup2_embedding_usage.json').read_text()); query_cache=ROOT/usage['cache_path']
inputs=[CHUNKS,HIER,OUT/'followup1_contract.json',OUT/'followup2_queries.csv',OUT/'followup2_ranking_freeze.jsonl',OUT/'followup2_bundles.jsonl',OUT/'followup2_pair_scores.csv',OUT/'followup2_scored_rankings.jsonl',OUT/'followup2_per_query.csv',OUT/'followup2_summary.json',OUT/'followup3_bundles.jsonl',OUT/'followup3_pair_scores.csv',query_cache]
input_before={str(p.relative_to(ROOT)):sha(p) for p in inputs}; bge_before=tree(BGE); started=time.perf_counter();cpu0=time.process_time();rss0=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
queries=[r for r in csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination']; qtext={r['query_id']:r['query_text'] for r in queries};assert [r['query_id'] for r in queries]==[f'CMB{i:02d}' for i in range(1,11)]
ranks=[r for r in jl(OUT/'followup2_ranking_freeze.jsonl') if r['corpus']=='structural' and r['cohort']=='and_combination']; assert len(ranks)==10 and {r['query_id'] for r in ranks}==set(qtext)
chunks=jl(CHUNKS);hier=jl(HIER);by_id={r['chunk_id']:r for r in chunks};node_by={r['node_id']:r for r in hier};assert len(chunks)==147 and len(by_id)==147
children=defaultdict(list)
for n in hier:
    if n['parent_id'] is not None: children[n['parent_id']].append(n)
for x in children.values(): x.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
f1=json.loads((OUT/'followup1_contract.json').read_text());assert f1['max_unique_chunks_per_bundle']==5 and f1['bundle_document_token_cap']==4096 and 'root_fanout' in f1['forbidden']
f2_bundle={(r['query_id'],r['card_key']):r for r in jl(OUT/'followup2_bundles.jsonl') if r['cohort']=='and_combination'}; candidates=[];bundles=[];traces=[]
for r in sorted(ranks,key=lambda x:x['query_id']):
    assert len(r['rrf_top50'])==50 and r['eligible_top20']==r['rrf_top50'][:20] and len(set(r['rrf_top50']))==50
    for depth in contract['search']['depths']:
        raw=r['rrf_top50'][:depth];assert raw==r['rrf_top50'][:depth]
        by_card=defaultdict(list)
        for rank,cid in enumerate(raw,1): by_card[by_id[cid]['metadata']['card_key']].append((rank,cid))
        card_order=sorted(by_card,key=lambda c:(by_card[c][0][0],c))
        candidates.append({'query_id':r['query_id'],'depth':depth,'raw_chunk_count':len(raw),'raw_chunk_ids':raw,'unique_card_count':len(card_order),'duplicate_chunk_count':len(raw)-len(card_order),'card_keys':card_order,'card_chunk_counts':{c:len(by_card[c]) for c in card_order},'card_best_seed_ranks':{c:by_card[c][0][0] for c in card_order}})
        for card in card_order:
            seeds=by_card[card];best_rank,seed_id=seeds[0];seed=by_id[seed_id];sn=node_by[seed['metadata']['node_id']];selected=[];relation=[];parent_context=None
            def add(cid,kind,source_node):
                if len(selected)>=5 or cid in selected:return
                c=by_id[cid];assert c['metadata']['card_key']==card;selected.append(cid);relation.append({'selection_order':len(selected),'chunk_id':cid,'relation':kind,'node_id':c['metadata']['node_id'],'parent_id':c['metadata']['parent_id'],'part_index':c['metadata']['part_index'],'part_count':c['metadata']['part_count'],'source_relation_node_id':source_node})
            add(seed_id,'best_rrf_seed',sn['node_id'])
            same=[by_id[c] for c in sn['search_chunk_ids'] if c!=seed_id];same.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
            for c in same:add(c['chunk_id'],'same_node_adjacent_part',sn['node_id'])
            parent=node_by.get(sn['parent_id'])
            if parent is not None and parent['parent_id'] is not None:
                if not parent['heading_only']:
                    for cid in parent['search_chunk_ids']:add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
                else:
                    parent_context=parent['heading_text'];sl=sn['heading_line_number'] if sn['heading_line_number'] is not None else 10**12;siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']];siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-sl),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                    for n in siblings:
                        for cid in n['search_chunk_ids']:add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
            if sn['parent_id'] is not None:
                for child in children[sn['node_id']]:
                    if not child['heading_only']:
                        for cid in child['search_chunk_ids']:add(cid,'seed_node_direct_child',sn['node_id'])
            for _,cid in seeds[1:]:add(cid,f'same_card_other_top{depth}_seed',sn['node_id'])
            assert 1<=len(selected)<=5 and len(selected)==len(set(selected));sections=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
            if parent_context:sections.append('[상위 제목]\n'+parent_context)
            for i,cid in enumerate(selected,1):
                c=by_id[cid];heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)';sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
            text='\n\n'.join(sections);bh=hashlib.sha256(text.encode()).hexdigest();bundle={'query_id':r['query_id'],'depth':depth,'card_key':card,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'candidate_card_keys':card_order,'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':text,'bundle_sha256':bh,'bundle_characters':len(text),'relation_trace':relation,'root_seed':sn['parent_id'] is None,'root_children_fanout':False};bundles.append(bundle)
            for item in relation:traces.append({'query_id':r['query_id'],'depth':depth,'card_key':card,'best_seed_chunk_id':seed_id,'best_seed_rrf_rank':best_rank,'root_seed':sn['parent_id'] is None,'root_children_fanout':False,'optional_parent_heading':parent_context or '',**item})
assert {d:sum(b['depth']==d for b in bundles) for d in (20,30,50)}=={20:68,30:85,50:98} and len(candidates)==30
assert all(not b['root_children_fanout'] for b in bundles) and all(not (b['root_seed'] and any(x['relation']=='seed_node_direct_child' for x in b['relation_trace'])) for b in bundles)
for b in [x for x in bundles if x['depth']==20]: assert f2_bundle[(b['query_id'],b['card_key'])]['bundle_sha256']==b['bundle_sha256'] and f2_bundle[(b['query_id'],b['card_key'])]['selected_chunk_ids']==b['selected_chunk_ids']
unique={(b['query_id'],b['bundle_sha256']):b for b in bundles};assert all(len({x['card_key'] for x in bundles if x['query_id']==q and x['bundle_sha256']==h})==1 for q,h in unique)
write_jsonl('followup5_candidate_sets.jsonl',candidates);write_jsonl('followup5_bundles.jsonl',bundles);write_csv('followup5_bundle_trace.csv',traces)
cpu={'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu0,'peak_rss_mib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,'incremental_ru_maxrss_mib':max(0,resource.getrusage(resource.RUSAGE_SELF).ru_maxrss-rss0)/1024}
freeze={'status':'PASS_CPU_PREFLIGHT','declared_before_gold_semantic_read':True,'gold_semantic_reads':0,'query_count':10,'depths':[20,30,50],'candidate_pair_rows_by_depth':{str(d):sum(b['depth']==d for b in bundles) for d in (20,30,50)},'unique_query_bundle_pairs':len(unique),'candidate_rows':len(candidates),'bundle_rows':len(bundles),'trace_rows':len(traces),'d20_f2_bundle_exact_pairs':68,'ranking_prefix_exact':True,'root_seed_allowed':True,'root_children_fanout_count':0,'contract_sha256':sha(OUT/'followup5_contract.json'),'candidate_sets_sha256':sha(OUT/'followup5_candidate_sets.jsonl'),'bundles_sha256':sha(OUT/'followup5_bundles.jsonl'),'trace_sha256':sha(OUT/'followup5_bundle_trace.csv'),'input_before':input_before,'prior_before':prior_before,'bge_before':bge_before,'cpu':cpu,'execution':{'api':0,'network':0,'new_embedding':0,'chroma':0,'gpu':0,'model':0}}
write_json('followup5_preflight_freeze.json',freeze);assert {str(p.relative_to(ROOT)):sha(p) for p in inputs}==input_before and all(sha(OUT/n)==h for n,h in prior_before.items()) and tree(BGE)==bge_before
print(json.dumps({'followup5_preflight':'PASS','pairs_by_depth':freeze['candidate_pair_rows_by_depth'],'unique_query_bundle_pairs':len(unique),'d20_f2_bundle_exact':68,'gold_semantic_reads':0,'external_gpu_model':0},indent=2))


{
  "followup5_preflight": "PASS",
  "pairs_by_depth": {
    "20": 68,
    "30": 85,
    "50": 98
  },
  "unique_query_bundle_pairs": 176,
  "d20_f2_bundle_exact": 68,
  "gold_semantic_reads": 0,
  "external_gpu_model": 0
}


In [1]:
# Follow-up5 local BGE scoring; only exact query+bundle misses are scored.
from pathlib import Path
from collections import defaultdict
import csv, gc, hashlib, json, math, os, statistics, subprocess, time
import numpy as np
cwd=Path.cwd().resolve();ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None);assert ROOT is not None
assert os.environ.get('RUN_APPROVED_24_FOLLOWUP5_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP5_GPU')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0' and os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation';BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def jl(p):return [json.loads(x) for x in Path(p).read_text(encoding='utf-8').splitlines() if x]
def write_json(n,x):(OUT/n).write_text(json.dumps(x,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def tree(root):
    m={str(p.relative_to(root)):sha(p) for p in sorted(root.rglob('*')) if p.is_file()};return {'file_count':len(m),'total_bytes':sum((root/n).stat().st_size for n in m),'digest':hashlib.sha256(canon(m).encode()).hexdigest(),'files':m}
def gpu_snapshot():
    lines=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip().splitlines();line=next(x for x in lines if x.split(',')[0].strip()=='0');uuid=line.split(',')[1].strip();raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip();procs=[]
    for x in raw.splitlines():
        p=[v.strip() for v in x.split(',',3)]
        if len(p)==4 and p[0]==uuid:procs.append({'pid':int(p[1]),'process_name':p[2],'used_memory_mib':float(p[3])})
    return {'gpu_line':line,'processes':procs,'captured_at_unix':time.time()}
freeze=json.loads((OUT/'followup5_preflight_freeze.json').read_text());assert freeze['status']=='PASS_CPU_PREFLIGHT' and freeze['gold_semantic_reads']==0 and freeze['candidate_pair_rows_by_depth']=={'20':68,'30':85,'50':98}
assert all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and all(sha(ROOT/n)==h for n,h in freeze['input_before'].items()) and tree(BGE)==freeze['bge_before']
bundles=jl(OUT/'followup5_bundles.jsonl');queries={r['query_id']:r['query_text'] for r in csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination'};unique={}
for b in bundles:unique.setdefault((b['query_id'],b['bundle_sha256']),b)
assert len(unique)==freeze['unique_query_bundle_pairs'] and set(q for q,_ in unique)==set(queries)
critical_files=['config.json','model.safetensors','sentencepiece.bpe.model','special_tokens_map.json','tokenizer.json','tokenizer_config.json'];model_files={n:freeze['bge_before']['files'][n] for n in critical_files};model_files_fingerprint=hashlib.sha256(canon(model_files).encode()).hexdigest()
scorer_fingerprint={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','model_files_sha256':model_files,'model_files_fingerprint':model_files_fingerprint,'tokenizer_files':['sentencepiece.bpe.model','special_tokens_map.json','tokenizer.json','tokenizer_config.json'],'max_length':8192,'truncation':'only_second','dtype':'float16','local_files_only':True,'trust_remote_code':False};scorer_fingerprint_sha=hashlib.sha256(canon(scorer_fingerprint).encode()).hexdigest()
f2sf=json.loads((OUT/'followup2_scoring_freeze.json').read_text());f3sf=json.loads((OUT/'followup3_scoring_freeze.json').read_text());f2res=json.loads((OUT/'followup2_model_resources.json').read_text());f3res=json.loads((OUT/'followup3_resources.json').read_text());f3contract=json.loads((OUT/'followup3_contract.json').read_text())
assert f2sf['input_audit_sha256']==sha(OUT/'followup2_scorer_input_audit.csv') and f2sf['pair_scores_sha256']==sha(OUT/'followup2_pair_scores.csv') and f3sf['audit_sha256']==sha(OUT/'followup3_scorer_input_audit.csv') and f3sf['pair_scores_sha256']==sha(OUT/'followup3_pair_scores.csv')
for source_freeze in (f2sf,f3sf):assert {n:source_freeze['bge_before']['files'][n] for n in critical_files}==model_files
assert all([f2res['revision']==scorer_fingerprint['revision'],f2res['max_length']==8192,f2res['truncation']=='only_second',f2res['dtype']=='float16',f2res['local_files_only'] is True,f2res['trust_remote_code'] is False]) and all([f3res['revision']==scorer_fingerprint['revision'],f3contract['scoring']['max_length']==8192,f3contract['scoring']['truncation']=='only_second',f3contract['scoring']['dtype']=='float16',f3contract['scoring']['local_files_only'] is True,f3contract['scoring']['trust_remote_code'] is False])
source_cache={};source_name={};source_provenance=[]
def add_source(source,qid,card,bundle_sha,qsha,raw):
    assert qsha==hashlib.sha256(queries[qid].encode()).hexdigest();key=(qsha,bundle_sha,scorer_fingerprint_sha);raw=float(raw)
    if key in source_cache:assert source_cache[key]==raw
    source_cache[key]=raw
    if source=='followup2' or key not in source_name:source_name[key]=source
    source_provenance.append({'source':source,'query_id':qid,'card_key':card,'query_text_sha256':qsha,'bundle_sha256':bundle_sha,'scorer_fingerprint_sha256':scorer_fingerprint_sha,'raw_logit':raw})
f3audit={(r['query_id'],r['card_key']):r for r in csv.DictReader((OUT/'followup3_scorer_input_audit.csv').open(encoding='utf-8',newline=''))}
for r in csv.DictReader((OUT/'followup3_pair_scores.csv').open(encoding='utf-8',newline='')):
    a=f3audit[(r['query_id'],r['card_key'])];assert a['bundle_sha256']==r['bundle_sha256'];add_source('followup3',r['query_id'],r['card_key'],r['bundle_sha256'],a['query_sha256'],r['raw_logit'])
f2b={(r['query_id'],r['card_key']):r for r in jl(OUT/'followup2_bundles.jsonl')};f2audit={(r['query_id'],r['candidate_id']):r for r in csv.DictReader((OUT/'followup2_scorer_input_audit.csv').open(encoding='utf-8',newline='')) if r['pair_type']=='structural_bundle'}
for r in csv.DictReader((OUT/'followup2_pair_scores.csv').open(encoding='utf-8',newline='')):
    if r['pair_type']=='structural_bundle':
        b=f2b[(r['query_id'],r['card_key'])];a=f2audit[(r['query_id'],r['card_key'])];assert a['document_sha256']==b['bundle_sha256'];add_source('followup2',r['query_id'],r['card_key'],b['bundle_sha256'],a['query_sha256'],r['raw_logit'])
candidate_strong_key={(qid,bsha):(hashlib.sha256(queries[qid].encode()).hexdigest(),bsha,scorer_fingerprint_sha) for qid,bsha in unique};todo=[(k,b) for k,b in sorted(unique.items()) if candidate_strong_key[k] not in source_cache]
gpu_before=gpu_snapshot();import torch
from transformers import AutoModelForSequenceClassification,AutoTokenizer
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
tokenizer=AutoTokenizer.from_pretrained(BGE,local_files_only=True,trust_remote_code=False);token_rows={}
for k,b in sorted(unique.items()):
    q=queries[k[0]];qn=len(tokenizer.encode(q,add_special_tokens=False));dn=len(tokenizer.encode(b['bundle_text'],add_special_tokens=False));enc=tokenizer(q,b['bundle_text'],truncation='only_second',max_length=8192);tr=max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192);assert dn<=4096 and tr==0;token_rows[k]={'query_tokens':qn,'document_tokens':dn,'input_tokens':len(enc['input_ids']),'document_truncated_tokens':tr}
def score_new(batch):
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();t=time.perf_counter();model=AutoModelForSequenceClassification.from_pretrained(BGE,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0');load=time.perf_counter()-t;warm=tokenizer(['준비'],['준비'],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0');t=time.perf_counter()
    with torch.inference_mode():assert torch.isfinite(model(**warm).logits).all()
    torch.cuda.synchronize();warm_s=time.perf_counter()-t;del warm;scores={};t=time.perf_counter()
    try:
        for i in range(0,len(todo),batch):
            part=todo[i:i+batch];x=tokenizer([queries[k[0]] for k,_ in part],[b['bundle_text'] for _,b in part],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
            with torch.inference_mode():z=model(**x).logits.reshape(-1).float().cpu().numpy()
            assert len(z)==len(part) and np.isfinite(z).all();scores.update({k:float(v) for (k,_),v in zip(part,z)});del x,z
        torch.cuda.synchronize();sec=time.perf_counter()-t;meta={'load_seconds':load,'warmup_seconds':warm_s,'scoring_seconds':sec,'pairs_per_second':len(todo)/sec if todo else None,'batch_size':batch,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30};ok=True
    except torch.cuda.OutOfMemoryError:scores={};meta={};ok=False
    del model;gc.collect();torch.cuda.empty_cache();return (scores,meta) if ok else None
scored=score_new(2);oom2=scored is None
if scored is None:scored=score_new(1)
assert scored is not None;new_scores,meta=scored;assert set(new_scores)=={k for k,_ in todo} and all(math.isfinite(v) for v in new_scores.values());all_scores={k:(source_cache[candidate_strong_key[k]] if candidate_strong_key[k] in source_cache else new_scores[k]) for k in unique}
rows=[]
for k,b in sorted(unique.items()):
    strong_key=candidate_strong_key[k];depths=sorted(x['depth'] for x in bundles if (x['query_id'],x['bundle_sha256'])==k);rows.append({'query_id':k[0],'card_key':b['card_key'],'bundle_sha256':k[1],'depths_json':json.dumps(depths),'best_seed_rrf_rank':min(x['best_seed_rrf_rank'] for x in bundles if (x['query_id'],x['bundle_sha256'])==k),'score_source':source_name[strong_key] if strong_key in source_cache else 'followup5_new','score_reused':strong_key in source_cache,'raw_logit':all_scores[k],**token_rows[k]})
with (OUT/'followup5_pair_scores.csv').open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
with (OUT/'followup5_scorer_input_audit.csv').open('w',encoding='utf-8',newline='') as f:
    a=[{'query_id':r['query_id'],'card_key':r['card_key'],'query_sha256':hashlib.sha256(queries[r['query_id']].encode()).hexdigest(),'bundle_sha256':r['bundle_sha256'],'allowlist':'combined query text + frozen automatic bundle text','gold_or_evaluation_fields_used':False,'document_truncated_tokens':r['document_truncated_tokens']} for r in rows];w=csv.DictWriter(f,fieldnames=list(a[0]));w.writeheader();w.writerows(a)
score_by={(r['query_id'],r['bundle_sha256']):float(r['raw_logit']) for r in rows};rankings=[]
for depth in (20,30,50):
    for q in sorted(queries):
        x=sorted((b for b in bundles if b['depth']==depth and b['query_id']==q),key=lambda b:(-score_by[(q,b['bundle_sha256'])],b['best_seed_rrf_rank'],b['card_key']));rankings.append({'query_id':q,'depth':depth,'card_keys':[b['card_key'] for b in x],'bundle_sha256s':[b['bundle_sha256'] for b in x],'raw_logits':[score_by[(q,b['bundle_sha256'])] for b in x],'best_seed_rrf_ranks':[b['best_seed_rrf_rank'] for b in x]})
f2rank={r['query_id']:r for r in jl(OUT/'followup2_scored_rankings.jsonl') if r['cohort']=='and_combination'}
for r in [x for x in rankings if x['depth']==20]:assert r['card_keys']==f2rank[r['query_id']]['structural_card_keys'] and r['raw_logits']==f2rank[r['query_id']]['structural_raw_logits'] and r['best_seed_rrf_ranks']==f2rank[r['query_id']]['structural_best_seed_ranks']
(OUT/'followup5_rankings.jsonl').write_text(''.join(canon(r)+'\n' for r in rankings),encoding='utf-8');gpu_after=gpu_snapshot();tok=[r['document_tokens'] for r in rows]
resources={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','physical_gpu':0,'shared_ollama_allowed':True,'local_files_only':True,'trust_remote_code':False,'dtype':'float16','max_length':8192,'document_token_cap':4096,'total_unique_pairs':len(rows),'reused_pairs':sum(r['score_reused'] for r in rows),'new_scored_pairs':len(todo),'oom_batch2':oom2,'truncated_pairs':0,'token_p50':statistics.median(tok),'token_p95':float(np.percentile(tok,95)),'token_max':max(tok),'gpu_before':gpu_before,'gpu_after':gpu_after,'api_network_new_embedding_chroma_index_package_install':0,**meta}
write_json('followup5_resources.json',resources);sc={'status':'PASS_SCORE_FREEZE_BEFORE_GOLD','gold_semantic_reads_before_score_freeze':0,'pair_rows':len(rows),'pair_scores_sha256':sha(OUT/'followup5_pair_scores.csv'),'audit_sha256':sha(OUT/'followup5_scorer_input_audit.csv'),'rankings_sha256':sha(OUT/'followup5_rankings.jsonl'),'bundles_sha256':sha(OUT/'followup5_bundles.jsonl'),'resources_sha256':sha(OUT/'followup5_resources.json'),'d20_f2_ranking_exact_queries':10,'score_reuse_key_fields':['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'],'scorer_fingerprint':scorer_fingerprint,'scorer_fingerprint_sha256':scorer_fingerprint_sha,'source_provenance_rows':len(source_provenance),'source_unique_strong_keys':len(source_cache),'reused_pairs_strong_key_exact':sum(r['score_reused'] for r in rows),'input_before':freeze['input_before'],'prior_before':freeze['prior_before'],'bge_before':freeze['bge_before']};write_json('followup5_scoring_freeze.json',sc)
del tokenizer;gc.collect();torch.cuda.empty_cache();assert tree(BGE)==freeze['bge_before'] and all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and all(sha(ROOT/n)==h for n,h in freeze['input_before'].items())
print(json.dumps({'followup5_scoring':'PASS','unique_pairs':len(rows),'reused':resources['reused_pairs'],'new':len(todo),'batch':meta['batch_size'],'oom_batch2':oom2,'truncated':0},indent=2))


/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "followup5_scoring": "PASS",
  "unique_pairs": 176,
  "reused": 74,
  "new": 102,
  "batch": 2,
  "oom_batch2": false,
  "truncated": 0
}


In [1]:
# Follow-up5 CPU evaluation. Gold is first semantically loaded after score freeze verification.
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, math, os, re, resource, time, unicodedata
cwd=Path.cwd().resolve();ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None);assert ROOT is not None and os.environ.get('RUN_APPROVED_24_FOLLOWUP5_EXTERNAL','0')=='0' and os.environ.get('RUN_APPROVED_24_FOLLOWUP5_GPU','0')=='0'
OUT=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def jl(p):return [json.loads(x) for x in Path(p).read_text(encoding='utf-8').splitlines() if x]
def write_json(n,x):(OUT/n).write_text(json.dumps(x,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_csv(n,x):
    with (OUT/n).open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=list(x[0]));w.writeheader();w.writerows(x)
freeze=json.loads((OUT/'followup5_preflight_freeze.json').read_text());sc=json.loads((OUT/'followup5_scoring_freeze.json').read_text());assert sc['status']=='PASS_SCORE_FREEZE_BEFORE_GOLD' and sc['gold_semantic_reads_before_score_freeze']==0 and sc['pair_scores_sha256']==sha(OUT/'followup5_pair_scores.csv') and sc['rankings_sha256']==sha(OUT/'followup5_rankings.jsonl')
assert all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and all(sha(ROOT/n)==h for n,h in freeze['input_before'].items())
review_targets=['followup5_contract.json','followup5_preflight_freeze.json','followup5_scorer_input_audit.csv','followup5_scoring_freeze.json','followup5_summary.json','followup5_decision.json','followup5_evaluation_resources.json','followup5_README.md'];review_before={n:sha(OUT/n) for n in review_targets};pair_scores_sha_before=sha(OUT/'followup5_pair_scores.csv');summary_csv_sha_before=sha(OUT/'followup5_summary.csv');summary_numeric_before=json.loads((OUT/'followup5_summary.json').read_text())['summary'];decision_before=json.loads((OUT/'followup5_decision.json').read_text())
critical_files=['config.json','model.safetensors','sentencepiece.bpe.model','special_tokens_map.json','tokenizer.json','tokenizer_config.json'];model_files={n:freeze['bge_before']['files'][n] for n in critical_files};model_files_fingerprint=hashlib.sha256(canon(model_files).encode()).hexdigest()
scorer_fingerprint={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','model_files_sha256':model_files,'model_files_fingerprint':model_files_fingerprint,'tokenizer_files':['sentencepiece.bpe.model','special_tokens_map.json','tokenizer.json','tokenizer_config.json'],'max_length':8192,'truncation':'only_second','dtype':'float16','local_files_only':True,'trust_remote_code':False};scorer_fingerprint_sha=hashlib.sha256(canon(scorer_fingerprint).encode()).hexdigest()
f2sf=json.loads((OUT/'followup2_scoring_freeze.json').read_text());f3sf=json.loads((OUT/'followup3_scoring_freeze.json').read_text());f2res=json.loads((OUT/'followup2_model_resources.json').read_text());f3res=json.loads((OUT/'followup3_resources.json').read_text());f3contract=json.loads((OUT/'followup3_contract.json').read_text())
for source_freeze in (f2sf,f3sf):assert {n:source_freeze['bge_before']['files'][n] for n in critical_files}==model_files
assert all([f2res['revision']==scorer_fingerprint['revision'],f2res['max_length']==8192,f2res['truncation']=='only_second',f2res['dtype']=='float16',f2res['local_files_only'] is True,f2res['trust_remote_code'] is False])
assert all([f3res['revision']==scorer_fingerprint['revision'],f3contract['scoring']['max_length']==8192,f3contract['scoring']['truncation']=='only_second',f3contract['scoring']['dtype']=='float16',f3contract['scoring']['local_files_only'] is True,f3contract['scoring']['trust_remote_code'] is False])
queries={r['query_id']:r['query_text'] for r in csv.DictReader((OUT/'followup2_queries.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination'};assert len(queries)==10
assert f2sf['input_audit_sha256']==sha(OUT/'followup2_scorer_input_audit.csv') and f2sf['pair_scores_sha256']==sha(OUT/'followup2_pair_scores.csv') and f3sf['audit_sha256']==sha(OUT/'followup3_scorer_input_audit.csv') and f3sf['pair_scores_sha256']==sha(OUT/'followup3_pair_scores.csv')
source_cache={};source_name={};source_provenance=[]
def add_source(source,qid,card,bundle_sha,qsha,raw):
    assert qid in queries and qsha==hashlib.sha256(queries[qid].encode()).hexdigest();key=(qsha,bundle_sha,scorer_fingerprint_sha);raw=float(raw)
    if key in source_cache:assert source_cache[key]==raw
    source_cache[key]=raw
    if source=='followup2' or key not in source_name:source_name[key]=source
    source_provenance.append({'source':source,'query_id':qid,'card_key':card,'query_text_sha256':qsha,'bundle_sha256':bundle_sha,'scorer_fingerprint_sha256':scorer_fingerprint_sha,'raw_logit':raw})
f3audit={(r['query_id'],r['card_key']):r for r in csv.DictReader((OUT/'followup3_scorer_input_audit.csv').open(encoding='utf-8',newline=''))}
for r in csv.DictReader((OUT/'followup3_pair_scores.csv').open(encoding='utf-8',newline='')):
    a=f3audit[(r['query_id'],r['card_key'])];assert a['bundle_sha256']==r['bundle_sha256'];add_source('followup3',r['query_id'],r['card_key'],r['bundle_sha256'],a['query_sha256'],r['raw_logit'])
f2b={(r['query_id'],r['card_key']):r for r in jl(OUT/'followup2_bundles.jsonl')};f2audit={(r['query_id'],r['candidate_id']):r for r in csv.DictReader((OUT/'followup2_scorer_input_audit.csv').open(encoding='utf-8',newline='')) if r['pair_type']=='structural_bundle'}
for r in csv.DictReader((OUT/'followup2_pair_scores.csv').open(encoding='utf-8',newline='')):
    if r['pair_type']=='structural_bundle' and r['query_id'] in queries:
        b=f2b[(r['query_id'],r['card_key'])];a=f2audit[(r['query_id'],r['card_key'])];assert a['document_sha256']==b['bundle_sha256'];add_source('followup2',r['query_id'],r['card_key'],b['bundle_sha256'],a['query_sha256'],r['raw_logit'])
assert len(source_provenance)==150 and len(source_cache)==139;write_csv('followup5_source_score_provenance.csv',source_provenance)
pair_rows=list(csv.DictReader((OUT/'followup5_pair_scores.csv').open(encoding='utf-8',newline='')));input_audit={(r['query_id'],r['bundle_sha256']):r for r in csv.DictReader((OUT/'followup5_scorer_input_audit.csv').open(encoding='utf-8',newline=''))};reuse_audit=[];new_score_cache={};reconstructed={}
for r in pair_rows:
    qsha=hashlib.sha256(queries[r['query_id']].encode()).hexdigest();strong_key=(qsha,r['bundle_sha256'],scorer_fingerprint_sha);reuse_key_sha=hashlib.sha256(canon({'query_text_sha256':qsha,'bundle_sha256':r['bundle_sha256'],'scorer_fingerprint_sha256':scorer_fingerprint_sha}).encode()).hexdigest();saved=float(r['raw_logit']);reused=strong_key in source_cache
    if reused:
        assert r['score_reused']=='True' and r['score_source']==source_name[strong_key] and saved==source_cache[strong_key];reconstructed[strong_key]=source_cache[strong_key];reuse_audit.append({'query_id':r['query_id'],'card_key':r['card_key'],'query_text_sha256':qsha,'bundle_sha256':r['bundle_sha256'],'score_source':source_name[strong_key],'source_raw_logit':source_cache[strong_key],'followup5_raw_logit':saved,'scorer_fingerprint_sha256':scorer_fingerprint_sha,'reuse_key_sha256':reuse_key_sha,'reuse_eligible_exact':True})
    else:
        assert r['score_reused']=='False' and r['score_source']=='followup5_new'
        if strong_key in new_score_cache:assert new_score_cache[strong_key]==saved
        new_score_cache[strong_key]=saved;reconstructed[strong_key]=saved
    a=input_audit[(r['query_id'],r['bundle_sha256'])];a.update({'query_text_sha256':qsha,'scorer_fingerprint_sha256':scorer_fingerprint_sha,'reuse_key_sha256':reuse_key_sha,'score_reused':reused,'score_source':r['score_source'],'reuse_eligible_exact':reused})
assert len(pair_rows)==176 and len(reconstructed)==176 and len(reuse_audit)==74 and len(new_score_cache)==102 and all(math.isfinite(v) for v in reconstructed.values());write_csv('followup5_score_reuse_audit.csv',reuse_audit);write_csv('followup5_scorer_input_audit.csv',[input_audit[(r['query_id'],r['bundle_sha256'])] for r in pair_rows])
source_join={'status':'PASS_STRONG_KEY_SAVED_SCORE_RECONSTRUCTION','key_fields':['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'],'query_source':{'file':'followup2_queries.csv','sha256':sha(OUT/'followup2_queries.csv'),'rows':10},'source_artifacts':{'followup2_scorer_input_audit.csv':sha(OUT/'followup2_scorer_input_audit.csv'),'followup2_pair_scores.csv':sha(OUT/'followup2_pair_scores.csv'),'followup2_bundles.jsonl':sha(OUT/'followup2_bundles.jsonl'),'followup2_scoring_freeze.json':sha(OUT/'followup2_scoring_freeze.json'),'followup3_scorer_input_audit.csv':sha(OUT/'followup3_scorer_input_audit.csv'),'followup3_pair_scores.csv':sha(OUT/'followup3_pair_scores.csv'),'followup3_scoring_freeze.json':sha(OUT/'followup3_scoring_freeze.json')},'source_provenance_rows':len(source_provenance),'source_unique_strong_keys':len(source_cache),'f5_rows':len(pair_rows),'reused_strong_key_exact':len(reuse_audit),'new_saved_score_strong_key_exact':len(new_score_cache),'pair_scores_sha256_preserved':sha(OUT/'followup5_pair_scores.csv')==pair_scores_sha_before,'api_gpu_model_search':0};source_join['source_provenance_sha256']=sha(OUT/'followup5_source_score_provenance.csv');source_join['canonical_digest']=hashlib.sha256(canon({k:v for k,v in source_join.items() if k!='canonical_digest'}).encode()).hexdigest();write_json('followup5_saved_score_reconstruction.json',source_join)
contract=json.loads((OUT/'followup5_contract.json').read_text());contract['scoring']['reuse']='actual cache lookup uses exact (query_text_sha256, bundle_sha256, scorer_fingerprint_sha256) only; query_id+bundle fallback forbidden';contract['scoring']['reuse_key_fields']=['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'];contract['scoring']['source_query_join']='followup2_queries.csv query_id -> canonical query_text SHA, exact checked against F2/F3 scorer input audit';contract['scoring']['scorer_fingerprint']=scorer_fingerprint;contract['scoring']['scorer_fingerprint_sha256']=scorer_fingerprint_sha;contract['scoring']['reuse_exact_verified_pairs']=74;contract['scoring']['new_saved_score_exact_verified_pairs']=102;contract['gate']['card_recall_at_3_min']=0.8416666666666668;contract['gate']['card_recall_at_3_source']='followup2_summary.json / and_combination / old_selective_bge_d20 raw macro';contract['gate']['tolerance']=1e-12;contract['resource_interpretation']='D20 to D50 +44.1% is candidate-card pair growth only; per-depth GPU latency was not measured';write_json('followup5_contract.json',contract)
freeze['contract_sha256']=sha(OUT/'followup5_contract.json');freeze['score_reuse_fingerprint_precheck']={'scorer_fingerprint_sha256':scorer_fingerprint_sha,'eligible_pairs':74,'ineligible_pairs':0,'actual_cache_key_fields':['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'],'query_source_sha256':sha(OUT/'followup2_queries.csv')};write_json('followup5_preflight_freeze.json',freeze)
sc['audit_sha256']=sha(OUT/'followup5_scorer_input_audit.csv');sc['reuse_audit_sha256']=sha(OUT/'followup5_score_reuse_audit.csv');sc['source_provenance_sha256']=sha(OUT/'followup5_source_score_provenance.csv');sc['saved_score_reconstruction_sha256']=sha(OUT/'followup5_saved_score_reconstruction.json');sc['scorer_fingerprint']=scorer_fingerprint;sc['scorer_fingerprint_sha256']=scorer_fingerprint_sha;sc['score_reuse_key_fields']=['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'];sc['source_unique_strong_keys']=139;sc['reused_pairs_strong_key_exact']=74;sc['new_saved_score_strong_key_exact']=102;write_json('followup5_scoring_freeze.json',sc)
started=time.perf_counter();cpu0=time.process_time();rss0=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss;bundles=jl(OUT/'followup5_bundles.jsonl');candidates={(r['query_id'],int(r['depth'])):r for r in jl(OUT/'followup5_candidate_sets.jsonl')};rankings={(r['query_id'],int(r['depth'])):r for r in jl(OUT/'followup5_rankings.jsonl')};bundle_by={(r['query_id'],int(r['depth']),r['card_key']):r for r in bundles};assert len(candidates)==len(rankings)==30 and len(bundles)==251
# Frozen F2 gold and claim audit are loaded only here.
labels=list(csv.DictReader((OUT/'followup2_gold_labels.csv').open(encoding='utf-8',newline='')));audits=jl(OUT/'followup2_atomic_claim_audit.jsonl');assert len(labels)==200 and len(audits)==33
labels=[r for r in labels if r['cohort']=='and_combination'];label_by={(r['query_id'],r['card_key']):r for r in labels};claim_by={r['claim_id']:r for r in audits};positives=defaultdict(set)
for r in labels:
    if r['label']=='positive':positives[r['query_id']].add(r['card_key'])
norm=lambda x:' '.join(unicodedata.normalize('NFKC',str(x)).lower().split())
def role_ok(role,text):
    raw=norm(role['text']);stripped=norm(re.sub(r'^#{1,6}\s*','',role['text']));hay=norm(text);return raw in hay or bool(stripped and stripped in hay)
def supported(q,card,text):
    row=label_by[(q,card)]
    return row['label']=='positive' and all(all(role_ok(role,text) for role in claim_by[cid]['roles']) for cid in json.loads(row['claim_ids_json']))
f2=list(csv.DictReader((OUT/'followup2_per_query.csv').open(encoding='utf-8',newline='')));f2={(r['system'],r['query_id']):r for r in f2 if r['cohort']=='and_combination'};qids=sorted(positives);assert qids==[f'CMB{i:02d}' for i in range(1,11)]
metric_names=['card_precision_at_3','card_recall_at_3','card_precision_at_5','card_recall_at_5','evidence_supported_card_recall_at_5','evidence_accuracy_at_3','evidence_accuracy_at_5','candidate_card_recall_at_n','bundle_reachable_evidence_recall_at_n','card_zero_hit_at_5','supported_zero_hit_at_5']
per=[]
for q in qids:
    r=f2[('old_selective_bge_d20',q)];per.append({'system':'old_selective_bge_d20_fixed','query_id':q,'depth':20,'output_card_count':int(r['output_card_count']),'missing_slots_at_5':int(r['missing_slots_at_5']),**{m:float(r[m]) for m in metric_names[:7]},'candidate_card_recall_at_n':float(r['candidate_card_recall_at_20']),'bundle_reachable_evidence_recall_at_n':float(r['bundle_reachable_evidence_recall_at_20']),'card_zero_hit_at_5':int(float(r['card_zero_hit_at_5'])),'supported_zero_hit_at_5':int(float(r['supported_zero_hit_at_5'])),'raw_chunk_count':20,'unique_candidate_cards':int(r['raw_top20_unique_cards']),'duplicate_chunk_count':int(r['raw_top20_duplicate_count']),'output_card_keys_json':r['output_card_keys_json'],'supported_json':r['supported_json'],'positive_count':int(r['positive_count']),'source':'followup2 exact fixed baseline'})
for depth in (20,30,50):
    system=f'structural_bundle_bge_d{depth}'
    for q in qids:
        c=candidates[(q,depth)];rr=rankings[(q,depth)];assert rr['card_keys']==sorted(c['card_keys'],key=lambda card:(-rr['raw_logits'][rr['card_keys'].index(card)],c['card_best_seed_ranks'][card],card)) and set(rr['card_keys'])==set(c['card_keys'])
        pos=positives[q];top5=rr['card_keys'][:5];support=[supported(q,card,bundle_by[(q,depth,card)]['bundle_text']) if card in pos else False for card in top5];support += [False]*(5-len(support));hit3=sum(card in pos for card in top5[:3]);hit5=sum(card in pos for card in top5);reachable={card:supported(q,card,bundle_by[(q,depth,card)]['bundle_text']) if card in pos else False for card in c['card_keys']}
        per.append({'system':system,'query_id':q,'depth':depth,'output_card_count':len(top5),'missing_slots_at_5':5-len(top5),'card_precision_at_3':hit3/3,'card_recall_at_3':hit3/len(pos),'card_precision_at_5':hit5/5,'card_recall_at_5':hit5/len(pos),'evidence_supported_card_recall_at_5':sum(support)/len(pos),'evidence_accuracy_at_3':sum(support[:3])/3,'evidence_accuracy_at_5':sum(support)/5,'candidate_card_recall_at_n':len(pos&set(c['card_keys']))/len(pos),'bundle_reachable_evidence_recall_at_n':sum(reachable.get(card,False) for card in pos)/len(pos),'card_zero_hit_at_5':int(hit5==0),'supported_zero_hit_at_5':int(sum(support)==0),'raw_chunk_count':depth,'unique_candidate_cards':c['unique_card_count'],'duplicate_chunk_count':c['duplicate_chunk_count'],'output_card_keys_json':json.dumps(top5,ensure_ascii=False),'supported_json':json.dumps(support),'positive_count':len(pos),'source':'followup5 stored score ranking'})
assert len(per)==40 and all(0<=float(r[m])<=1 for r in per for m in metric_names)
for q in qids:
    a=next(r for r in per if r['system']=='structural_bundle_bge_d20' and r['query_id']==q);b=f2[('structural_bundle_all_bge_d20',q)];mapping={'candidate_card_recall_at_n':'candidate_card_recall_at_20','bundle_reachable_evidence_recall_at_n':'bundle_reachable_evidence_recall_at_20'}
    assert a['output_card_keys_json']==b['output_card_keys_json'] and a['supported_json']==b['supported_json'] and all(abs(float(a[m])-float(b[mapping.get(m,m)]))<=1e-12 for m in metric_names)
summary=[]
for system in ['old_selective_bge_d20_fixed','structural_bundle_bge_d20','structural_bundle_bge_d30','structural_bundle_bge_d50']:
    x=[r for r in per if r['system']==system];assert len(x)==10;summary.append({'system':system,'depth':int(x[0]['depth']),'denominator':10,**{m:sum(float(r[m]) for r in x)/10 for m in metric_names},'mean_unique_candidate_cards':sum(int(r['unique_candidate_cards']) for r in x)/10,'mean_duplicate_chunk_count':sum(int(r['duplicate_chunk_count']) for r in x)/10,'total_candidate_card_pairs':sum(int(r['unique_candidate_cards']) for r in x)})
sb={r['system']:r for r in summary};f2_source_summary=next(r for r in json.loads((OUT/'followup2_summary.json').read_text())['summary'] if r['cohort']=='and_combination' and r['system']=='old_selective_bge_d20');old_card_recall_at_3=float(f2_source_summary['card_recall_at_3']);assert abs(old_card_recall_at_3-0.8416666666666668)<=1e-12
assert abs(sb['old_selective_bge_d20_fixed']['card_recall_at_3']-old_card_recall_at_3)<=1e-12 and abs(sb['old_selective_bge_d20_fixed']['candidate_card_recall_at_n']-.95)<1e-12 and abs(sb['old_selective_bge_d20_fixed']['card_recall_at_5']-.925)<1e-12 and abs(sb['old_selective_bge_d20_fixed']['card_precision_at_5']-.42)<1e-12 and abs(sb['structural_bundle_bge_d20']['evidence_supported_card_recall_at_5']-.6333333333333333)<1e-12 and abs(sb['structural_bundle_bge_d20']['evidence_accuracy_at_5']-.28)<1e-12
delta_metrics=metric_names[:9];comparisons=[('old_selective_bge_d20_fixed',f'structural_bundle_bge_d{d}') for d in (20,30,50)]+[('structural_bundle_bge_d20',f'structural_bundle_bge_d{d}') for d in (30,50)];paired=[];wlt=[]
for ref,cand in comparisons:
    rb={r['query_id']:r for r in per if r['system']==ref};cb={r['query_id']:r for r in per if r['system']==cand}
    for q in qids:paired.append({'reference':ref,'candidate':cand,'query_id':q,**{'delta_'+m:float(cb[q][m])-float(rb[q][m]) for m in delta_metrics}})
    for m in delta_metrics:
        vals=[float(cb[q][m])-float(rb[q][m]) for q in qids];wins=sum(v>1e-12 for v in vals);losses=sum(v<-1e-12 for v in vals);wlt.append({'reference':ref,'candidate':cand,'metric':m,'denominator':10,'wins':wins,'losses':losses,'ties':10-wins-losses,'mean_delta':sum(vals)/10})
old_by={r['query_id']:r for r in per if r['system']=='old_selective_bge_d20_fixed'};gates={};catastrophic={}
for depth in (30,50):
    s=sb[f'structural_bundle_bge_d{depth}'];cb={r['query_id']:r for r in per if r['system']==f'structural_bundle_bge_d{depth}'};cat={q:(float(old_by[q]['evidence_supported_card_recall_at_5'])>0 and (float(cb[q]['evidence_supported_card_recall_at_5'])==0 or float(cb[q]['evidence_supported_card_recall_at_5'])-float(old_by[q]['evidence_supported_card_recall_at_5'])<=-.5+1e-12)) for q in qids};catastrophic[str(depth)]={q:v for q,v in cat.items() if v}
    checks={'candidate_card_recall_at_n_gte_old_0_95':s['candidate_card_recall_at_n']>=.95-1e-12,'new_zero_ceiling_count_0':sum(float(cb[q]['candidate_card_recall_at_n'])==0 for q in qids)==0,'card_recall_at_5_gte_0_925':s['card_recall_at_5']>=.925-1e-12,'card_precision_at_5_gte_0_42':s['card_precision_at_5']>=.42-1e-12,'card_recall_at_3_gte_f2_old_raw':s['card_recall_at_3']>=old_card_recall_at_3-1e-12,'supported_recall_at_5_gte_structural_d20_0_6333':s['evidence_supported_card_recall_at_5']>=.6333-1e-12,'evidence_accuracy_at_5_gte_structural_d20_0_28':s['evidence_accuracy_at_5']>=.28-1e-12,'catastrophic_loss_count_0':not any(cat.values()),'contract_or_truncation_count_0':json.loads((OUT/'followup5_resources.json').read_text())['truncated_pairs']==0};gates[str(depth)]={'checks':checks,'pass':all(checks.values())}
passing=[d for d in (30,50) if gates[str(d)]['pass']];selected=min(passing) if passing else None;decision={'gates':gates,'selected_depth':selected,'selection':'structural_bundle_bge_d'+str(selected) if selected else 'old_selective_bge_d20_fixed','disposition':('exploratory_structural_depth_candidate_d'+str(selected) if selected else 'retain_old_selective_bge_d20'),'catastrophic_queries':catastrophic,'technical_gate_only':True,'gate_threshold_sources':{'card_recall_at_3':{'value':old_card_recall_at_3,'source_file':'followup2_summary.json','source_key':['and_combination','old_selective_bge_d20','card_recall_at_3'],'tolerance':1e-12}},'score_reuse_strong_fingerprint_exact_pairs':74,'always':['same_10_card_development_only','not_eligible_for_promotion','not_operational_or_holdout_evidence']}
special=[];card_id_key={(r['query_id'],r['card_id']):r['card_key'] for r in labels};targets=[('CMB05','C02','NH'),('CMB08','C02','NH'),('CMB08','C09','Shinhan')]
for q,cid,label in targets:
    card=card_id_key[(q,cid)]
    for depth in (20,30,50):
        c=candidates[(q,depth)];rr=rankings[(q,depth)];present=card in c['card_keys'];top5=card in rr['card_keys'][:5];special.append({'query_id':q,'card_id':cid,'issuer_label':label,'card_key':card,'depth':depth,'candidate_present':present,'best_seed_rrf_rank':c['card_best_seed_ranks'].get(card,''),'top5_present':top5,'top5_rank':(rr['card_keys'][:5].index(card)+1 if top5 else ''),'bundle_supported':(supported(q,card,bundle_by[(q,depth,card)]['bundle_text']) if present else False)})
assert all(not r['candidate_present'] for r in special if r['depth']==20)
write_csv('followup5_per_query.csv',per);write_csv('followup5_summary.csv',summary);write_json('followup5_summary.json',{'summary':summary,'decision':decision,'metric_notes':{'candidate_card_recall_at_n':'raw chunk prefix를 카드 단위로 접은 후보 recall','evidence_supported_card_recall_at_5':'Top5 카드 중 자동 번들이 두 frozen atomic claim을 모두 담은 정답 카드 recall','fixed_k':'빈 슬롯은 실패'}});write_csv('followup5_paired_deltas.csv',paired);write_csv('followup5_wlt.csv',wlt);write_csv('followup5_missing_pair_audit.csv',special);write_json('followup5_decision.json',decision)
resources=json.loads((OUT/'followup5_resources.json').read_text())
for key in ['evaluation_cpu','quality_cost_by_depth','api_cost_usd','new_embeddings','index_or_storage_change']:resources.pop(key,None)
write_json('followup5_resources.json',resources);assert sha(OUT/'followup5_resources.json')==sc['resources_sha256']
evaluation_resources={'evaluation_cpu':{'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu0,'peak_rss_mib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,'incremental_ru_maxrss_mib':max(0,resource.getrusage(resource.RUSAGE_SELF).ru_maxrss-rss0)/1024},'quality_cost_by_depth':{str(d):{'candidate_card_pairs':int(sb[f'structural_bundle_bge_d{d}']['total_candidate_card_pairs']),'candidate_card_pair_increase_vs_d20':int(sb[f'structural_bundle_bge_d{d}']['total_candidate_card_pairs'])/68-1,'mean_unique_cards':sb[f'structural_bundle_bge_d{d}']['mean_unique_candidate_cards'],'near_full_10_card_corpus':sb[f'structural_bundle_bge_d{d}']['mean_unique_candidate_cards']>=9.5} for d in (20,30,50)},'scoring_resources_sha256':sc['resources_sha256'],'new_scoring_measurement':{'pairs':resources['new_scored_pairs'],'scoring_seconds':resources['scoring_seconds'],'scope':'aggregate 102 newly scored unique query-bundle pairs only'},'depth_specific_gpu_latency':'not_measured','latency_reduction_or_increase_claim_allowed':False,'api_cost_usd':0,'new_embeddings':0,'index_or_storage_change':0};write_json('followup5_evaluation_resources.json',evaluation_resources)
(OUT/'followup5_README.md').write_text('# Follow-up5 — structural candidate depth D20/D30/D50 ablation\n\n복합 조건 질의에서 structural RRF 청크 후보를 20개, 30개, 50개로 늘렸을 때 빠진 카드와 근거가 돌아오는지 본 개발셋 진단입니다. Precision은 상위 고정 슬롯 중 정답 비율, Recall은 전체 정답 카드 중 찾은 비율, Evidence Accuracy는 상위 고정 슬롯 중 두 조건 근거를 자동 번들로 함께 뒷받침한 비율입니다. 저장 점수 재사용은 query text SHA-256, bundle SHA-256, BGE scorer fingerprint SHA-256의 정확한 3항 key로만 수행하며 query ID와 bundle만을 쓴 fallback은 없습니다. D50은 147개 구조 청크 안에서 질의당 평균 카드 후보 수가 10개에 가까워 거의 전수 카드 비교이며, 시장 전체 추천을 뜻하지 않습니다. D20 대비 D50의 +44.1%는 candidate-card pair 수(68→98)의 증가율입니다. depth별 GPU latency는 따로 측정하지 않았으므로 시간 증가율이나 절감률을 주장하지 않습니다. 기록된 scoring 시간은 신규 102개 unique query-bundle pair 전체의 합계입니다. 기술 gate 통과 여부와 관계없이 운영·holdout 승격 근거가 아닙니다.\n',encoding='utf-8')
assert summary==summary_numeric_before and decision['disposition']==decision_before['disposition'] and decision['selected_depth']==decision_before['selected_depth'] and {d:decision['gates'][d]['pass'] for d in ('30','50')}=={d:decision_before['gates'][d]['pass'] for d in ('30','50')} and sha(OUT/'followup5_pair_scores.csv')==pair_scores_sha_before and sha(OUT/'followup5_summary.csv')==summary_csv_sha_before
review_after={n:sha(OUT/n) for n in review_targets};write_json('followup5_review_hash_changes.json',{'status':'PASS_REVIEWER_MAJOR_STRONG_KEY_PATH_REMEDIATION','changed_evidence_before':review_before,'changed_evidence_after':review_after,'pair_scores_sha256_before_after':pair_scores_sha_before,'summary_csv_sha256_before_after':summary_csv_sha_before,'summary_numeric_exact':True,'decision_disposition_and_gate_pass_exact':True,'actual_cache_key_fields':['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'],'source_provenance_rows':150,'source_unique_strong_keys':139,'reused_pairs_strong_fingerprint_exact':74,'new_saved_score_strong_fingerprint_exact':102,'query_id_bundle_fallback':False,'api_gpu_model_search':0})
output_names=['followup5_contract.json','followup5_candidate_sets.jsonl','followup5_bundles.jsonl','followup5_bundle_trace.csv','followup5_preflight_freeze.json','followup5_pair_scores.csv','followup5_scorer_input_audit.csv','followup5_score_reuse_audit.csv','followup5_source_score_provenance.csv','followup5_saved_score_reconstruction.json','followup5_rankings.jsonl','followup5_resources.json','followup5_evaluation_resources.json','followup5_scoring_freeze.json','followup5_per_query.csv','followup5_summary.csv','followup5_summary.json','followup5_paired_deltas.csv','followup5_wlt.csv','followup5_missing_pair_audit.csv','followup5_decision.json','followup5_README.md','followup5_review_hash_changes.json']
assert all(sha(OUT/n)==h for n,h in freeze['prior_before'].items()) and all(sha(ROOT/n)==h for n,h in freeze['input_before'].items())
integrity={'status':'PASS','compile_checked_separately':True,'execution_errors':0,'query_count':10,'depths':[20,30,50],'candidate_rows':30,'bundle_rows':251,'pair_score_rows':sc['pair_rows'],'pair_scores_finite':True,'pair_scores_raw_sha_preserved':sha(OUT/'followup5_pair_scores.csv')==pair_scores_sha_before,'score_reuse_actual_cache_key_fields':['query_text_sha256','bundle_sha256','scorer_fingerprint_sha256'],'score_reuse_query_id_bundle_fallback':False,'score_reuse_fingerprint_sha256':scorer_fingerprint_sha,'source_provenance_rows':150,'source_unique_strong_keys':139,'score_reuse_strong_key_exact_pairs':74,'new_saved_score_strong_key_exact_pairs':102,'score_reuse_ineligible_pairs':0,'source_provenance_sha256':sha(OUT/'followup5_source_score_provenance.csv'),'saved_score_reconstruction_sha256':sha(OUT/'followup5_saved_score_reconstruction.json'),'card_recall_at_3_gate_source_value':old_card_recall_at_3,'card_recall_at_3_gate_tolerance':1e-12,'depth_gpu_latency_measured':False,'ranking_rows':30,'per_query_rows':40,'summary_rows':4,'paired_rows':len(paired),'wlt_rows':len(wlt),'missing_pair_audit_rows':len(special),'d20_f2_bundle_ranking_metric_exact':True,'fixed_k_missing_slots_failures':True,'gold_read_after_candidate_bundle_score_ranking_freeze':True,'root_children_fanout_count':0,'truncated_pairs':resources['truncated_pairs'],'api_network_new_embedding_chroma_index_package_install':0,'input_before':freeze['input_before'],'input_after':freeze['input_before'],'prior_before':freeze['prior_before'],'prior_after':freeze['prior_before'],'bge_before':freeze['bge_before'],'bge_after':freeze['bge_before'],'output_hashes':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','run_manifest_sha256':'FINALIZE_AFTER_SAVE'};write_json('followup5_integrity.json',integrity)
manifest={'phase':'followup5_structural_candidate_depth_ablation','cpu_preflight_fresh_kernel':True,'gpu_scoring_fresh_kernel':True,'evaluation_fresh_kernel':True,'api_network_new_embedding_chroma':0,'gpu_physical':0,'inputs':freeze['input_before'],'prior_outputs':freeze['prior_before'],'outputs':{n:sha(OUT/n) for n in output_names},'notebook_sha256':'FINALIZE_AFTER_SAVE','self_hash_policy':'followup5_run_manifest.json and followup5_integrity.json excluded'};write_json('followup5_run_manifest.json',manifest);integrity['run_manifest_sha256']=sha(OUT/'followup5_run_manifest.json');write_json('followup5_integrity.json',integrity)
print(json.dumps({'followup5_evaluation':'PASS','summary':summary,'decision':decision,'missing_pair_audit':special},ensure_ascii=False,indent=2))


{
  "followup5_evaluation": "PASS",
  "summary": [
    {
      "system": "old_selective_bge_d20_fixed",
      "depth": 20,
      "denominator": 10,
      "card_precision_at_3": 0.6333333333333334,
      "card_recall_at_3": 0.8416666666666668,
      "card_precision_at_5": 0.42000000000000004,
      "card_recall_at_5": 0.925,
      "evidence_supported_card_recall_at_5": 0.38333333333333336,
      "evidence_accuracy_at_3": 0.3,
      "evidence_accuracy_at_5": 0.18,
      "candidate_card_recall_at_n": 0.95,
      "bundle_reachable_evidence_recall_at_n": 0.7666666666666667,
      "card_zero_hit_at_5": 0.0,
      "supported_zero_hit_at_5": 0.3,
      "mean_unique_candidate_cards": 5.4,
      "mean_duplicate_chunk_count": 14.6,
      "total_candidate_card_pairs": 54
    },
    {
      "system": "structural_bundle_bge_d20",
      "depth": 20,
      "denominator": 10,
      "card_precision_at_3": 0.5666666666666667,
      "card_recall_at_3": 0.7583333333333333,
      "card_precision_at_5": 0.38